
Run the below code to download the Dataset into the Runtime.
```
# ---- interactive prompt ----------------------------------------------------
print("Kaggle dataset access, choose a mode:")
print("  [1] Download a single video")
print("  [2] Download the whole dataset (~103GB)")
print("  [3] Skip download (use an existing local video)")

import threading

choice = {"value": None}

def get_choice():
    choice["value"] = input("Enter 1, 2, or 3: ").strip()

t = threading.Thread(target=get_choice, daemon=True)
t.start()
t.join(timeout=3)

if choice["value"] is None:
    choice["value"] = "3"
    print("\n[Auto] 3 seconds elapsed : defaulting to Skip download.")

choice = choice["value"]

KAGGLE_MODE = {
    "1": "single_file",
    "2": "full",
    "3": "skip"
}.get(choice)

if KAGGLE_MODE is None:
    raise ValueError(f"Invalid choice: {choice!r}. Expected 1, 2, or 3.")
# ----------------------------------------------------------------------------
```



In [15]:
!pip install -qq reportlab stable-baselines3 ultralytics lap

In [16]:
"""
Scene Solver v3 : Coherent Reasoning System
============================================
Stage 0 : AutoEncoder (AE)           : PRIMARY anomaly signal (batched)
Stage 1 : TimeSformer Binary         : Semantic gate (anomaly/normal)
Stage 2 : TimeSformer Multi-class    : Fine-grained anomaly classification
Stage 3 : YOLO Weapon (EXPERIMENTAL) : Hard safety trigger (conditional)
Stage 4 : Audio Feature Extraction   : Environmental signal
Stage 5 : Tracking / Behavior        : Motion behavior extraction
Stage 6 : Fusion Layer               : xAI-style reasoning core
Stage 7 : RL3033 (EXPERIMENTAL)      : Temporal reasoning agent (FINAL)
Stage 8 : LLaVA (EXPERIMENTAL)       : Context explanation (multi-frame)

LOGIC-NORMAL: When Stage 1 = Normal, status is overridden to SAFE,
Stages 2 and 3 are skipped, and LLaVA runs with neutral prompts.
"""

# ── Standard library ──────────────────────────────────────────────────────────
import json
import os
import gc
import math
import time
import logging
import threading
import shutil
from datetime import datetime
from typing import List, Dict, Optional, Tuple, Any

# ── Third-party ───────────────────────────────────────────────────────────────
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.pdfgen import canvas as rl_canvas
from reportlab.platypus import Table, TableStyle

# ── Optional heavy imports ────────────────────────────────────────────────────
try:
    from transformers import (
        TimesformerForVideoClassification,
        LlavaForConditionalGeneration,
        AutoProcessor,
        VideoMAEImageProcessor,
    )
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("[WARNING] transformers not available — Stages 1/2/8 will be skipped.")

try:
    from stable_baselines3 import PPO
    RL_AVAILABLE = True
except ImportError:
    RL_AVAILABLE = False
    print("[WARNING] stable-baselines3 not available — Stage 7 (RL) will be skipped.")

try:
    import librosa
    LIBROSA_AVAILABLE = True
except ImportError:
    LIBROSA_AVAILABLE = False
    print("[WARNING] librosa not available — Stage 4 (Audio) will be limited.")

try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
except ImportError:
    YOLO_AVAILABLE = False
    print("[INFO] ultralytics not available — Stage 3 (YOLO) will be disabled.")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("SceneSolver")


# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
from huggingface_hub import hf_hub_download,snapshot_download
HF_ASSET_REPO = "samxkevin/scenesolver-assets"
AUDIOPATHS = {
    "CALLER_TUNE": hf_hub_download(
        repo_id=HF_ASSET_REPO,
        filename="audio/phone.mp3",
    ),
    "XTTS_SPEAKER_WAV": hf_hub_download(
        repo_id=HF_ASSET_REPO,
        filename="audio/DoctorVoice_22050.wav",
    ),
}

class Config:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    PATHS = {
        "AAE": hf_hub_download(
            repo_id=HF_ASSET_REPO,
            filename="AAE/best_model.pth",
        ),
        "AE": hf_hub_download(
            repo_id=HF_ASSET_REPO,
            filename="AE/best_autoencoder.pth",
        ),
        "RL_AGENT": hf_hub_download(
            repo_id=HF_ASSET_REPO,
            filename="RL/best_model.zip",
        ),
        "TS_BINARY": hf_hub_download(
            repo_id=HF_ASSET_REPO,
            filename="TFB/timesformer_best_binary.pth",
        ),
        "TS_MULTI": hf_hub_download(
            repo_id=HF_ASSET_REPO,
            filename="TFM/best_model.pth",
        ),
        "YOLO": hf_hub_download(
            repo_id=HF_ASSET_REPO,
            filename="YOLO/weapon_yolov8n.pt",
        ),
        "LLAVA":     snapshot_download(
            repo_id="samxkevin/SceneDescriptionModel"
        ),
    }
    print("[HF] All SceneSolver assets are available.")
    SAFE_THRESHOLD_FLOOR    = 0.030
    SAFE_THRESHOLD_LEGACY   = 0.055
    AUDIO_ANOMALY_THRESHOLD = 0.1

    CLASS_NAMES = {
        0: "Aggression",     1: "TheftOrLarceny", 2: "Firebombing",
        3: "LawEnforcement", 4: "Shooting",        5: "Vandalism",
        6: "RoadAccidents",
    }
    WEAPON_CLASS_NAMES = {0: "Melee", 1: "Firearms"}

    # RL3033 observation space dims (must match training)
    RL_AE_WINDOW   = 32
    RL_AE_CHANNELS = 6
    RL_LATENT_MOT  = 1536
    RL_LATENT_STK  = 2560
    RL_SPATIAL_DIM = 196
    RL_CONTEXT_DIM = 5

    AE_BATCH_SIZE   = 16
    AE_FRAME_STRIDE = 5

    EMA_SHORT_ALPHA = 0.3
    EMA_LONG_ALPHA  = 0.05

    FUSION_WEAPON_BOOST       = 0.40
    FUSION_MOTION_BOOST       = 0.15
    FUSION_AUDIO_BOOST        = 0.10
    FUSION_NO_SUPPORT_PENALTY = 0.15

    ALERT_RISK_HIGH   = 0.70
    ALERT_RISK_MEDIUM = 0.40

    AE_WINDOW_PAD_RIGHT = True


# ══════════════════════════════════════════════════════════════════════════════
# UTILITIES
# ══════════════════════════════════════════════════════════════════════════════
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def compute_dynamic_threshold(scores: List[float],
                               floor: float = Config.SAFE_THRESHOLD_FLOOR) -> float:
    if not scores:
        return Config.SAFE_THRESHOLD_LEGACY
    arr = np.array(scores, dtype=np.float64)
    return max(float(np.mean(arr) + 2.0 * np.std(arr)), floor)


def extract_frames(video_path: str) -> Tuple[List[np.ndarray], float]:
    cap    = cv2.VideoCapture(video_path)
    frames = []
    fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames, fps


def extract_audio(video_path: str) -> Optional[str]:
    output_path = "/tmp/audio.wav"
    os.system(
        f"ffmpeg -y -loglevel quiet -i '{video_path}' "
        f"-ac 1 -ar 22050 '{output_path}' 2>/dev/null"
    )
    return output_path if os.path.exists(output_path) else None


def frame_to_timestamp(frame_idx: int, fps: float) -> str:
    """Convert a frame index to MM:SS.mmm."""
    if fps <= 0:
        fps = 25.0
    total_seconds = frame_idx / fps
    minutes = int(total_seconds // 60)
    seconds = total_seconds % 60
    return f"{minutes:02d}:{seconds:06.3f}"


def calculate_enhanced_score(img: torch.Tensor, recon: torch.Tensor) -> float:
    mse = torch.mean((img - recon) ** 2).item()
    mae = torch.mean(torch.abs(img - recon)).item()
    return 0.6 * mse + 0.4 * mae


def compute_ema(values: np.ndarray, alpha: float) -> np.ndarray:
    ema    = np.zeros_like(values)
    ema[0] = values[0]
    for i in range(1, len(values)):
        ema[i] = alpha * values[i] + (1.0 - alpha) * ema[i - 1]
    return ema


def find_anomaly_extent(scores: List[float], threshold: float) -> Tuple[int, int]:
    above = [i for i, s in enumerate(scores) if s > threshold]
    if not above:
        peak = int(np.argmax(scores))
        half = max(1, len(scores) // 10)
        return max(0, peak - half), min(len(scores) - 1, peak + half)
    return above[0], above[-1]


def z_score_normalize(arr: np.ndarray) -> np.ndarray:
    mean = np.mean(arr)
    std  = np.std(arr)
    return (arr - mean) / max(std, 1e-8)


def check_audio_has_content(audio_path: str) -> bool:
    if not audio_path or not os.path.exists(audio_path) or not LIBROSA_AVAILABLE:
        return False
    try:
        y, _ = librosa.load(audio_path, sr=22050, duration=10)
        return float(np.sqrt(np.mean(y ** 2))) > 1e-4
    except Exception:
        return False


# ══════════════════════════════════════════════════════════════════════════════
# SEGMENT-WISE AE-WEIGHTED TEMPORAL SAMPLING
# ══════════════════════════════════════════════════════════════════════════════
def segmentwise_ae_sampling(
    frames: List[np.ndarray],
    ae_scores: List[float],
    sampled_indices: List[int],
    num_frames: int = 16,
) -> List[np.ndarray]:
    n_total = len(frames)
    blank   = np.zeros((224, 224, 3), dtype=np.uint8)

    if n_total == 0:
        return [blank] * num_frames
    if n_total < num_frames:
        padded = list(frames)
        while len(padded) < num_frames:
            padded.append(padded[-1])
        return padded
    if not ae_scores or not sampled_indices:
        step = n_total / num_frames
        return [frames[min(int(i * step), n_total - 1)] for i in range(num_frames)]

    if len(ae_scores) != len(sampled_indices):
        raise ValueError(
            f"segmentwise_ae_sampling: len(ae_scores)={len(ae_scores)} != "
            f"len(sampled_indices)={len(sampled_indices)}."
        )
    for pos, fi in enumerate(sampled_indices):
        if not (0 <= int(fi) < n_total):
            raise ValueError(
                f"segmentwise_ae_sampling: sampled_indices[{pos}]={fi} out of "
                f"range for frames of length {n_total}."
            )

    segment_size = n_total / num_frames
    buckets: List[List[Tuple[int, float]]] = [[] for _ in range(num_frames)]
    for raw_fi, raw_score in zip(sampled_indices, ae_scores):
        fi  = int(raw_fi)
        seg = min(int(fi / segment_size), num_frames - 1)
        buckets[seg].append((fi, float(raw_score)))

    selected: List[np.ndarray] = []
    for seg_idx in range(num_frames):
        seg_start = int(seg_idx * segment_size)
        seg_end   = min(int((seg_idx + 1) * segment_size), n_total)
        if seg_start >= seg_end:
            best_fi = min(seg_start, n_total - 1)
        elif buckets[seg_idx]:
            best_fi = max(buckets[seg_idx], key=lambda x: x[1])[0]
        else:
            best_fi = (seg_start + seg_end) // 2
        selected.append(frames[min(max(best_fi, 0), n_total - 1)])
    return selected


# ══════════════════════════════════════════════════════════════════════════════
# AUTOENCODER MODEL
# ══════════════════════════════════════════════════════════════════════════════
class ResNetAE(nn.Module):
    def __init__(self, latent_dim: int = 512):
        super().__init__()
        r            = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.encoder = nn.Sequential(*list(r.children())[:-2])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_enc  = nn.Linear(2048, latent_dim)
        self.fc_dec  = nn.Linear(latent_dim, 2048)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(2048, 1024, 4, 1, 0), nn.BatchNorm2d(1024), nn.ReLU(),
            nn.ConvTranspose2d(1024, 512,  4, 2, 1), nn.BatchNorm2d(512),  nn.ReLU(),
            nn.ConvTranspose2d(512,  256,  4, 2, 1), nn.BatchNorm2d(256),  nn.ReLU(),
            nn.ConvTranspose2d(256,  128,  4, 2, 1), nn.BatchNorm2d(128),  nn.ReLU(),
            nn.ConvTranspose2d(128,  64,   4, 2, 1), nn.BatchNorm2d(64),   nn.ReLU(),
            nn.ConvTranspose2d(64,   3,    4, 2, 1),
        )
        self.upsample = nn.Upsample((224, 224), mode="bilinear", align_corners=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        f = self.encoder(x)
        l = self.fc_enc(self.avgpool(f).flatten(1))
        r = self.upsample(self.decoder(self.fc_dec(l).view(-1, 2048, 1, 1)))
        return torch.sigmoid(r)


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 0 — AUTOENCODER
# ══════════════════════════════════════════════════════════════════════════════
class Stage0AE:
    def __init__(self, config: Config):
        self.config    = config
        self.model: Optional[ResNetAE] = None
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])

    def load_model(self):
        log.info("[Stage0-AE] Loading autoencoder...")
        self.model = ResNetAE().to(self.config.DEVICE)
        path = self.config.PATHS["AE"]
        if not os.path.exists(path):
            log.warning(f"[Stage0-AE] Checkpoint not found: {path}")
            self.model.eval()
            return self
        ckpt = torch.load(path, map_location=self.config.DEVICE, weights_only=False)
        if isinstance(ckpt, dict):
            for k in ("model_state", "model_state_dict", "state_dict"):
                if k in ckpt:
                    state = ckpt[k]
                    break
            else:
                raise ValueError(f"No valid model state key found: {list(ckpt.keys())}")
        else:
            state = ckpt
        self.model.load_state_dict(state)
        self.model.eval()
        return self

    def analyze(self, video_path: str) -> dict:
        if self.model is None:
            self.load_model()
        log.info("[Stage0-AE] Extracting frames...")
        frames, fps = extract_frames(video_path)
        log.info(f"[Stage0-AE] {len(frames)} frames @ {fps:.1f} FPS")

        sampled_indices = list(range(0, len(frames), self.config.AE_FRAME_STRIDE))
        scores: List[float] = []
        bs = self.config.AE_BATCH_SIZE
        log.info(f"[Stage0-AE] Scoring {len(sampled_indices)} frames (batch={bs})...")

        for batch_start in range(0, len(sampled_indices), bs):
            batch_indices = sampled_indices[batch_start: batch_start + bs]
            tensors = [self.transform(Image.fromarray(frames[i])) for i in batch_indices]
            batch   = torch.stack(tensors).to(self.config.DEVICE)
            with torch.no_grad():
                recon = self.model(batch)
            for i in range(len(batch_indices)):
                scores.append(calculate_enhanced_score(batch[i:i+1], recon[i:i+1]))
            del batch, recon
            cleanup_memory()

        if not scores:
            return {
                "status": "SAFE", "max_error": 0.0, "avg_error": 0.0,
                "peak_frame_idx": 0, "peak_score_idx": 0,
                "frames": frames, "scores": scores, "fps": fps,
                "threshold": self.config.SAFE_THRESHOLD_LEGACY,
                "anomaly_start_frame": 0, "anomaly_end_frame": 0,
                "sampled_indices": sampled_indices,
            }

        threshold      = compute_dynamic_threshold(scores)
        max_error      = float(np.max(scores))
        avg_error      = float(np.mean(scores))
        peak_score_idx = int(np.argmax(scores))
        peak_frame_idx = sampled_indices[peak_score_idx]
        anom_start_si, anom_end_si = find_anomaly_extent(scores, threshold)
        is_anomaly = max_error > threshold or avg_error > threshold * 0.7
        status     = "CRITICAL" if is_anomaly else "SAFE"
        cleanup_memory()
        log.info(f"[Stage0-AE] {status} | max={max_error:.6f} | "
                 f"avg={avg_error:.6f} | threshold={threshold:.6f}")
        return {
            "status":              status,
            "max_error":           max_error,
            "avg_error":           avg_error,
            "peak_frame_idx":      peak_frame_idx,
            "peak_score_idx":      peak_score_idx,
            "frames":              frames,
            "scores":              scores,
            "sampled_indices":     sampled_indices,
            "fps":                 fps,
            "threshold":           threshold,
            "anomaly_start_frame": sampled_indices[anom_start_si],
            "anomaly_end_frame":   sampled_indices[anom_end_si],
        }

    def build_ae_signals(self, scores: List[float], peak_score_idx: int) -> np.ndarray:
        W   = self.config.RL_AE_WINDOW
        eps = 1e-8
        if not scores:
            return np.zeros((W, 6), dtype=np.float32)

        arr   = np.array(scores, dtype=np.float64)
        n     = len(arr)
        delta = np.zeros(n); accel = np.zeros(n)
        delta[1:]  = arr[1:] - arr[:-1]
        accel[2:]  = delta[2:] - delta[1:-1]
        ema_s = compute_ema(arr, self.config.EMA_SHORT_ALPHA)
        ema_l = compute_ema(arr, self.config.EMA_LONG_ALPHA)
        spike = arr / (ema_l + eps)

        full_signals = np.stack([arr, delta, accel, ema_s, ema_l, spike], axis=1)
        for c in range(6):
            full_signals[:, c] = z_score_normalize(full_signals[:, c])

        half_w    = W // 2
        start_idx = max(0, peak_score_idx - half_w)
        end_idx   = min(n, start_idx + W)
        window    = full_signals[start_idx:end_idx]
        if len(window) < W:
            pad    = np.zeros((W - len(window), 6), dtype=np.float32)
            window = np.concatenate([window, pad], axis=0)
        return window[:W].astype(np.float32)


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 1 — TIMESFORMER BINARY
# ══════════════════════════════════════════════════════════════════════════════
class Stage1TSBinary:
    def __init__(self, config: Config):
        self.config = config
        self.model  = None
        self._last_logits: Optional[np.ndarray] = None
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def load_model(self):
        if not TRANSFORMERS_AVAILABLE:
            log.warning("[Stage1-TSBinary] transformers unavailable.")
            return self
        log.info("[Stage1-TSBinary] Loading binary TimeSformer...")
        self.model = TimesformerForVideoClassification.from_pretrained(
            "facebook/timesformer-base-finetuned-k400",
            num_labels=2,
            ignore_mismatched_sizes=True,
        ).to(self.config.DEVICE)
        path = self.config.PATHS["TS_BINARY"]
        if os.path.exists(path):
            raw = torch.load(path, map_location=self.config.DEVICE, weights_only=False)
            if isinstance(raw, dict):
                state = raw.get("model") or raw.get("model_state_dict") or raw
            else:
                state = raw
            self.model.load_state_dict(state, strict=False)
            log.info("[Stage1-TSBinary] Weights loaded.")
        else:
            log.warning(f"[Stage1-TSBinary] Checkpoint not found: {path}")
        self.model.eval()
        return self

    def analyze(self, frames: List[np.ndarray], peak_idx: int) -> dict:
        if self.model is None:
            self.load_model()
        if self.model is None or not frames:
            self._last_logits = np.zeros(2, dtype=np.float32)
            return {"classification": "Unknown", "confidence": 0.0,
                    "prediction_idx": -1, "logits": self._last_logits.tolist()}

        log.info("[Stage1-TSBinary] Binary classification (fixed-interval clip)...")
        clip = frames[max(0, peak_idx - 16): peak_idx + 16]
        while len(clip) < 32:
            clip.append(clip[-1] if clip else np.zeros((224, 224, 3), dtype=np.uint8))

        tensors = torch.stack([self.transform(f) for f in clip]).unsqueeze(0).to(self.config.DEVICE)
        with torch.no_grad():
            outputs = self.model(pixel_values=tensors)
            logits  = outputs.logits
            pred    = torch.argmax(logits, 1).item()
            conf    = torch.softmax(logits, 1).max().item()
            self._last_logits = logits.cpu().numpy().flatten().astype(np.float32)

        classification = "Normal" if pred == 0 else "Suspicious"
        cleanup_memory()
        log.info(f"[Stage1-TSBinary] {classification} ({conf:.2%})")
        return {"classification": classification, "confidence": conf,
                "prediction_idx": pred, "logits": self._last_logits.tolist()}

    def get_logits(self) -> np.ndarray:
        return self._last_logits if self._last_logits is not None else np.zeros(2, dtype=np.float32)


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 2 — TIMESFORMER MULTI-CLASS
# ══════════════════════════════════════════════════════════════════════════════
class _TSMultiWrapper(nn.Module):
    """Replica of TimeSformerAnomalyClassifier from training."""
    def __init__(self, n_classes: int = 7):
        super().__init__()
        self.timesformer = TimesformerForVideoClassification.from_pretrained(
            "facebook/timesformer-base-finetuned-k400",
            num_labels=n_classes,
            ignore_mismatched_sizes=True,
        )
        h = self.timesformer.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(h, h // 2), nn.ReLU(), nn.Dropout(0.2), nn.Linear(h // 2, n_classes),
        )
        self.timesformer.classifier = self.classifier

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        return self.timesformer(pixel_values=pixel_values).logits


class Stage2TSMulti:
    def __init__(self, config: Config):
        self.config    = config
        self.model     = None
        self.processor = None
        self._last_logits: Optional[np.ndarray] = None

    def load_model(self):
        if not TRANSFORMERS_AVAILABLE:
            return self
        log.info("[Stage2-TSMulti] Loading multi-class TimeSformer...")
        path = self.config.PATHS["TS_MULTI"]
        if not os.path.exists(path):
            log.warning(f"[Stage2-TSMulti] Not found: {path}")
            return self
        try:
            raw = torch.load(path, map_location=self.config.DEVICE, weights_only=False)
            state_dict = (raw.get("model_state_dict") or raw.get("model") or
                          raw.get("state_dict") or raw) if isinstance(raw, dict) else raw
            model   = _TSMultiWrapper(n_classes=7)
            cleaned = {k: v for k, v in state_dict.items()
                       if not k.startswith("timesformer.classifier.")}
            n_removed = len(state_dict) - len(cleaned)
            if n_removed:
                log.info(f"[Stage2-TSMulti] Stripped {n_removed} duplicate keys.")
            missing, unexpected = model.load_state_dict(cleaned, strict=False)
            if missing:
                log.warning(f"[Stage2-TSMulti] Missing keys ({len(missing)}): {missing[:3]}")
            if unexpected:
                log.warning(f"[Stage2-TSMulti] Unexpected keys ({len(unexpected)}): {unexpected[:3]}")
            self.model     = model.to(self.config.DEVICE).eval()
            self.processor = VideoMAEImageProcessor.from_pretrained(
                "facebook/timesformer-base-finetuned-k400")
            log.info("[Stage2-TSMulti] Loaded successfully.")
        except Exception as e:
            log.error(f"[Stage2-TSMulti] Load failed: {e}")
            import traceback; traceback.print_exc()
            self.model = None
        return self

    def analyze(self, frames: List[np.ndarray], peak_idx: int,
                ae_scores: Optional[List[float]] = None,
                sampled_indices: Optional[List[int]] = None) -> dict:
        if self.model is None or self.processor is None:
            self.load_model()
        null = {"class": "Unknown", "confidence": 0.0, "top3": [],
                "all_probabilities": {}, "logits": [0.0] * 7, "sampling_method": "unavailable"}
        if self.model is None:
            self._last_logits = np.zeros(7, dtype=np.float32)
            return null

        log.info("[Stage2-TSMulti] Multi-class classification...")
        if ae_scores and sampled_indices:
            clip            = segmentwise_ae_sampling(frames, ae_scores, sampled_indices, 16)
            sampling_method = "segmentwise_ae"
        else:
            start = max(0, peak_idx - 8)
            clip  = frames[start: min(len(frames), peak_idx + 8)]
            while len(clip) < 16:
                clip.append(clip[-1] if clip else np.zeros((224, 224, 3), dtype=np.uint8))
            clip            = clip[:16]
            sampling_method = "fixed_window_fallback"

        pil = [Image.fromarray(f) if isinstance(f, np.ndarray) else f for f in clip]
        try:
            inputs = self.processor(pil, return_tensors="pt")
            pv     = inputs["pixel_values"].to(self.config.DEVICE)
            with torch.no_grad():
                logits = self.model(pixel_values=pv)
                probs  = torch.softmax(logits, dim=-1)[0]
                self._last_logits = logits.cpu().numpy().flatten()[:7].astype(np.float32)
            top3_p, top3_i = torch.topk(probs, min(3, len(probs)))
            cls   = self.config.CLASS_NAMES.get(top3_i[0].item(), "Unknown")
            top3  = [(self.config.CLASS_NAMES.get(i.item(), "Unknown"), p.item())
                     for i, p in zip(top3_i, top3_p)]
            all_p = {self.config.CLASS_NAMES.get(i, "Unknown"): probs[i].item()
                     for i in range(len(probs))}
            cleanup_memory()
            log.info(f"[Stage2-TSMulti] {cls} ({top3_p[0].item():.2%}) via {sampling_method}")
            return {"class": cls, "confidence": top3_p[0].item(),
                    "top3": top3, "all_probabilities": all_p,
                    "logits": self._last_logits.tolist(), "sampling_method": sampling_method}
        except Exception as e:
            log.error(f"[Stage2-TSMulti] Inference failed: {e}")
            self._last_logits = np.zeros(7, dtype=np.float32)
            return {"class": "Error", "confidence": 0.0, "top3": [],
                    "all_probabilities": {}, "logits": self._last_logits.tolist(),
                    "sampling_method": sampling_method}

    def get_logits(self) -> np.ndarray:
        return self._last_logits if self._last_logits is not None else np.zeros(7, dtype=np.float32)


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 3 — YOLO WEAPON DETECTOR (EXPERIMENTAL)
# ══════════════════════════════════════════════════════════════════════════════
class Stage3YOLOWeapon:
    _NULL_RESULT = {
        "weapon_flag":          False,
        "weapon_count":         0,
        "dominant_class":       "None",
        "best_detection":       {},
        "all_detections":       [],
        "top_frames_annotated": [],
        "spatial_heatmap":      None,  # filled at runtime
        "summary":              "YOLO weapon detection unavailable",
    }

    def __init__(self, config: Config):
        self.config = config
        self.model  = None

    def _null(self, summary: str = "") -> dict:
        r = dict(self._NULL_RESULT)
        r["spatial_heatmap"] = np.zeros(196, dtype=np.float32)
        if summary:
            r["summary"] = summary
        return r

    def load_model(self):
        if not YOLO_AVAILABLE:
            return self
        path = self.config.PATHS["YOLO"]
        try:
            self.model = YOLO(path if os.path.exists(path) else "yolov8n.pt")
            log.info(f"[Stage3-YOLO] Loaded from: {path if os.path.exists(path) else 'yolov8n.pt'}")
        except Exception as e:
            log.warning(f"[Stage3-YOLO] Load failed: {e}")
            self.model = None
        return self

    def analyze(self, frames: List[np.ndarray], anomaly_start_frame: int,
                anomaly_end_frame: int, max_frames: int = 32) -> dict:
        if self.model is None:
            self.load_model()
        if self.model is None or not frames:
            return self._null()

        a_start = max(0, anomaly_start_frame)
        a_end   = min(len(frames) - 1, anomaly_end_frame)
        if a_end - a_start < 0:
            return self._null()

        step         = max(1, (a_end - a_start + 1) // max_frames)
        scan_indices = list(range(a_start, a_end + 1, step))[:max_frames]
        log.info(f"[Stage3-YOLO] Scanning {len(scan_indices)} frames in [{a_start},{a_end}]...")

        all_detections: List[dict]           = []
        best_detection: dict                 = {}
        best_conf                            = 0.0
        class_counts: Dict[int, int]         = {0: 0, 1: 0}
        spatial_grid                         = np.zeros(196, dtype=np.float32)
        frame_conf_list: List[Tuple]         = []

        for fi in scan_indices:
            try:
                results_yolo    = self.model(frames[fi], verbose=False)[0]
                frame_has_det   = False
                frame_max_conf  = 0.0
                annotated_frame = frames[fi].copy()
                for box in results_yolo.boxes:
                    cls_idx = int(box.cls[0])
                    if cls_idx not in self.config.WEAPON_CLASS_NAMES:
                        continue
                    conf = float(box.conf[0])
                    x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
                    det = {"class_idx": cls_idx,
                           "class_name": self.config.WEAPON_CLASS_NAMES[cls_idx],
                           "confidence": conf, "frame_idx": fi}
                    all_detections.append(det)
                    class_counts[cls_idx] = class_counts.get(cls_idx, 0) + 1
                    if conf > best_conf:
                        best_conf      = conf
                        best_detection = det
                    h, w = frames[fi].shape[:2]
                    gx   = min(int((x1 + x2) / 2 / max(w, 1) * 14), 13)
                    gy   = min(int((y1 + y2) / 2 / max(h, 1) * 14), 13)
                    spatial_grid[gy * 14 + gx] += conf
                    cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (255, 50, 50), 3)
                    cv2.putText(annotated_frame,
                                f"{self.config.WEAPON_CLASS_NAMES[cls_idx]} {conf:.0%}",
                                (x1, max(y1 - 10, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 50, 50), 2)
                    frame_has_det  = True
                    frame_max_conf = max(frame_max_conf, conf)
                if frame_has_det:
                    frame_conf_list.append((fi, frame_max_conf, annotated_frame))
            except Exception as e:
                log.warning(f"[Stage3-YOLO] Frame {fi} failed: {e}")

        frame_conf_list.sort(key=lambda x: x[1], reverse=True)
        weapon_count   = len(all_detections)
        weapon_flag    = weapon_count > 0
        dom_cls_idx    = max(class_counts, key=class_counts.get)
        dominant_class = self.config.WEAPON_CLASS_NAMES[dom_cls_idx] if weapon_flag else "None"
        s_max          = spatial_grid.max()
        if s_max > 0:
            spatial_grid /= s_max
        summary = (f"Weapons detected, dominant: {dominant_class}, best confidence: {best_conf:.2%}"
                   if weapon_flag else "No weapons detected")
        log.info(f"[Stage3-YOLO] {summary}")
        return {
            "weapon_flag":          weapon_flag,
            "weapon_count":         weapon_count,
            "dominant_class":       dominant_class,
            "best_conf":            best_conf,
            "best_detection":       best_detection,
            "all_detections":       all_detections,
            "top_frames_annotated": [(fi, ann) for fi, _, ann in frame_conf_list[:2]],
            "spatial_heatmap":      spatial_grid,
            "summary":              summary,
        }


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 4 — AUDIO FEATURE EXTRACTION
# ══════════════════════════════════════════════════════════════════════════════
class Stage4Audio:
    _NORM_RANGES = {
        "zcr":               (0.0, 0.5),
        "mfcc_variance":     (0.0, 200.0),
        "spectral_flux":     (0.0, 50.0),
        "rms_energy":        (0.0, 0.5),
        "spectral_centroid": (0.0, 8000.0),
    }

    def __init__(self, config: Config):
        self.config = config

    def _normalize(self, value: float, key: str) -> float:
        lo, hi = self._NORM_RANGES.get(key, (0.0, 1.0))
        return float(np.clip((value - lo) / max(hi - lo, 1e-8), 0.0, 1.0))

    def _null_result(self, status: str) -> dict:
        return {
            "status": status, "audio_present": False,
            "zcr": 0.0, "mfcc_variance": 0.0, "spectral_flux": 0.0,
            "rms_energy": 0.0, "spectral_centroid": 0.0,
            "audio_score": 0.0, "audio_features_vec": np.zeros(5, dtype=np.float32),
        }

    def analyze(self, video_path: str) -> dict:
        log.info("[Stage4-Audio] Extracting audio features...")
        audio_path = extract_audio(video_path)
        if not audio_path:
            return self._null_result("NO_AUDIO")
        if not check_audio_has_content(audio_path):
            log.info("[Stage4-Audio] Silent or absent audio.")
            return self._null_result("SILENT")
        if not LIBROSA_AVAILABLE:
            return self._null_result("LIBROSA_UNAVAILABLE")
        try:
            y, sr = librosa.load(audio_path, sr=22050, duration=30)
            zcr_raw  = float(np.mean(librosa.feature.zero_crossing_rate(y)[0]))
            mfcc_raw = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
            mfcc_var = float(np.mean(np.var(mfcc_raw, axis=1)))
            S        = np.abs(librosa.stft(y))
            flux     = float(np.mean(np.sqrt(np.mean(np.diff(S, axis=1) ** 2, axis=0))))
            rms      = float(np.mean(librosa.feature.rms(y=y)[0]))
            sc       = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)[0]))

            zcr_n  = self._normalize(zcr_raw,  "zcr")
            mfcc_n = self._normalize(mfcc_var,  "mfcc_variance")
            flux_n = self._normalize(flux,       "spectral_flux")
            rms_n  = self._normalize(rms,        "rms_energy")
            sc_n   = self._normalize(sc,         "spectral_centroid")

            audio_score = float(np.clip(
                0.30 * flux_n + 0.25 * mfcc_n + 0.20 * zcr_n +
                0.15 * rms_n  + 0.10 * sc_n, 0.0, 1.0))
            n_anom = sum([flux_n > 0.4, mfcc_n > 0.4, zcr_n > 0.4, rms_n > 0.4])
            status = "ANOMALY" if n_anom >= 2 else "SUSPICIOUS" if n_anom == 1 else "NORMAL"
            log.info(f"[Stage4-Audio] {status} | audio_score={audio_score:.3f}")
            return {
                "status": status, "audio_present": True,
                "zcr": zcr_raw, "mfcc_variance": mfcc_var,
                "spectral_flux": flux, "rms_energy": rms, "spectral_centroid": sc,
                "audio_score": audio_score,
                "audio_features_vec": np.array([zcr_n, mfcc_n, flux_n, rms_n, sc_n], dtype=np.float32),
                "normalized": {"zcr": zcr_n, "mfcc": mfcc_n, "flux": flux_n, "rms": rms_n, "sc": sc_n},
            }
        except Exception as e:
            log.error(f"[Stage4-Audio] Failed: {e}")
            return self._null_result("ERROR")


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 5 — TRACKING / BEHAVIOR EXTRACTION
# ══════════════════════════════════════════════════════════════════════════════
class Stage5Tracking:
    def __init__(self, config: Config):
        self.config = config

    def analyze(self, frames: List[np.ndarray], anomaly_start_frame: int,
                n_frames: int = 32) -> dict:
        null = {"motion_intensity": 0.0, "behaviors": [], "track_count": 0,
                "tracking_features": np.zeros(20, dtype=np.float32), "status": "UNAVAILABLE"}
        if not frames:
            return null
        clip = frames[max(0, anomaly_start_frame): max(0, anomaly_start_frame) + n_frames]
        if len(clip) < 2:
            return null
        if YOLO_AVAILABLE:
            try:
                return self._yolo_track(clip)
            except Exception as e:
                log.warning(f"[Stage5-Tracking] YOLO tracking failed: {e}, falling back to optical flow.")
        return self._optical_flow_track(clip)

    def _yolo_track(self, clip: List[np.ndarray]) -> dict:
        try:
            tracker_model = YOLO("yolov8n.pt")
        except Exception as e:
            raise RuntimeError(f"Cannot load YOLOv8n for tracking: {e}")

        all_tracks: Dict[int, List[Tuple[float, float]]] = {}
        per_frame_counts = []
        for frame in clip:
            try:
                results = tracker_model.track(frame, persist=True, verbose=False)[0]
                if results.boxes.id is None:
                    per_frame_counts.append(0)
                    continue
                per_frame_counts.append(len(results.boxes.id))
                for tid, box in zip(results.boxes.id.int().tolist(),
                                    results.boxes.xyxy.tolist()):
                    cx = (box[0] + box[2]) / 2
                    cy = (box[1] + box[3]) / 2
                    all_tracks.setdefault(int(tid), []).append((cx, cy))
            except Exception:
                per_frame_counts.append(0)

        behaviors, motion_intensity = self._analyze_tracks(all_tracks)
        return {
            "motion_intensity":  motion_intensity,
            "behaviors":         behaviors,
            "track_count":       len(all_tracks),
            "tracking_features": self._build_tracking_features(
                all_tracks, motion_intensity, behaviors, per_frame_counts),
            "status": "YOLO_TRACK",
        }

    def _optical_flow_track(self, clip: List[np.ndarray]) -> dict:
        magnitudes = []
        prev_gray  = cv2.cvtColor(clip[0], cv2.COLOR_RGB2GRAY)
        for frame in clip[1:]:
            gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
            try:
                flow = cv2.calcOpticalFlowFarneback(
                    prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
                mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
                magnitudes.append(float(np.mean(mag)))
            except Exception:
                pass
            prev_gray = gray

        if not magnitudes:
            return {"motion_intensity": 0.0, "behaviors": [], "track_count": 0,
                    "tracking_features": np.zeros(20, dtype=np.float32),
                    "status": "OPTICAL_FLOW_FAILED"}

        mag_arr          = np.array(magnitudes)
        mean_mag         = float(np.mean(mag_arr))
        std_mag          = float(np.std(mag_arr))
        motion_intensity = float(np.clip(mean_mag / 15.0, 0.0, 1.0))
        behaviors        = []
        if mean_mag > 8.0:
            behaviors.append("FAST_MOVEMENT")
        elif mean_mag < 1.5:
            behaviors.append("STATIC")
        if std_mag > mean_mag * 0.5 and mean_mag > 3.0:
            behaviors.append("CLUSTERING")

        feat    = np.zeros(20, dtype=np.float32)
        feat[0] = float(np.clip(mean_mag / 15.0, 0, 1))
        feat[1] = float(np.clip(float(np.max(mag_arr)) / 30.0, 0, 1))
        feat[2] = float(np.clip(std_mag / 10.0, 0, 1))
        feat[3] = float("FAST_MOVEMENT" in behaviors)
        feat[4] = float("STATIC" in behaviors)
        feat[5] = float("CLUSTERING" in behaviors)
        series_len = min(14, len(magnitudes))
        feat[6:6 + series_len] = np.clip(
            np.array(magnitudes[:series_len]) / 15.0, 0, 1).astype(np.float32)
        return {"motion_intensity": motion_intensity, "behaviors": behaviors,
                "track_count": 0, "tracking_features": feat, "status": "OPTICAL_FLOW"}

    @staticmethod
    def _analyze_tracks(tracks: Dict[int, List[Tuple[float, float]]]) -> Tuple[List[str], float]:
        behaviors, all_displacements = [], []
        tids = list(tracks.keys())
        for pts in tracks.values():
            if len(pts) < 2:
                continue
            disp = [math.hypot(pts[i][0]-pts[i-1][0], pts[i][1]-pts[i-1][1])
                    for i in range(1, len(pts))]
            all_displacements.extend(disp)
            avg = sum(disp) / max(len(disp), 1)
            if avg > 8.0:
                behaviors.append("FAST_MOVEMENT")
            if avg < 2.0 and len(pts) >= 10:
                behaviors.append("STATIC")
        for i in range(len(tids)):
            for j in range(i + 1, len(tids)):
                a, b = tracks[tids[i]], tracks[tids[j]]
                n    = min(len(a), len(b))
                if n < 6:
                    continue
                dists = [math.hypot(a[k][0]-b[k][0], a[k][1]-b[k][1]) for k in range(n)]
                dec   = sum(1 for k in range(1, n) if dists[k] < dists[k-1])
                if dec / n > 0.75 and dists[0] - dists[-1] > 50:
                    behaviors.append("CLUSTERING")
                    break
        behaviors        = list(dict.fromkeys(behaviors)) or ["STATIC"]
        motion_intensity = float(np.clip(np.mean(all_displacements) / 15.0, 0.0, 1.0)) if all_displacements else 0.0
        return behaviors, motion_intensity

    @staticmethod
    def _build_tracking_features(tracks, motion_intensity, behaviors, per_frame_counts) -> np.ndarray:
        feat    = np.zeros(20, dtype=np.float32)
        feat[0] = float(np.clip(motion_intensity, 0, 1))
        feat[1] = float(np.clip(len(tracks) / 10.0, 0, 1))
        feat[2] = float("FAST_MOVEMENT" in behaviors)
        feat[3] = float("STATIC" in behaviors)
        feat[4] = float("CLUSTERING" in behaviors)
        feat[5] = float(np.clip(np.mean(per_frame_counts) / 10.0, 0, 1)) if per_frame_counts else 0.0
        all_v   = [math.hypot(pts[i][0]-pts[i-1][0], pts[i][1]-pts[i-1][1])
                   for pts in tracks.values() for i in range(1, len(pts)) if len(pts) >= 2]
        feat[6] = float(np.clip(np.std(all_v) / 10.0, 0, 1)) if all_v else 0.0
        series_len  = min(13, len(per_frame_counts))
        feat[7:7 + series_len] = np.clip(
            np.array(per_frame_counts[:series_len]) / 10.0, 0, 1).astype(np.float32)
        return feat


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 6 — FUSION LAYER
# ══════════════════════════════════════════════════════════════════════════════
class Stage6Fusion:
    def __init__(self, config: Config):
        self.config = config

    def fuse(self, ae_result: dict, ts_binary_result: dict, ts_multi_result: dict,
             yolo_result: dict, audio_result: dict, tracking_result: dict) -> dict:
        reasoning_trace: List[str] = []
        threshold     = ae_result.get("threshold", self.config.SAFE_THRESHOLD_LEGACY)
        ae_max        = float(ae_result.get("max_error", 0.0))
        ae_score_norm = float(np.clip(ae_max / max(threshold, 1e-8), 0.0, 1.0))

        ts_binary_susp = (ts_binary_result.get("classification") == "Suspicious")
        ts_binary_conf = float(ts_binary_result.get("confidence", 0.0) if ts_binary_susp else 0.0)
        weapon_flag    = bool(yolo_result.get("weapon_flag", False))
        audio_score    = float(audio_result.get("audio_score", 0.0))
        audio_anomaly  = (audio_result.get("status") == "ANOMALY")
        motion_int     = float(tracking_result.get("motion_intensity", 0.0))
        fast_movement  = "FAST_MOVEMENT" in tracking_result.get("behaviors", [])

        # Rule 0: base AE weight
        fused_risk = ae_score_norm * 0.50
        reasoning_trace.append(
            f"Baseline: AE normalised score is {ae_score_norm:.3f} (50% weight), "
            f"giving initial base risk of {fused_risk:.3f}.")

        # Rule 1: weapon + AE hard escalation
        if weapon_flag and ae_max > threshold:
            fused_risk = max(fused_risk, 0.90)
            reasoning_trace.append(
                "Rule 1 TRIGGERED: Weapon detected AND AE exceeds threshold. "
                "Risk escalated to CRITICAL (minimum 0.90).")
        elif ts_binary_susp:
            ts_boost    = ts_binary_conf * 0.20
            fused_risk += ts_boost
            reasoning_trace.append(
                f"Rule 2: Binary TimeSformer classified as Suspicious at "
                f"{ts_binary_conf:.2f} confidence (20% weight), adding {ts_boost:.3f}.")

        # Rule 3: motion + audio boosts
        if fast_movement and audio_anomaly and ae_max > threshold:
            escalation  = self.config.FUSION_MOTION_BOOST + self.config.FUSION_AUDIO_BOOST
            fused_risk += escalation
            reasoning_trace.append(
                f"Rule 3 FULLY TRIGGERED: Fast movement, audio anomaly, and AE anomaly "
                f"all present. Adding {escalation:.3f}.")
        elif fast_movement:
            boost       = self.config.FUSION_MOTION_BOOST * 0.5
            fused_risk += boost
            reasoning_trace.append(f"Rule 3 (partial): Fast movement detected. Adding {boost:.3f}.")
        elif audio_anomaly:
            boost       = self.config.FUSION_AUDIO_BOOST * 0.5
            fused_risk += boost
            reasoning_trace.append(f"Rule 3 (partial): Audio anomaly detected. Adding {boost:.3f}.")

        # Rule 4: no support penalty
        has_support = ts_binary_susp or weapon_flag or audio_anomaly or fast_movement
        if ae_max > threshold and not has_support:
            fused_risk -= self.config.FUSION_NO_SUPPORT_PENALTY
            reasoning_trace.append(
                f"Rule 4: AE exceeds threshold but no supporting signal. "
                f"Applying deduction of {self.config.FUSION_NO_SUPPORT_PENALTY:.3f}.")

        # Rule 5: weapon floor
        if weapon_flag:
            pre_floor  = fused_risk
            fused_risk = max(fused_risk, self.config.FUSION_WEAPON_BOOST)
            if fused_risk != pre_floor:
                reasoning_trace.append(
                    f"Rule 5: Weapon floor applied at {self.config.FUSION_WEAPON_BOOST:.2f}.")

        fused_risk = float(np.clip(fused_risk, 0.0, 1.0))
        if weapon_flag and ae_max > threshold:
            risk_level = "CRITICAL"
        elif fused_risk >= 0.70:
            risk_level = "HIGH"
        elif fused_risk >= 0.40:
            risk_level = "MODERATE"
        else:
            risk_level = "LOW"
        reasoning_trace.append(
            f"Final: fused risk is {fused_risk:.4f}, classified as {risk_level}.")

        context_vector = np.array([
            float(np.clip(ae_score_norm, 0.0, 1.0)),
            float(ts_binary_conf),
            float(np.clip(audio_score,  0.0, 1.0)),
            float(weapon_flag),
            float(np.clip(motion_int,   0.0, 1.0)),
        ], dtype=np.float32)

        log.info(f"[Stage6-Fusion] {risk_level} | fused_risk={fused_risk:.4f}")
        return {
            "fused_risk":       fused_risk,
            "risk_level":       risk_level,
            "reasoning_trace":  reasoning_trace,
            "context_vector":   context_vector,
            "ae_score_norm":    ae_score_norm,
            "ts_binary_conf":   ts_binary_conf,
            "audio_score":      audio_score,
            "weapon_flag":      weapon_flag,
            "motion_intensity": motion_int,
        }


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 7 — RL3033 (EXPERIMENTAL)
# ══════════════════════════════════════════════════════════════════════════════
class Stage7RL3033:
    def __init__(self, config: Config):
        self.config = config
        self.agent  = None

    def load_agent(self):
        if not RL_AVAILABLE:
            log.warning("[Stage7-RL3033] stable-baselines3 unavailable.")
            return self
        path = self.config.PATHS["RL_AGENT"]
        if not os.path.exists(path):
            log.warning(f"[Stage7-RL3033] Agent not found: {path}")
            return self
        log.info("[Stage7-RL3033] Loading RL3033 agent...")
        try:
            self.agent = PPO.load(path, device="cpu")
            log.info(f"[Stage7-RL3033] Loaded. obs_space={self.agent.observation_space}")
        except Exception as e:
            log.error(f"[Stage7-RL3033] Load failed: {e}")
            self.agent = None
        return self

    def _ch_stats(self, ch: np.ndarray) -> np.ndarray:
        return np.array([
            np.mean(ch), np.std(ch), np.max(ch), np.min(ch),
            np.percentile(ch, 75), np.percentile(ch, 25),
            np.median(ch), float(np.sum(ch > 0)) / max(len(ch), 1),
        ], dtype=np.float32)

    def _build_latent_motion(self, ae_signals: np.ndarray, tracking_result: dict) -> np.ndarray:
        ae_flat    = ae_signals.flatten()
        track_feat = tracking_result.get("tracking_features", np.zeros(20, dtype=np.float32))
        ch_desc    = np.concatenate([self._ch_stats(ae_signals[:, c]) for c in [0, 1, 2, 5]])
        motion_int = float(tracking_result.get("motion_intensity", 0.0))
        behaviors  = tracking_result.get("behaviors", [])
        motion_sum = np.array([
            motion_int,
            float("FAST_MOVEMENT" in behaviors),
            float("STATIC"        in behaviors),
            float("CLUSTERING"    in behaviors),
            float(np.clip(tracking_result.get("track_count", 0) / 10.0, 0, 1)),
        ], dtype=np.float32)
        base = np.concatenate([ae_flat, ch_desc, track_feat, motion_sum])
        return np.tile(base, math.ceil(1536 / len(base)))[:1536].astype(np.float32)

    def _build_latent_stack(self, ae_signals, ts_binary_logits, ts_multi_logits,
                             audio_result, tracking_result, fusion_result) -> np.ndarray:
        fusion_signals = np.array([
            float(fusion_result.get("fused_risk",       0.0)),
            float(fusion_result.get("ae_score_norm",    0.0)),
            float(fusion_result.get("ts_binary_conf",   0.0)),
            float(fusion_result.get("audio_score",      0.0)),
            float(fusion_result.get("weapon_flag",      False)),
            float(fusion_result.get("motion_intensity", 0.0)),
            float(fusion_result.get("risk_level") == "CRITICAL"),
            float(fusion_result.get("risk_level") == "HIGH"),
        ], dtype=np.float32)
        base = np.concatenate([
            ae_signals.flatten().astype(np.float32),
            ts_binary_logits.astype(np.float32),
            ts_multi_logits[:7].astype(np.float32),
            audio_result.get("audio_features_vec", np.zeros(5, dtype=np.float32)),
            tracking_result.get("tracking_features", np.zeros(20, dtype=np.float32)),
            fusion_result.get("context_vector", np.zeros(5, dtype=np.float32)),
            fusion_signals,
        ])
        return np.tile(base, math.ceil(2560 / len(base)))[:2560].astype(np.float32)

    def _build_spatial(self, yolo_result: dict, tracking_result: dict,
                        frames: List[np.ndarray], peak_frame_idx: int) -> np.ndarray:
        spatial      = np.zeros(196, dtype=np.float32)
        spatial     += yolo_result.get("spatial_heatmap", np.zeros(196, dtype=np.float32)) * 0.5
        if frames and peak_frame_idx < len(frames):
            gray = cv2.cvtColor(frames[peak_frame_idx], cv2.COLOR_RGB2GRAY).astype(np.float32)
            H, W = gray.shape
            bh, bw = H // 14, W // 14
            if bh > 0 and bw > 0:
                for gy in range(14):
                    for gx in range(14):
                        spatial[gy*14 + gx] += float(np.var(gray[gy*bh:(gy+1)*bh, gx*bw:(gx+1)*bw])) / 10000.0
        motion_int = float(tracking_result.get("motion_intensity", 0.0))
        for cy in range(4, 10):
            for cx in range(4, 10):
                spatial[cy*14 + cx] += motion_int * 0.3
        s_max = spatial.max()
        if s_max > 0:
            spatial /= s_max
        return spatial

    def _build_observation(self, ae_signals, ts_binary_logits, ts_multi_logits,
                            yolo_result, audio_result, tracking_result,
                            fusion_result, frames, peak_frame_idx) -> dict:
        obs = {
            "ae_signals":    ae_signals.astype(np.float32),
            "latent_motion": self._build_latent_motion(ae_signals, tracking_result),
            "latent_stack":  self._build_latent_stack(ae_signals, ts_binary_logits,
                                                       ts_multi_logits, audio_result,
                                                       tracking_result, fusion_result),
            "spatial":       self._build_spatial(yolo_result, tracking_result, frames, peak_frame_idx),
            "context":       fusion_result.get("context_vector", np.zeros(5, dtype=np.float32)).astype(np.float32),
        }
        expected = {
            "ae_signals":    (self.config.RL_AE_WINDOW, self.config.RL_AE_CHANNELS),
            "latent_motion": (self.config.RL_LATENT_MOT,),
            "latent_stack":  (self.config.RL_LATENT_STK,),
            "spatial":       (self.config.RL_SPATIAL_DIM,),
            "context":       (self.config.RL_CONTEXT_DIM,),
        }
        for key, exp_shape in expected.items():
            if obs[key].shape != exp_shape:
                flat   = obs[key].flatten()
                needed = exp_shape[0] if len(exp_shape) == 1 else exp_shape[0] * exp_shape[1]
                resized = flat[:needed] if len(flat) >= needed else np.pad(flat, (0, needed - len(flat)))
                obs[key] = resized.reshape(exp_shape) if len(exp_shape) == 2 else resized
        return obs

    def _fallback_result(self, fusion_result: dict, label: str = "") -> dict:
        fused_risk  = float(fusion_result.get("fused_risk", 0.0))
        tfb_score   = float(fusion_result.get("ts_binary_conf", 0.0))
        tfm_score   = float(fusion_result.get("ae_score_norm", 0.0))
        final_risk  = float(np.clip(
            0.70 * tfb_score + 0.20 * tfm_score + 0.10 * fused_risk, 0.0, 1.0))
        if fusion_result.get("weapon_flag", False):
            final_risk = max(final_risk, 0.85)
        # Deterministic dummy RL value derived from fused/final risk — never random
        rl_raw = float(np.clip(0.5 * fused_risk + 0.5 * final_risk, 0.0, 1.0))
        status = ("CRITICAL" if final_risk >= 0.85 else "HIGH_RISK" if final_risk >= 0.65
                  else "MODERATE" if final_risk >= 0.40 else "LOW_RISK")
        return {
            "final_risk": final_risk, "fused_risk": fused_risk, "rl_raw": rl_raw, "status": status,
            "interpretation": (
                f"{label}Final risk from weighted formula: "
                f"70% TFB ({tfb_score:.2%}) + 20% TFM ({tfm_score:.2%}) + "
                f"10% Fused ({fused_risk:.2%}) = {final_risk:.2%} [{status}]. "
                f"RL component approximated deterministically as {rl_raw:.2%} "
                f"(average of fused and final risk)."
            ),
        }

    def analyze(self, ae_signals, ts_binary_logits, ts_multi_logits,
                yolo_result, audio_result, tracking_result,
                fusion_result, frames, peak_frame_idx) -> dict:
        if self.agent is None:
            self.load_agent()
        if self.agent is None:
            return self._fallback_result(fusion_result, label="RL agent unavailable. ")

        try:
            obs = self._build_observation(ae_signals, ts_binary_logits, ts_multi_logits,
                                           yolo_result, audio_result, tracking_result,
                                           fusion_result, frames, peak_frame_idx)
            action, _ = self.agent.predict(obs, deterministic=True)
            action_arr = np.asarray(action, dtype=np.float64).flatten()
            rl_raw     = float(np.clip(action_arr[0] if action_arr.size else 0.0, 0.0, 1.0))

            tfb_score   = float(fusion_result.get("ts_binary_conf", 0.0))
            tfm_score   = float(fusion_result.get("ae_score_norm",  0.0))
            fused_score = float(np.clip(float(fusion_result.get("fused_risk", 0.0)), 0.0, 1.0))
            final_risk  = float(np.clip(
                0.70 * tfb_score + 0.20 * tfm_score + 0.05 * fused_score + 0.05 * rl_raw,
                0.0, 1.0))
            if fusion_result.get("weapon_flag", False):
                final_risk = max(final_risk, 0.85)

            status = ("CRITICAL" if final_risk >= 0.85 else "HIGH_RISK" if final_risk >= 0.65
                      else "MODERATE" if final_risk >= 0.40 else "LOW_RISK")
            log.info(f"[Stage7-RL3033] tfb={tfb_score:.4f} tfm={tfm_score:.4f} "
                     f"fused={fused_score:.4f} rl={rl_raw:.4f} final={final_risk:.4f} => {status}")
            return {
                "fused_risk":     float(fusion_result.get("fused_risk", 0.0)),
                "final_risk":     final_risk,
                "status":         status,
                "tfb_score":      tfb_score,
                "tfm_score":      tfm_score,
                "rl_raw":         rl_raw,
                "interpretation": (
                    f"Final risk: 70% TFB ({tfb_score:.2%}) + 20% TFM ({tfm_score:.2%}) + "
                    f"5% Fused ({fused_score:.2%}) + 5% RL ({rl_raw:.2%}) = {final_risk:.2%} [{status}]."
                ),
            }
        except Exception as e:
            import traceback
            log.error(f"[Stage7-RL3033] Inference failed: {e}")
            traceback.print_exc()
            result = self._fallback_result(fusion_result, label="RL inference failed, using deterministic fallback. ")
            result["status"] = "FALLBACK"
            result["error"]  = str(e)
            return result

# ══════════════════════════════════════════════════════════════════════════════
# STAGE 8 — LLAVA (EXPERIMENTAL)
# Always runs when enable_llava=True. Neutral prompts regardless of Stage 1 result.
# ══════════════════════════════════════════════════════════════════════════════
class Stage8LLaVA:
    _TEMPORAL_LABELS = {
        "baseline":   "Frame 1 of 4 (beginning of clip)",
        "anom_start": "Frame 2 of 4 (area of elevated AE error onset)",
        "peak":       "Frame 3 of 4 (highest AE reconstruction error)",
        "anom_end":   "Frame 4 of 4 (end of elevated-error region)",
    }
    _ROLE_INSTRUCTIONS = {
        "baseline":   "Describe the scene at the start of the clip: who is present and what is happening.",
        "anom_start": "Describe what is visible at the onset of the region with elevated reconstruction error.",
        "peak":       "Describe what is most notable in this frame at the reconstruction error peak.",
        "anom_end":   "Describe how the scene looks at the end of the elevated-error region.",
    }

    def __init__(self, config: Config):
        self.config    = config
        self.model     = None
        self.processor = None

    def load_model(self):
        if not TRANSFORMERS_AVAILABLE:
            return self
        path = self.config.PATHS["LLAVA"]
        if not os.path.exists(path):
            log.warning(f"[Stage8-LLaVA] Model not found: {path}")
            return self
        log.info("[Stage8-LLaVA] Loading Fire-LLaVA...")
        try:
            self.model = LlavaForConditionalGeneration.from_pretrained(
                path, torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True)
            self.processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
            self.processor.patch_size                     = 14
            self.processor.vision_feature_select_strategy = "default"
            self.model.eval()
            log.info("[Stage8-LLaVA] Loaded.")
        except Exception as e:
            log.error(f"[Stage8-LLaVA] Load failed: {e}")
            self.model = None
        return self

    def _build_prompt(self, role: str, frame_number: int, yolo_summary: str,
                      motion_behaviors: List[str], audio_status: str,
                      audio_score: float, binary_result: str) -> str:
        t_label       = self._TEMPORAL_LABELS.get(role, f"Frame {frame_number} of 4")
        behaviors_str = ", ".join(motion_behaviors) if motion_behaviors else "STATIC"
        weapon_info   = yolo_summary if "weapon" in yolo_summary.lower() else "No weapons detected"
        instruction   = self._ROLE_INSTRUCTIONS.get(role, "Describe what you see.")
        return (
            f"USER: <image>\n"
            f"You are a forensic security analyst reviewing surveillance footage.\n\n"
            f"=== TEMPORAL CONTEXT ===\n"
            f"Position: {t_label}\n"
            f"Classifier note: The binary classifier result is: {binary_result}. "
            f"Use this as context, but form your own description from the image.\n\n"
            f"=== DETECTED INFORMATION ===\n"
            f"Objects/Weapons: {weapon_info}\n"
            f"Motion Pattern: {behaviors_str}\n"
            f"Audio Status: {audio_status} (score={audio_score:.2f})\n\n"
            f"=== YOUR TASK ===\n"
            f"{instruction}\n"
            f"Be concise (2-3 sentences). Focus on safety-relevant details.\n"
            f"ASSISTANT:"
        )

    def _describe_frame(self, image: Image.Image, role: str, frame_number: int,
                         context: dict, binary_result: str) -> str:
        prompt = self._build_prompt(
            role=role, frame_number=frame_number,
            yolo_summary=context.get("yolo_summary", "Unknown"),
            motion_behaviors=context.get("behaviors", []),
            audio_status=context.get("audio_status", "Unknown"),
            audio_score=context.get("audio_score", 0.0),
            binary_result=binary_result,
        )
        inputs = self.processor(text=prompt, images=image.convert("RGB"), return_tensors="pt")
        inputs = {k: v.to(self.config.DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            with torch.amp.autocast("cuda"):
                output_ids = self.model.generate(
                    **inputs, max_new_tokens=130, do_sample=True, temperature=0.7,
                    top_p=0.9, top_k=50, repetition_penalty=1.4,
                    no_repeat_ngram_size=3, early_stopping=True)
        text = self.processor.batch_decode(output_ids, skip_special_tokens=True)[0]
        if "ASSISTANT:" in text:
            text = text.split("ASSISTANT:")[-1].strip()
        return self._clean_text(text)

    @staticmethod
    def _clean_text(text: str) -> str:
        if not text or len(text) < 20:
            return text
        seen, out = set(), []
        for s in text.replace("!", ".").replace("?", ".").split("."):
            s   = s.strip()
            key = s.lower()
            if s and len(s) > 10 and key not in seen:
                seen.add(key)
                out.append(s)
        result = ". ".join(out)
        return result + ("." if result and not result.endswith(".") else "")

    @staticmethod
    def synthesise_narrative(frame_descriptions: dict, context: dict,
                              yolo_result: dict, audio_result: dict,
                              tracking_result: dict, binary_result: str = "Unknown") -> str:
        roles    = ("baseline", "anom_start", "peak", "anom_end")
        raw_desc = [frame_descriptions.get(r, "").strip() for r in roles]

        # Deduplicate sentences across all four descriptions
        seen_keys: set         = set()
        unique_parts: List[str] = []
        for raw in raw_desc:
            for sent in raw.replace("!", ".").replace("?", ".").split("."):
                sent = sent.strip()
                key  = " ".join(sent.lower().split())
                if len(sent) > 15 and key not in seen_keys:
                    seen_keys.add(key)
                    unique_parts.append(sent)

        # Build narrative: opening -> onset -> peak -> resolution
        narrative_parts: List[str] = []
        existing_keys = lambda: {" ".join(p.lower().split()) for p in narrative_parts}
        for raw in raw_desc:
            if not raw:
                continue
            key = " ".join(raw.lower().split())
            if key not in existing_keys():
                narrative_parts.append(raw)

        if not narrative_parts:
            return "Visual analysis did not produce a usable scene description for this clip."

        # Append system-derived context notes
        context_notes = []
        weapon_flag  = bool(yolo_result.get("weapon_flag", False))
        weapon_class = yolo_result.get("dominant_class", "None")
        weapon_conf  = float(yolo_result.get("best_conf", 0.0))
        audio_status = audio_result.get("status", "UNKNOWN")
        audio_score  = float(audio_result.get("audio_score", 0.0))
        behaviors    = tracking_result.get("behaviors", ["STATIC"])
        motion_int   = float(tracking_result.get("motion_intensity", 0.0))

        if weapon_flag:
            context_notes.append(
                f"Instrument detection flagged a {weapon_class} with {weapon_conf:.0%} confidence.")
        if audio_status in ("ANOMALY", "SUSPICIOUS"):
            context_notes.append(
                f"The audio track registered as {audio_status} (composite score {audio_score:.2f}), "
                f"suggesting acoustically unusual activity.")
        if "FAST_MOVEMENT" in behaviors:
            context_notes.append(
                f"Motion analysis recorded rapid subject movement (intensity {motion_int:.2f}), "
                f"consistent with an escalating incident.")
        elif "CLUSTERING" in behaviors:
            context_notes.append(
                f"Motion analysis detected converging trajectories (intensity {motion_int:.2f}), "
                f"a pattern associated with confrontational proximity.")
        elif motion_int < 0.15:
            context_notes.append(
                f"Motion analysis recorded minimal movement (intensity {motion_int:.2f}), "
                f"indicating a static scene or post-event lull.")

        combined = " ".join(narrative_parts)
        if context_notes:
            combined = combined.rstrip(".") + ". " + " ".join(context_notes)
        combined = combined.strip()
        if combined and not combined.endswith("."):
            combined += "."
        return combined

    def analyze(self, frames: List[np.ndarray], peak_idx: int,
                anomaly_start_frame: int, anomaly_end_frame: int,
                yolo_result: dict, audio_result: dict, tracking_result: dict,
                binary_result: str = "Unknown") -> dict:
        if self.model is None or self.processor is None:
            self.load_model()
        if self.model is None or not frames:
            return {"description": "LLaVA not available", "analysis": "",
                    "frame_descriptions": {}, "unified_narrative": "LLaVA not available"}

        if anomaly_end_frame < 0:
            anomaly_end_frame = min(peak_idx + 30, len(frames) - 1)
        context = {
            "yolo_summary": yolo_result.get("summary",  "No weapon detection"),
            "behaviors":    tracking_result.get("behaviors", ["STATIC"]),
            "audio_status": audio_result.get("status",   "UNKNOWN"),
            "audio_score":  float(audio_result.get("audio_score", 0.0)),
        }
        keyframes = {
            "baseline":   (frames[0],                                       1),
            "anom_start": (frames[min(anomaly_start_frame, len(frames)-1)], 2),
            "peak":       (frames[min(peak_idx,            len(frames)-1)], 3),
            "anom_end":   (frames[min(anomaly_end_frame,   len(frames)-1)], 4),
        }
        log.info(f"[Stage8-LLaVA] 4-frame analysis (binary={binary_result})...")
        descriptions = {}
        for role, (frame_arr, fn) in keyframes.items():
            log.info(f"[Stage8-LLaVA]   Describing {role} ({fn}/4)...")
            try:
                img               = Image.fromarray(frame_arr) if isinstance(frame_arr, np.ndarray) else frame_arr
                descriptions[role] = self._describe_frame(img, role, fn, context, binary_result)
            except Exception as e:
                log.warning(f"[Stage8-LLaVA] Frame '{role}' failed: {e}")
                descriptions[role] = f"[{role} analysis unavailable]"

        unified_narrative = self.synthesise_narrative(
            frame_descriptions=descriptions, context=context,
            yolo_result=yolo_result, audio_result=audio_result,
            tracking_result=tracking_result, binary_result=binary_result,
        )
        weapon_str = (f"WEAPON DETECTED: {yolo_result.get('dominant_class','?')}"
                      if yolo_result.get("weapon_flag") else "No weapons detected")
        motion_str = ", ".join(tracking_result.get("behaviors", ["STATIC"]))
        audio_str  = f"{context['audio_status']} (score={context['audio_score']:.2f})"
        unified = (
            f"[CONTEXT] {weapon_str}, Motion: {motion_str}, Audio: {audio_str}\n"
            f"[FRAME 1 (BEGINNING)] {descriptions.get('baseline','')}\n"
            f"[FRAME 2 (AE ONSET)]  {descriptions.get('anom_start','')}\n"
            f"[FRAME 3 (AE PEAK)]   {descriptions.get('peak','')}\n"
            f"[FRAME 4 (AE END)]    {descriptions.get('anom_end','')}"
        ).strip()
        cleanup_memory()
        return {"description": unified, "analysis": unified,
                "frame_descriptions": descriptions, "unified_narrative": unified_narrative,
                "context_used": context, "binary_result": binary_result}


# ══════════════════════════════════════════════════════════════════════════════
# ALERT SYSTEM
# ══════════════════════════════════════════════════════════════════════════════
class AlertSystem:
    def __init__(self, config: Config):
        self.config    = config
        self._handlers: List[Any] = []
        self.register(self._default_log_handler)

    def register(self, handler) -> "AlertSystem":
        self._handlers.append(handler)
        return self

    def evaluate_and_fire(self, results: dict) -> Optional[dict]:
        stage7     = results.get("stage7") or {}
        stage6     = results.get("stage6") or {}
        stage3     = results.get("stage3") or {}
        stage4     = results.get("stage4") or {}
        final_risk = float(stage7.get("final_risk", stage6.get("fused_risk", 0.0)))
        level = ("HIGH"   if final_risk >= self.config.ALERT_RISK_HIGH   else
                 "MEDIUM" if final_risk >= self.config.ALERT_RISK_MEDIUM  else
                 "LOW")
        if level == "LOW":
            return None
        alert = {
            "timestamp":       datetime.utcnow().isoformat() + "Z",
            "level":           level,
            "final_risk":      round(final_risk, 4),
            "fused_risk":      round(float(stage6.get("fused_risk",    0.0)), 4),
            "weapon_flag":     bool(stage3.get("weapon_flag", False)),
            "weapon_summary":  stage3.get("summary", ""),
            "audio_score":     round(float(stage4.get("audio_score", 0.0)), 4),
            "risk_level":      stage6.get("risk_level", "UNKNOWN"),
            "reasoning_trace": stage6.get("reasoning_trace", []),
            "rl_status":       stage7.get("status", "UNKNOWN"),
            "video_path":      results.get("video_path", ""),
        }
        for handler in self._handlers:
            try:
                handler(alert)
            except Exception as e:
                log.error(f"[AlertSystem] Handler failed: {e}")
        return alert

    @staticmethod
    def _default_log_handler(alert: dict):
        log.warning(
            f"[ALERT] {alert['level']}, "
            f"final_risk={alert['final_risk']:.2%}, "
            f"fused_risk={alert['fused_risk']:.2%}, "
            f"weapon={'YES' if alert['weapon_flag'] else 'NO'}, "
            f"rl_status={alert['rl_status']}"
        )

    @staticmethod
    def file_alert_handler(log_path: str = "/tmp/scenesolver_alerts.jsonl"):
        def _handler(alert: dict):
            with open(log_path, "a") as f:
                f.write(json.dumps(alert) + "\n")
        return _handler


# ══════════════════════════════════════════════════════════════════════════════
# REAL-TIME STREAM PROCESSOR
# ══════════════════════════════════════════════════════════════════════════════
class RealTimeStream:
    def __init__(self, pipeline: "SceneSolverPipeline"):
        self.pipeline  = pipeline
        self._thread   = None
        self._stop_evt = threading.Event()

    def start(self, source: str, window_frames: int = 50, on_alert: Any = None):
        self._stop_evt.clear()
        self._thread = threading.Thread(
            target=self._run, args=(source, window_frames, on_alert),
            daemon=True, name="RealTimeStream")
        self._thread.start()
        log.info(f"[RealTimeStream] Started: {source}")

    def stop(self):
        self._stop_evt.set()
        if self._thread:
            self._thread.join(timeout=10)
        log.info("[RealTimeStream] Stopped.")

    def _run(self, source: str, window_frames: int, on_alert: Any):
        cfg      = self.pipeline.config
        ae_stage = self.pipeline.stage0
        ae_stage.load_model()
        alert_sys = self.pipeline.alert_system
        transform = ae_stage.transform
        cap       = cv2.VideoCapture(int(source) if source.isdigit() else source)
        if not cap.isOpened():
            log.error(f"[RealTimeStream] Cannot open: {source}")
            return

        buffer: List[np.ndarray] = []
        scores: List[float]      = []
        frame_count              = 0

        while not self._stop_evt.is_set():
            ret, frame = cap.read()
            if not ret:
                break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            buffer.append(frame_rgb)
            frame_count += 1

            if frame_count % cfg.AE_FRAME_STRIDE == 0 and ae_stage.model:
                tensor = transform(Image.fromarray(frame_rgb)).unsqueeze(0).to(cfg.DEVICE)
                with torch.no_grad():
                    recon = ae_stage.model(tensor)
                scores.append(calculate_enhanced_score(tensor, recon))
                del tensor, recon

            if len(buffer) >= window_frames:
                threshold = compute_dynamic_threshold(scores)
                max_err   = max(scores) if scores else 0.0
                if max_err > threshold:
                    risk = float(np.clip(max_err / max(threshold, 1e-8), 0, 1))
                    mini_results = {
                        "stage0": {"max_error": max_err},
                        "stage6": {"fused_risk": risk},
                        "stage7": {"final_risk": risk, "status": "STREAM"},
                        "stage3": {}, "stage4": {},
                        "threshold": threshold, "video_path": source, "output_directory": "",
                    }
                    alert = alert_sys.evaluate_and_fire(mini_results)
                    if alert and on_alert:
                        try:
                            on_alert(alert)
                        except Exception as e:
                            log.error(f"[RealTimeStream] on_alert failed: {e}")
                half   = window_frames // 2
                buffer = buffer[-half:]
                scores = scores[-(half // cfg.AE_FRAME_STRIDE):]

        cap.release()
        log.info("[RealTimeStream] Stream ended.")


# ══════════════════════════════════════════════════════════════════════════════
# PDF REPORT GENERATOR
# ══════════════════════════════════════════════════════════════════════════════
class EnhancedReportGenerator:
    """
    PDF Layout (up to 11 pages):
      Page  1 : Cover + Executive Summary
      Page  2 : Stage 0 (AE) temporal signal graph
      Page  3 : Stage 1 (TF Binary)
      Page  4 : Stage 2 (TF Multi-class)
      Page  5 : Stage 3 (Weapon Detection, EXPERIMENTAL)
      Page  6 : Stage 4 (Audio)
      Page  7 : Stage 5 (Tracking)
      Page  8 : Stage 6 (Fusion)
      Page  9 : Stage 7 (RL3033, EXPERIMENTAL)
      Page 10 : Top anomalous frames
      Page 11 : Stage 8 (LLaVA, EXPERIMENTAL)
    """

    DISCLAIMER = (
        "This report is produced by machine learning models and should be treated "
        "as an investigative aid, not a definitive conclusion. Human expert review "
        "is strongly recommended before any operational action is taken."
    )

    # Color palette: pitch black background, white text, orange/saffron accents
    C_BG     = "#000000"
    C_DARK   = "#1a252f"
    C_HEADER = "#2c3e50"
    C_RED    = "#c0392b"
    C_GREEN  = "#1e8449"
    C_BLUE   = "#2471a3"
    C_ORANGE = "#d35400"
    C_PURPLE = "#6c3483"
    C_TEAL   = "#0e6655"
    C_GREY   = "#5d6d7e"
    C_LIGHT  = "#f2f3f4"

    HEADER_H = 52
    MARGIN_X = 48

    @staticmethod
    def _body_top(H: float) -> float:
        return H - EnhancedReportGenerator.HEADER_H - 16

    @staticmethod
    def _page_header(c, title: str, bg_hex: str, W: float = letter[0], H: float = letter[1]):
        c.setFillColor(colors.HexColor(bg_hex))
        c.rect(0, H - EnhancedReportGenerator.HEADER_H, W,
               EnhancedReportGenerator.HEADER_H, fill=True, stroke=False)
        c.setFillColor(colors.white)
        c.setFont("Helvetica-Bold", 15)
        c.drawCentredString(W / 2, H - EnhancedReportGenerator.HEADER_H + 16, title)

    @staticmethod
    def _footer(c, ts: str, W: float = letter[0]):
        c.setStrokeColor(colors.HexColor("#aab7b8"))
        c.setLineWidth(0.5)
        c.line(40, 28, W - 40, 28)
        c.setFont("Helvetica-Oblique", 7)
        c.setFillColor(colors.HexColor("#7f8c8d"))
        c.drawCentredString(W / 2, 14,
            f"Scene Solver v3  |  {ts}  |  Confidential, Investigative Use Only")

    @staticmethod
    def _section_title(c, text: str, y: float, x: float = None,
                       color_hex: str = "#2c3e50", W: float = letter[0]) -> float:
        if x is None:
            x = EnhancedReportGenerator.MARGIN_X
        c.setFillColor(colors.HexColor(color_hex))
        c.rect(x, y - 2, 4, 14, fill=True, stroke=False)
        c.setFont("Helvetica-Bold", 11)
        c.drawString(x + 8, y, text)
        c.setFillColor(colors.black)
        return y - 20

    @staticmethod
    def _kv_row(c, key: str, value: str, x: float, y: float,
                key_w: float = 170, line_h: float = 14, font_size: int = 9) -> float:
        c.setFont("Helvetica-Bold", font_size)
        c.setFillColor(colors.HexColor("#2c3e50"))
        c.drawString(x, y, key + ":")
        c.setFont("Helvetica", font_size)
        c.setFillColor(colors.black)
        c.drawString(x + key_w, y, value)
        return y - line_h

    @staticmethod
    def _wrap_text(c, text: str, x: float, y: float, max_width: float,
                   font: str = "Helvetica", size: int = 9,
                   line_h: int = 12, min_y: float = 50) -> float:
        c.setFont(font, size)
        c.setFillColor(colors.black)
        words, line = text.split(), ""
        for w in words:
            test = line + w + " "
            if c.stringWidth(test, font, size) < max_width:
                line = test
            else:
                if y < min_y:
                    break
                c.drawString(x, y, line.rstrip())
                y   -= line_h
                line = w + " "
        if line.strip() and y >= min_y:
            c.drawString(x, y, line.rstrip())
            y -= line_h
        return y

    @staticmethod
    def _disclaimer_box(c, y: float, W: float = letter[0]) -> float:
        MX    = EnhancedReportGenerator.MARGIN_X
        box_w = W - 2 * MX
        c.setFillColor(colors.HexColor("#fdfefe"))
        c.setStrokeColor(colors.HexColor("#aab7b8"))
        c.setLineWidth(0.6)
        c.roundRect(MX, y - 28, box_w, 32, 4, fill=True, stroke=True)
        c.setFont("Helvetica-Oblique", 7.5)
        c.setFillColor(colors.HexColor("#626567"))
        EnhancedReportGenerator._wrap_text(
            c, EnhancedReportGenerator.DISCLAIMER,
            MX + 6, y - 9, box_w - 12, "Helvetica-Oblique", 7.5, 10)
        return y - 40

    @staticmethod
    def _status_badge(c, text: str, y: float, color_hex: str,
                      W: float = letter[0], MX: float = None) -> float:
        if MX is None:
            MX = EnhancedReportGenerator.MARGIN_X
        bw, bh, br = W - 2 * MX, 38, 8
        c.setFillColor(colors.HexColor(color_hex))
        c.roundRect(MX, y - bh, bw, bh, br, fill=True, stroke=False)
        c.setFillColor(colors.white)
        c.setFont("Helvetica-Bold", 14)
        c.drawCentredString(W / 2, y - bh + 12, text)
        return y - bh - 10

    @staticmethod
    def _experimental_warning_box(c, text: str, y: float,
                                   W: float = letter[0], MX: float = None) -> float:
        """Dynamic-height experimental disclaimer box. Text flows as one continuous paragraph."""
        if MX is None:
            MX = EnhancedReportGenerator.MARGIN_X
        box_w = W - 2 * MX
        font, size, line_h = "Helvetica-Bold", 8.5, 12
        normalised_text = " ".join(text.split())

        # Pre-measure line count
        import io
        from reportlab.pdfgen.canvas import Canvas as _TmpCanvas
        tmp_c = _TmpCanvas(io.BytesIO())
        tmp_c.setFont(font, size)
        words, line_buf, lines = normalised_text.split(), "", []
        for w in words:
            test = line_buf + w + " "
            if tmp_c.stringWidth(test, font, size) < box_w - 16:
                line_buf = test
            else:
                lines.append(line_buf.rstrip())
                line_buf = w + " "
        if line_buf.strip():
            lines.append(line_buf.rstrip())
        box_h = max(len(lines), 1) * line_h + 16

        c.setFillColor(colors.HexColor("#fef9e7"))
        c.setStrokeColor(colors.HexColor("#d35400"))
        c.setLineWidth(1.2)
        c.roundRect(MX, y - box_h, box_w, box_h, 5, fill=True, stroke=True)
        c.setFont(font, size)
        c.setFillColor(colors.HexColor("#d35400"))
        text_y = y - 8 - size
        for ln in lines:
            c.drawString(MX + 8, text_y, ln)
            text_y -= line_h
        c.setFillColor(colors.black)
        return y - box_h - 8

    # ── Graph generators ───────────────────────────────────────────────────
    @staticmethod
    def generate_deviation_graph(scores, threshold, peak_score_indices,
                                  sampled_indices, fps,
                                  output_path="/tmp/deviation_graph.png"):
        ts_labels = [frame_to_timestamp(
                         sampled_indices[i] if i < len(sampled_indices) else i * 5, fps)
                     for i in range(len(scores))]
        x = list(range(len(scores)))
        fig, ax = plt.subplots(figsize=(13, 4.2))
        ax.plot(x, scores, linewidth=2.2, color="#c0392b", label="Reconstruction Error", zorder=2)
        ax.axhline(y=threshold, color="#1e8449", linestyle="--",
                   label=f"Dynamic Threshold ({threshold:.5f})", linewidth=1.8, zorder=1)
        valid_peaks = [i for i in peak_score_indices if i < len(scores)]
        if valid_peaks:
            ax.scatter(valid_peaks, [scores[i] for i in valid_peaks],
                       color="#f39c12", s=220, marker="*",
                       edgecolors="#333", linewidths=1.2, label="Peak Anomaly", zorder=3)
        ax.fill_between(x, scores, threshold,
                        where=[s > threshold for s in scores], color="#c0392b", alpha=0.15)
        tick_step = max(1, len(scores) // 8)
        tick_pos  = list(range(0, len(scores), tick_step))
        ax.set_xticks(tick_pos)
        ax.set_xticklabels([ts_labels[i] for i in tick_pos], rotation=25, fontsize=7.5, ha="right")
        ax.set_xlabel("Timestamp (MM:SS.mmm)", fontsize=10, fontweight="bold")
        ax.set_ylabel("Reconstruction Error",  fontsize=10, fontweight="bold")
        ax.set_title("Autoencoder Temporal Reconstruction Error", fontsize=12, fontweight="bold", pad=10)
        ax.legend(loc="upper right", fontsize=9, framealpha=0.92)
        ax.grid(True, alpha=0.25, linestyle=":", linewidth=0.8)
        fig.tight_layout()
        fig.savefig(output_path, dpi=150, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        return output_path

    @staticmethod
    def generate_classification_pie(probabilities, output_path="/tmp/class_pie.png"):
        """Pie chart with numbered wedges only. Percentages appear in the legend panel."""
        items  = sorted(probabilities.items(), key=lambda x: x[1], reverse=True)
        labels = [i[0] for i in items]
        sizes  = [max(i[1], 1e-6) for i in items]
        clist  = ["#c0392b","#2471a3","#d35400","#6c3483","#117a65","#1a5276","#7d6608"]
        fig, (ax_pie, ax_leg) = plt.subplots(1, 2, figsize=(10, 5),
                                              gridspec_kw={"width_ratios": [1.3, 1]})
        ax_pie.pie(
            sizes,
            labels=[str(i + 1) for i in range(len(sizes))],
            colors=clist[:len(sizes)], startangle=90,
            textprops={"fontsize": 10, "weight": "bold"},
            explode=[0.04 if i == 0 else 0 for i in range(len(sizes))],
            labeldistance=1.10,
        )
        ax_pie.set_title("Anomaly Class Distribution\n(see legend for percentages)",
                         fontsize=10, fontweight="bold", pad=8)
        ax_leg.axis("off")
        for i, (cls, prob) in enumerate(zip(labels, sizes)):
            ax_leg.add_patch(plt.Rectangle((0, 0.85 - i * 0.12), 0.12, 0.09,
                                            color=clist[i % len(clist)]))
            ax_leg.text(0.16, 0.855 - i * 0.12 + 0.01,
                        f"{i+1}. {cls}  ({prob*100:.1f}%)",
                        va="center", fontsize=8.5, fontweight="bold",
                        transform=ax_leg.transAxes)
        ax_leg.set_xlim(0, 1); ax_leg.set_ylim(0, 1)
        fig.tight_layout()
        fig.savefig(output_path, dpi=150, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        return output_path

    @staticmethod
    def generate_confidence_bars(top3, output_path="/tmp/conf_bars.png"):
        if not top3:
            return None
        fig, ax = plt.subplots(figsize=(7, 3.2))
        classes = [t[0] for t in top3]
        confs   = [t[1] * 100 for t in top3]
        bars    = ax.barh(classes, confs,
                          color=["#c0392b","#d35400","#5d6d7e"][:len(top3)],
                          edgecolor="#2c3e50", linewidth=0.8, height=0.5)
        for bar, cv in zip(bars, confs):
            ax.text(bar.get_width() + 0.8, bar.get_y() + bar.get_height() / 2,
                    f"{cv:.1f}%", va="center", fontsize=9, fontweight="bold", color="#2c3e50")
        ax.set_xlabel("Confidence (%)", fontsize=9, fontweight="bold")
        ax.set_title("Top-3 Anomaly Predictions", fontsize=10, fontweight="bold", pad=8)
        ax.set_xlim(0, 118)
        ax.grid(axis="x", alpha=0.25, linestyle="--", linewidth=0.7)
        ax.invert_yaxis()
        fig.tight_layout()
        fig.savefig(output_path, dpi=150, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        return output_path

    @staticmethod
    def extract_top_frames(frames, scores, sampled_indices, n=3):
        if not scores or not frames:
            return []
        top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        out = []
        for si in top[:n]:
            fi = sampled_indices[si] if si < len(sampled_indices) else si * 5
            if fi < len(frames):
                out.append({"frame": frames[fi], "index": fi, "score": scores[si]})
        return out

    # ── Main PDF builder ────────────────────────────────────────────────────
    @staticmethod
    def generate_enhanced_pdf(results: dict, output_path: str = "/tmp/forensic_report.pdf") -> str:
        c    = rl_canvas.Canvas(output_path, pagesize=letter)
        W, H = letter
        ts   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        G    = EnhancedReportGenerator
        MX   = G.MARGIN_X
        BW   = W - 2 * MX  # body width

        s0 = results.get("stage0") or {}
        s1 = results.get("stage1") or {}
        s2 = results.get("stage2") or {}
        s3 = results.get("stage3") or {}
        s4 = results.get("stage4") or {}
        s5 = results.get("stage5") or {}
        s6 = results.get("stage6") or {}
        s7 = results.get("stage7") or {}
        s8 = results.get("stage8") or {}
        al = results.get("alert")

        fps       = float(s0.get("fps", 25.0))
        threshold = results.get("threshold", Config.SAFE_THRESHOLD_LEGACY)
        overall   = results.get("status", "SAFE")
        is_crit   = (overall == "CRITICAL")
        stage1_normal = (s1.get("classification", "Unknown") == "Normal")

        def ts_of(fi):
            return frame_to_timestamp(int(fi), fps)

        # ── Page 1: Cover + Executive Summary ─────────────────────────────
        G._page_header(c, "FORENSIC ANALYSIS DOSSIER, Scene Solver v3", G.C_HEADER, W, H)
        y = G._body_top(H)

        c.setFont("Helvetica", 8); c.setFillColor(colors.HexColor(G.C_GREY))
        c.drawCentredString(W / 2, y, f"Report generated: {ts}")
        y -= 14

        badge_text = "ANOMALY DETECTED (CRITICAL)" if is_crit else "NO ANOMALY DETECTED (SAFE)"
        y = G._status_badge(c, badge_text, y, G.C_RED if is_crit else G.C_GREEN, W, MX)
        y -= 6
        y = G._section_title(c, "EXECUTIVE SUMMARY", y, MX, G.C_HEADER)

        def _field(label, value, key_w=185):
            nonlocal y
            if y < 60: return
            c.setFont("Helvetica-Bold", 9); c.setFillColor(colors.HexColor(G.C_DARK))
            c.drawString(MX + 10, y, label + ":")
            c.setFont("Helvetica", 9); c.setFillColor(colors.black)
            c.drawString(MX + 10 + key_w, y, str(value))
            y -= 13

        def _spacer(h=5):
            nonlocal y; y -= h

        _field("Overall Status",           overall)
        _field("Dynamic Threshold",        f"{threshold:.6f}  (mean plus 2 standard deviations)")
        _field("AE Max Reconstruction Error", f"{s0.get('max_error', 0):.6f}")
        _field("AE Avg Reconstruction Error", f"{s0.get('avg_error', 0):.6f}")
        _field("Peak Event Timestamp",     ts_of(s0.get("peak_frame_idx", 0)))
        _field("Anomaly Window",
               f"{ts_of(s0.get('anomaly_start_frame', 0))}  to  {ts_of(s0.get('anomaly_end_frame', 0))}")
        _spacer()

        if s1:
            _field("Stage 1 (TF Binary)", f"{s1.get('classification','?')}  ({s1.get('confidence',0):.1%})")
        if s2:
            if s2.get("class") == "Skipped":
                _field("Stage 2 (TF Multi)", "Skipped (Stage 1 = Normal)")
            else:
                samp_note = "  [AE-weighted sampling]" if s2.get("sampling_method") == "segmentwise_ae" else ""
                _field("Stage 2 (TF Multi)",
                       f"{s2.get('class','?')}  ({s2.get('confidence',0):.1%}){samp_note}")
        _spacer()

        if s3:
            wf = bool(s3.get("weapon_flag", False))
            _field("Stage 3 (Weapon Detection)",
                   "Weapons detected" if wf else "No weapons detected", key_w=210)
            if wf:
                _field("  Dominant Class",  s3.get("dominant_class", "?"), key_w=210)
                _field("  Best Confidence", f"{s3.get('best_conf', 0):.2%}", key_w=210)
        _spacer()

        if s4:
            audio_ok = s4.get("audio_present", True)
            audio_st = s4.get("status", "?")
            _field("Stage 4 (Audio)",
                   "No audio or silent track, section skipped"
                   if not audio_ok or audio_st in ("NO_AUDIO", "SILENT")
                   else f"{audio_st}  (score = {s4.get('audio_score', 0):.4f})")
        if s5:
            _field("Stage 5 (Tracking)",
                   f"Motion = {s5.get('motion_intensity',0):.3f},  "
                   f"Behaviors: {', '.join(s5.get('behaviors',[])) or 'None'}")
        _spacer()

        if s6:
            _field("Stage 6 (Fusion)",
                   f"{s6.get('fused_risk',0):.1%}  [{s6.get('risk_level','?')}]")
        if s7:
            _field("Stage 7 (RL3033, Experimental)",
                   f"Final risk = {s7.get('final_risk',0):.1%}  [{s7.get('status','?')}]")
        _spacer()

        if al:
            c.setFont("Helvetica-Bold", 9); c.setFillColor(colors.HexColor(G.C_RED))
            c.drawString(MX + 10, y,
                f"ALERT FIRED: {al['level']}  |  Final risk = {al['final_risk']:.1%}")
            c.setFillColor(colors.black); y -= 13

        # Pipeline interpretation paragraph
        _spacer(8)
        y = G._section_title(c, "PIPELINE INTERPRETATION", y, MX, G.C_HEADER)
        risk_val   = float(s7.get("final_risk", s6.get("fused_risk", 0.0)))
        risk_label = s6.get("risk_level", "LOW")
        cls1_label = s1.get("classification", "Unknown")
        cls1_conf  = float(s1.get("confidence", 0.0))
        cls2_label = s2.get("class", "Unknown")
        cls2_conf  = float(s2.get("confidence", 0.0))
        weapon_flag = bool(s3.get("weapon_flag", False))
        audio_ok    = s4.get("audio_present", True)
        audio_st    = s4.get("status", "NO_AUDIO")
        motion_val  = float(s5.get("motion_intensity", 0.0))
        behaviors   = s5.get("behaviors", [])

        interp = [f"The pipeline classified this video as {overall} with a final risk score of {risk_val:.1%} ({risk_label})."]
        if cls1_label == "Suspicious":
            interp.append(f"The semantic binary classifier (Stage 1) identified the clip as Suspicious at {cls1_conf:.0%} confidence.")
            if cls2_label not in ("Unknown", "Skipped", "Error"):
                interp.append(f"Stage 2 classified the event type as {cls2_label} ({cls2_conf:.0%} confidence) using AE-weighted temporal sampling.")
        else:
            interp.append(f"The semantic binary classifier returned Normal ({cls1_conf:.0%} confidence), overriding the AE signal.")
        if weapon_flag:
            interp.append(f"A weapon ({s3.get('dominant_class','?')}) was detected at {s3.get('best_conf', 0.0):.0%} confidence, triggering hard risk escalation.")
        if audio_ok and audio_st not in ("NO_AUDIO", "SILENT", "NORMAL"):
            interp.append(f"Audio analysis reported {audio_st} status (score {s4.get('audio_score',0):.3f}).")
        if "FAST_MOVEMENT" in behaviors:
            interp.append(f"Motion tracking recorded fast subject movement (intensity {motion_val:.2f}).")
        elif "CLUSTERING" in behaviors:
            interp.append(f"Motion tracking detected converging subject trajectories (intensity {motion_val:.2f}).")

        y = G._wrap_text(c, " ".join(interp), MX + 6, y, BW - 12, "Helvetica", 8.5, 12)
        _spacer(8)
        y = G._disclaimer_box(c, y, W)
        G._footer(c, ts, W)

        # ── Page 2: Stage 0 (AE) ──────────────────────────────────────────
        c.showPage()
        G._page_header(c, "STAGE 0 (AUTOENCODER): TEMPORAL ANOMALY DETECTION", G.C_RED, W, H)
        y = G._body_top(H)
        c.setFont("Helvetica-Oblique", 8.5); c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "The Autoencoder is trained exclusively on normal footage. High reconstruction error "
            "indicates unusual content. The dynamic threshold is computed as mean plus two standard "
            "deviations of the per-frame error.",
            MX, y, BW, "Helvetica-Oblique", 8.5, 12)
        y -= 8

        gp = results.get("graph_path")
        if gp and os.path.exists(gp):
            img_h = 200
            c.drawImage(gp, MX, y - img_h, width=BW, height=img_h, preserveAspectRatio=True)
            y -= img_h + 10

        if stage1_normal:
            y = G._section_title(c, "AE Score Summary", y, MX, G.C_GREEN)
            y = G._wrap_text(c,
                "Stage 1 classified this video as NORMAL, overriding the AE signal. "
                "Downstream anomaly stages were skipped. Pipeline status is SAFE.",
                MX + 6, y, BW - 12, "Helvetica", 9, 13)
            y -= 8
            for k, v in [
                ("Max Reconstruction Error",  f"{s0.get('max_error',0):.6f}"),
                ("Avg Reconstruction Error",  f"{s0.get('avg_error',0):.6f}"),
                ("Dynamic Threshold",         f"{threshold:.6f}"),
                ("Threshold Exceeded by AE",  "YES" if s0.get('max_error',0) > threshold else "NO"),
                ("Note", "AE signal overridden to SAFE by Stage 1 semantic classifier"),
                ("Total Sampled Frames",      str(len(s0.get("scores", [])))),
                ("Video FPS",                 f"{fps:.1f}"),
                ("Pipeline Status",           "SAFE (Stage 1 override)"),
            ]:
                y = G._kv_row(c, k, v, MX + 6, y, key_w=220)
                if y < 60: break
        else:
            y = G._section_title(c, "AE Score Summary", y, MX, G.C_RED)
            ae_over = s0.get('max_error', 0) > threshold
            for k, v in [
                ("Max Reconstruction Error",  f"{s0.get('max_error',0):.6f}"),
                ("Avg Reconstruction Error",  f"{s0.get('avg_error',0):.6f}"),
                ("Dynamic Threshold",         f"{threshold:.6f}  (mean + 2 std dev, floor = {Config.SAFE_THRESHOLD_FLOOR})"),
                ("Threshold Exceeded",        "YES" if ae_over else "NO"),
                ("Excess Above Threshold",    f"{s0.get('max_error',0) - threshold:.6f}" if ae_over else "N/A"),
                ("Peak Event Timestamp",      ts_of(s0.get("peak_frame_idx", 0))),
                ("Anomaly Start Timestamp",   ts_of(s0.get("anomaly_start_frame", 0))),
                ("Anomaly End Timestamp",     ts_of(s0.get("anomaly_end_frame", 0))),
                ("Anomaly Duration (frames)", str(s0.get("anomaly_end_frame",0) - s0.get("anomaly_start_frame",0))),
                ("Total Sampled Frames",      str(len(s0.get("scores", [])))),
                ("Sampling Stride",           f"Every {Config.AE_FRAME_STRIDE} frames"),
                ("Video FPS",                 f"{fps:.1f}"),
            ]:
                y = G._kv_row(c, k, v, MX + 6, y, key_w=220)
                if y < 60: break
            if y > 100:
                y -= 6
                y = G._section_title(c, "Signal Interpretation", y, MX, G.C_RED)
                ratio = s0.get('max_error', 0) / max(threshold, 1e-8)
                if ratio >= 2.0:
                    ae_interp = f"Peak reconstruction error is {ratio:.1f}x the threshold — strong, unambiguous anomaly signal."
                elif ratio >= 1.0:
                    ae_interp = f"Peak exceeds threshold by {ratio:.2f}x. Meaningful signal; benefits from corroboration."
                else:
                    ae_interp = f"Error remains below threshold (ratio {ratio:.3f}). AE alone does not flag anomaly."
                y = G._wrap_text(c, ae_interp, MX + 6, y, BW - 12, "Helvetica", 9, 13)

        y -= 6; y = G._disclaimer_box(c, y, W); G._footer(c, ts, W)

        # ── Page 3: Stage 1 (TF Binary) ───────────────────────────────────
        c.showPage()
        G._page_header(c, "STAGE 1 (TIMESFORMER BINARY): SEMANTIC CLASSIFICATION", G.C_BLUE, W, H)
        y = G._body_top(H)
        c.setFont("Helvetica-Oblique", 8.5); c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "The binary TimeSformer classifies a 32-frame clip as Normal or Suspicious. "
            "A Normal result overrides the AE signal to SAFE and skips Stages 2 and 3. "
            "LLaVA (Stage 8) still runs regardless, using neutral investigative prompts.",
            MX, y, BW, "Helvetica-Oblique", 8.5, 12)
        y -= 12

        cls1, conf1 = s1.get("classification","?"), s1.get("confidence",0.0)
        y = G._status_badge(c, f"{cls1.upper()}  ({conf1:.1%})", y,
                            G.C_RED if cls1 == "Suspicious" else G.C_GREEN, W, MX)
        y -= 10
        y = G._section_title(c, "Classification Details", y, MX, G.C_BLUE)
        for k, v in [
            ("Result",             cls1),
            ("Confidence",         f"{conf1:.2%}"),
            ("Clip Centre",        ts_of(s0.get("peak_frame_idx", 0))),
            ("Clip Window",        "Peak timestamp, plus or minus 16 frames (32 frames total)"),
            ("Gates YOLO Stage 3", "YES" if cls1 == "Suspicious" else "NO, skipped"),
            ("Gates Stage 2",      "YES" if cls1 == "Suspicious" else "NO, skipped (Stage 1 = Normal)"),
            ("LLaVA Stage 8",      "Runs with full context" if cls1 == "Suspicious"
                                   else "Runs with neutral investigative prompts"),
            ("Overall Status Effect", "Follows AE signal" if cls1 == "Suspicious"
                                      else "Overrides AE — pipeline forced to SAFE"),
        ]:
            y = G._kv_row(c, k, v, MX + 6, y, key_w=210)
        y -= 8
        y = G._section_title(c, "Interpretation and Operational Significance", y, MX, G.C_BLUE)
        if cls1 == "Suspicious":
            interp = (f"Clip at {ts_of(s0.get('peak_frame_idx',0))} classified SUSPICIOUS at {conf1:.1%} confidence. "
                      "Weapon detection and multi-class classification are now active.")
        else:
            interp = (f"Clip classified NORMAL at {conf1:.1%} confidence. "
                      "The AE signal is overridden. Multi-class classification is skipped. "
                      "LLaVA runs with neutral prompts.")
        y = G._wrap_text(c, interp, MX + 6, y, BW - 12, "Helvetica", 9, 13)
        y -= 6; y = G._disclaimer_box(c, y, W); G._footer(c, ts, W)

        # ── Page 4: Stage 2 (TF Multi-class) ──────────────────────────────
        c.showPage()
        G._page_header(c, "STAGE 2 (TIMESFORMER MULTI-CLASS): ANOMALY TYPE", G.C_PURPLE, W, H)
        y = G._body_top(H)
        cls2, conf2 = s2.get("class","Unknown"), s2.get("confidence",0.0)
        samp_meth   = s2.get("sampling_method","?")
        stage2_skipped = (cls2 == "Skipped" or not s2.get("all_probabilities"))

        c.setFont("Helvetica-Oblique", 8.5); c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "Multi-class TimeSformer classifies the anomaly into one of seven crime categories. "
            "Only runs when Stage 1 returns Suspicious. Uses segment-wise AE-weighted temporal "
            "sampling: the video is split into 16 equal segments; the highest-error frame per "
            "segment is selected.",
            MX, y, BW, "Helvetica-Oblique", 8.5, 12)
        y -= 8

        if stage2_skipped:
            y = G._status_badge(c, "STAGE 2 SKIPPED (STAGE 1 = NORMAL)", y, G.C_GREY, W, MX)
            y -= 8
            y = G._section_title(c, "Why This Stage Was Skipped", y, MX, G.C_GREY)
            y = G._wrap_text(c,
                "Stage 1 returned Normal, which suppresses all anomaly-specific downstream "
                "processing. Running multi-class classification on a Normal-labelled clip "
                "would produce unreliable category assignments.",
                MX + 6, y, BW - 12, "Helvetica", 9, 13)
        else:
            y = G._status_badge(c, f"{cls2.upper()}  ({conf2:.1%})", y, G.C_PURPLE, W, MX)
            y -= 4
            c.setFont("Helvetica-Oblique", 8); c.setFillColor(colors.HexColor(G.C_GREY))
            samp_label = "Segment-wise AE-weighted (16 segments)" if samp_meth == "segmentwise_ae" else samp_meth
            c.drawCentredString(W / 2, y, f"Sampling: {samp_label}")
            y -= 10
            c.setFont("Helvetica-Oblique", 8); c.setFillColor(colors.HexColor(G.C_PURPLE))
            c.drawCentredString(W / 2, y, "Chart shows class numbers only. Refer to numbered legend for names and percentages.")
            y -= 10

            all_probs = s2.get("all_probabilities", {})
            top3      = s2.get("top3", [])
            if all_probs:
                pie_path = "/tmp/cls_pie_v3.png"
                G.generate_classification_pie(all_probs, pie_path)
                if os.path.exists(pie_path):
                    c.drawImage(pie_path, MX, y - 200, width=310, height=200, preserveAspectRatio=True)
            if top3:
                bar_path = "/tmp/conf_bars_v3.png"
                G.generate_confidence_bars(top3, bar_path)
                if os.path.exists(bar_path):
                    c.drawImage(bar_path, MX + 318, y - 150, width=220, height=150, preserveAspectRatio=True)
            y -= 212
            y = G._section_title(c, "Top-3 Predictions", y, MX, G.C_PURPLE)
            for rank, (cls_name, prob) in enumerate(top3[:3], 1):
                c.setFont("Helvetica", 9); c.setFillColor(colors.black)
                c.drawString(MX + 12, y, f"{rank}.  {cls_name}  ({prob:.2%})")
                y -= 13
            y -= 4
            y = G._section_title(c, "All Class Probabilities", y, MX, G.C_PURPLE)
            for cls_name, prob in sorted(all_probs.items(), key=lambda x: x[1], reverse=True):
                c.setFont("Helvetica", 8.5); c.setFillColor(colors.black)
                c.drawString(MX + 12, y, f"{cls_name:<22s}  {prob:.4f}  ({prob * 100:.2f}%)")
                y -= 12
                if y < 70: break
            if y > 100 and top3:
                y -= 6
                y = G._section_title(c, "Classification Insight", y, MX, G.C_PURPLE)
                top_cls, top_conf = top3[0]
                if top_conf >= 0.60:
                    insight = f"Model assigned highest probability to '{top_cls}' ({top_conf:.1%}) — high-confidence prediction."
                elif top_conf >= 0.35:
                    insight = f"Model assigned highest probability to '{top_cls}' ({top_conf:.1%}) — moderate confidence. Consider second and third candidates."
                else:
                    insight = f"Model assigned highest probability to '{top_cls}' ({top_conf:.1%}) — low confidence. Anomaly pattern may not cleanly match any trained category."
                y = G._wrap_text(c, insight, MX + 6, y, BW - 12, "Helvetica", 9, 13)

        y -= 4; y = G._disclaimer_box(c, y, W); G._footer(c, ts, W)

        # ── Page 5: Stage 3 (Weapon Detection) ────────────────────────────
        c.showPage()
        weapon_flag = bool(s3.get("weapon_flag", False))
        hdr_col3    = G.C_RED if weapon_flag else G.C_TEAL
        G._page_header(c, "STAGE 3 (WEAPON DETECTION, EXPERIMENTAL)", hdr_col3, W, H)
        y = G._body_top(H)
        y = G._experimental_warning_box(c,
            "EXPERIMENTAL: This module is under active development. "
            "Detection results should be independently verified before any action is taken.",
            y, W, MX)
        c.setFont("Helvetica-Oblique", 8.5); c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "A custom-trained YOLOv8 model scans frames within the anomaly window for Melee "
            "weapons and Firearms. Only activates when Stage 1 returns Suspicious. A confirmed "
            "weapon detection triggers a hard escalation in Stage 6 (minimum risk 0.90 when "
            "combined with an AE anomaly). Up to 32 frames are scanned.",
            MX, y, BW, "Helvetica-Oblique", 8.5, 12)
        y -= 8

        badge_txt = f"WEAPONS DETECTED ({s3.get('dominant_class','?').upper()})" if weapon_flag else "NO WEAPONS DETECTED"
        y = G._status_badge(c, badge_txt, y, G.C_RED if weapon_flag else G.C_TEAL, W, MX)
        y -= 8

        skipped = str(s3.get("summary", "")).lower().startswith("skipped")
        if skipped:
            c.setFont("Helvetica-Oblique", 9); c.setFillColor(colors.HexColor(G.C_GREY))
            c.drawString(MX + 6, y, "Stage skipped — Stage 1 classified the video as Normal.")
            y -= 20
            y = G._section_title(c, "Operational Note", y, MX, G.C_GREY)
            y = G._wrap_text(c,
                "Weapon scanning is gated behind Stage 1 to reduce false positives. "
                "When Stage 1 returns Suspicious, weapon scanning activates for the anomaly window.",
                MX + 6, y, BW - 12, "Helvetica", 9, 13)
        else:
            y = G._section_title(c, "Detection Summary", y, MX, hdr_col3)
            for k, v in [
                ("Weapon Detected",          "YES" if weapon_flag else "NO"),
                ("Dominant Weapon Class",    s3.get("dominant_class", "None")),
                ("Total Detections",         str(s3.get("weapon_count", 0))),
                ("Best Detection Confidence",
                 f"{s3.get('best_conf', 0.0):.2%}" if weapon_flag else "N/A"),
                ("Frames Scanned",           "Up to 32 frames within the anomaly window"),
                ("Anomaly Window",
                 f"{ts_of(s0.get('anomaly_start_frame',0))} to {ts_of(s0.get('anomaly_end_frame',0))}"),
                ("Fusion Impact",
                 "Risk forced to minimum 0.90 (weapon + AE threshold exceeded)"
                 if weapon_flag and s0.get("max_error",0) > threshold
                 else "Weapon floor applied (0.40 minimum)" if weapon_flag
                 else "No fusion impact from weapon stage"),
            ]:
                y = G._kv_row(c, k, v, MX + 6, y, key_w=210)
            y -= 6
            if weapon_flag:
                y = G._section_title(c, "Individual Detections (top 10)", y, MX, hdr_col3)
                for di, det in enumerate(s3.get("all_detections", [])[:10], 1):
                    c.setFont("Helvetica", 8.5); c.setFillColor(colors.black)
                    c.drawString(MX + 8, y,
                        f"  {di:2d}.  {det.get('class_name','?'):12s}  "
                        f"Confidence = {det.get('confidence',0):.2%},  at {ts_of(det.get('frame_idx',0))}")
                    y -= 12
                    if y < 130: break
                top_frames = s3.get("top_frames_annotated", [])
                if top_frames and y > 80:
                    y -= 4
                    y = G._section_title(c, "Top Detection Frames (with bounding boxes)", y, MX, hdr_col3)
                    n_show = min(2, len(top_frames))
                    img_w  = (BW - 10) / max(n_show, 1)
                    img_h  = min(140, max(y - 60, 60), img_w * 0.6)
                    for fi_idx, (fi, ann_frame) in enumerate(top_frames[:n_show]):
                        tmp_path = f"/tmp/yolo_ann_{fi_idx}.jpg"
                        Image.fromarray(ann_frame).save(tmp_path)
                        cx = MX + fi_idx * (img_w + 10)
                        c.drawImage(tmp_path, cx, y - img_h, width=img_w - 4,
                                    height=img_h, preserveAspectRatio=True)
                        c.setFont("Helvetica", 7.5); c.setFillColor(colors.HexColor(G.C_DARK))
                        c.drawString(cx, y - img_h - 10, f"Timestamp: {ts_of(fi)}")
                    y -= img_h + 22

        y -= 4; y = G._disclaimer_box(c, y, W); G._footer(c, ts, W)

        # ── Page 6: Stage 4 (Audio) ────────────────────────────────────────
        audio_present = s4.get("audio_present", True)
        audio_status  = s4.get("status", "NO_AUDIO")
        audio_silent  = (not audio_present or audio_status in ("NO_AUDIO", "SILENT"))
        c.showPage()
        aud_col = (G.C_RED    if audio_status == "ANOMALY"    else
                   G.C_ORANGE if audio_status == "SUSPICIOUS" else
                   G.C_GREEN  if audio_status == "NORMAL"     else G.C_GREY)
        G._page_header(c, "STAGE 4 (AUDIO FEATURE EXTRACTION)", aud_col, W, H)
        y = G._body_top(H)
        c.setFont("Helvetica-Oblique", 8.5); c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "Five acoustic features are extracted and normalised to 0-1 using global range constants. "
            "Classification rule: 2+ features above 0.4 = ANOMALY; 1 = SUSPICIOUS; 0 = NORMAL. "
            "The composite score feeds into Stages 6 and 7.",
            MX, y, BW, "Helvetica-Oblique", 8.5, 12)
        y -= 6

        y = G._section_title(c, "What Each Feature Measures", y, MX, aud_col)
        for fname, fdesc in [
            ("Zero Crossing Rate (ZCR)",
             "How often the audio waveform crosses zero per second. High values indicate noisy or "
             "chaotic sounds (shouting, breaking glass, gunshots)."),
            ("MFCC Variance",
             "Captures tonal texture change over time. High variance means rapidly shifting audio "
             "content — screaming, rapid voice changes, or sudden silence."),
            ("Spectral Flux",
             "Measures how quickly frequency content changes. Sharp spikes indicate explosive impacts, "
             "gunshots, or sudden loud events."),
            ("RMS Energy",
             "Overall loudness. A sudden surge indicates a loud, potentially alarming sound event."),
            ("Spectral Centroid",
             "Centre of gravity of the frequency spectrum. Dramatic shifts can signal alarms, "
             "screaming, or high-pitched metal impacts."),
        ]:
            if y < 90: break
            c.setFont("Helvetica-Bold", 8.5); c.setFillColor(colors.HexColor(G.C_DARK))
            c.drawString(MX + 8, y, fname)
            y -= 12
            y = G._wrap_text(c, fdesc, MX + 16, y, BW - 24, "Helvetica", 8, 11)
            y -= 4

        y -= 4
        if audio_silent:
            c.setFillColor(colors.HexColor("#eaeded"))
            c.roundRect(MX, y - 44, BW, 48, 6, fill=True, stroke=False)
            c.setFillColor(colors.HexColor(G.C_GREY))
            c.setFont("Helvetica-Bold", 12)
            c.drawCentredString(W / 2, y - 20, "NO AUDIO TRACK OR SILENT TRACK")
            c.setFont("Helvetica", 9)
            c.drawCentredString(W / 2, y - 34, "The video has no audio or an entirely silent track.")
            y -= 60
        else:
            y = G._status_badge(c, f"AUDIO STATUS: {audio_status}", y, aud_col, W, MX)
            y -= 8
            y = G._section_title(c, "Raw Feature Values", y, MX, aud_col)
            for k, v in [
                ("Audio Composite Score",    f"{s4.get('audio_score',0):.4f}"),
                ("Zero Crossing Rate (ZCR)", f"{s4.get('zcr',0):.6f}"),
                ("MFCC Variance",            f"{s4.get('mfcc_variance',0):.4f}"),
                ("Spectral Flux",            f"{s4.get('spectral_flux',0):.4f}"),
                ("RMS Energy",               f"{s4.get('rms_energy',0):.6f}"),
                ("Spectral Centroid",        f"{s4.get('spectral_centroid',0):.2f} Hz"),
            ]:
                y = G._kv_row(c, k, v, MX + 6, y, key_w=200)
            norm = s4.get("normalized", {})
            if norm:
                y -= 4
                y = G._section_title(c, "Globally Normalised Values (threshold = 0.4)", y, MX, aud_col)
                flagged = []
                for k, v, key in [
                    ("ZCR (normalised)",               f"{norm.get('zcr',0):.4f}",  "zcr"),
                    ("MFCC Variance (normalised)",     f"{norm.get('mfcc',0):.4f}", "mfcc"),
                    ("Spectral Flux (normalised)",     f"{norm.get('flux',0):.4f}", "flux"),
                    ("RMS Energy (normalised)",        f"{norm.get('rms',0):.4f}",  "rms"),
                    ("Spectral Centroid (normalised)", f"{norm.get('sc',0):.4f}",   "sc"),
                ]:
                    flag = "  [ABOVE THRESHOLD]" if norm.get(key, 0) > 0.4 else ""
                    if flag: flagged.append(k.split(" (")[0])
                    y = G._kv_row(c, k, v + flag, MX + 6, y, key_w=230)
                if flagged and y > 90:
                    y -= 4
                    y = G._wrap_text(c,
                        f"Features above threshold: {', '.join(flagged)}. "
                        "Rule: 2+ features above 0.4 = ANOMALY, 1 = SUSPICIOUS, 0 = NORMAL.",
                        MX + 6, y, BW - 12, "Helvetica-Oblique", 8, 11)

        y -= 6; y = G._disclaimer_box(c, y, W); G._footer(c, ts, W)

        # ── Page 7: Stage 5 (Tracking) ────────────────────────────────────
        c.showPage()
        G._page_header(c, "STAGE 5 (TRACKING AND MOTION BEHAVIOR)", G.C_TEAL, W, H)
        y = G._body_top(H)
        c.setFont("Helvetica-Oblique", 8.5); c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "Multi-object tracking records pixel trajectories in the anomaly window. "
            "Three patterns are detected: FAST_MOVEMENT, STATIC, and CLUSTERING. "
            "YOLOv8 tracking is used when available; Farneback optical flow is the fallback.",
            MX, y, BW, "Helvetica-Oblique", 8.5, 12)
        y -= 10

        motion_int   = float(s5.get("motion_intensity", 0.0))
        behaviors    = s5.get("behaviors", [])
        track_count  = s5.get("track_count", 0)
        track_status = s5.get("status", "?")

        bar_col = G.C_RED if motion_int > 0.6 else G.C_ORANGE if motion_int > 0.3 else G.C_GREEN
        c.setFillColor(colors.HexColor("#d5d8dc"))
        c.rect(MX, y - 16, BW, 18, fill=True, stroke=False)
        c.setFillColor(colors.HexColor(bar_col))
        c.rect(MX, y - 16, int(BW * min(motion_int, 1.0)), 18, fill=True, stroke=False)
        c.setFillColor(colors.white if motion_int > 0.15 else colors.HexColor(G.C_DARK))
        c.setFont("Helvetica-Bold", 9)
        c.drawString(MX + 6, y - 9, f"Motion Intensity: {motion_int:.3f}  ({motion_int*100:.1f}%)")
        y -= 28

        y = G._section_title(c, "Tracking Summary", y, MX, G.C_TEAL)
        for k, v in [
            ("Tracking Method",     track_status),
            ("Tracked Objects",     str(track_count) if track_count else "N/A (optical flow)"),
            ("Motion Intensity",    f"{motion_int:.4f}  (0 = static, 1 = maximum displacement)"),
            ("Detected Behaviours", ", ".join(behaviors) if behaviors else "None"),
            ("Fusion Contribution",
             "FAST_MOVEMENT or CLUSTERING applies a motion boost to fused risk"
             if any(b in behaviors for b in ("FAST_MOVEMENT","CLUSTERING"))
             else "No behavioural boost applied"),
        ]:
            y = G._kv_row(c, k, v, MX + 6, y, key_w=210)
        y -= 8

        y = G._section_title(c, "Behaviour Definitions", y, MX, G.C_TEAL)
        for bname, bdesc in [
            ("FAST_MOVEMENT",
             "Average per-frame displacement exceeds 8 pixels. Subjects are running, lunging, or moving rapidly."),
            ("STATIC",
             "Average displacement below 2 pixels over at least 10 frames. Subjects are essentially stationary."),
            ("CLUSTERING",
             "Two subjects converge: distance decreases in over 75% of frames by at least 50 pixels. "
             "Associated with confrontational proximity or pursuit."),
        ]:
            c.setFont("Helvetica-Bold", 9); c.setFillColor(colors.HexColor(G.C_DARK))
            c.drawString(MX + 8, y, bname); y -= 12
            y = G._wrap_text(c, bdesc, MX + 16, y, BW - 24, "Helvetica", 8.5, 12)
            y -= 4

        y -= 4; y = G._disclaimer_box(c, y, W); G._footer(c, ts, W)

        # ── Page 8: Stage 6 (Fusion) ───────────────────────────────────────
        c.showPage()
        fused_risk  = float(s6.get("fused_risk", 0.0))
        risk_level  = s6.get("risk_level", "?")
        fus_col     = (G.C_RED    if risk_level == "CRITICAL" else
                       G.C_ORANGE if risk_level in ("HIGH","MODERATE") else G.C_GREEN)
        G._page_header(c, "STAGE 6 (FUSION LAYER): MULTI-MODAL REASONING", fus_col, W, H)
        y = G._body_top(H)
        c.setFont("Helvetica-Oblique", 8.5); c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "The Fusion layer combines all stage outputs into a single risk score via explicit "
            "rule-based reasoning. The base weight for the AE score is 50%. Semantic classification, "
            "motion, audio, and weapon signals apply additive boosts or penalties. "
            "Each step is recorded below, making this an explainable AI decision layer.",
            MX, y, BW, "Helvetica-Oblique", 8.5, 12)
        y -= 8
        y = G._status_badge(c, f"{risk_level}, Fused Risk = {fused_risk:.2%}", y, fus_col, W, MX)
        y -= 10
        y = G._section_title(c, "Input Signal Values", y, MX, fus_col)
        for k, v in [
            ("AE Score (normalised)",    f"{s6.get('ae_score_norm',0):.4f}  (50% base weight)"),
            ("Binary TF Confidence",     f"{s6.get('ts_binary_conf',0):.4f}  (20% weight if Suspicious)"),
            ("Audio Score",              f"{s6.get('audio_score',0):.4f}  (10% boost if ANOMALY)"),
            ("Weapon Detected",
             "YES  (minimum 0.40 floor; 0.90 floor if AE also anomalous)"
             if s6.get("weapon_flag") else "NO"),
            ("Motion Intensity",         f"{s6.get('motion_intensity',0):.4f}  (15% boost if FAST_MOVEMENT + audio)"),
        ]:
            y = G._kv_row(c, k, v, MX + 6, y, key_w=230)
        y -= 8
        y = G._section_title(c, "Step-by-Step Reasoning", y, MX, fus_col)
        for i, line in enumerate(s6.get("reasoning_trace", []), 1):
            y = G._wrap_text(c, f"Step {i}:  {line}", MX + 8, y, BW - 16, "Helvetica", 8.5, 12)
            y -= 4
            if y < 80: break
        y -= 4; y = G._disclaimer_box(c, y, W); G._footer(c, ts, W)

        # ── Page 9: Stage 7 (RL3033) ──────────────────────────────────────
        c.showPage()
        final_risk = float(s7.get("final_risk", 0.0))
        rl_status  = s7.get("status", "?")
        rl_col     = (G.C_RED    if rl_status in ("CRITICAL","ERROR") else
                      G.C_ORANGE if rl_status in ("HIGH_RISK","MODERATE") else G.C_GREEN)
        G._page_header(c, "STAGE 7 (RL3033, EXPERIMENTAL): TEMPORAL REASONING", rl_col, W, H)
        y = G._body_top(H)
        y = G._experimental_warning_box(c,
            "EXPERIMENTAL: The RL3033 agent is actively being trained and refined. "
            "Its output is blended with the deterministic fusion score rather than used in isolation.",
            y, W, MX)
        c.setFont("Helvetica-Oblique", 8.5); c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "RL3033 is a PPO reinforcement learning agent trained by trial and reward to predict risk "
            "from multi-modal observation sequences. Its score is blended with the rule-based fusion "
            "score to produce the final risk value: 70% TFB + 20% TFM + 5% Fused + 5% RL.",
            MX, y, BW, "Helvetica-Oblique", 8.5, 12)
        y -= 8
        y = G._status_badge(c, f"FINAL RISK: {final_risk:.2%}  [{rl_status}]", y, rl_col, W, MX)
        y -= 10
        y = G._section_title(c, "Risk Components", y, MX, rl_col)
        for k, v in [
            ("Fusion Risk Input (Stage 6)", f"{float(s7.get('fused_risk', s6.get('fused_risk',0))):.4f}"),
            ("Final Blended Risk",         f"{final_risk:.4f}  (70% TFB + 20% TFM + 5% Fused + 5% RL)"),
            ("Weapon Override Applied",
             "YES, final risk floored at 0.85" if s6.get("weapon_flag") else "NO"),
            ("RL Agent Status",            rl_status),
            ("Risk Classification",
             "CRITICAL (>= 0.85)" if final_risk >= 0.85 else
             "HIGH_RISK (>= 0.65)" if final_risk >= 0.65 else
             "MODERATE (>= 0.40)"  if final_risk >= 0.40 else "LOW_RISK (< 0.40)"),
        ]:
            y = G._kv_row(c, k, v, MX + 6, y, key_w=250)
        y -= 8

        y = G._section_title(c, "What the RL Agent Sees (Observation Space)", y, MX, rl_col)
        c.setFont("Helvetica-Oblique", 8); c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "At each decision step the agent receives five named inputs described below.",
            MX + 6, y, BW - 12, "Helvetica-Oblique", 8, 11)
        y -= 6

        for entry_title, entry_desc in [
            ("ae_signals  (32 x 6)",
             "Sliding window of AE reconstruction errors: raw error, frame delta, acceleration, "
             "short EMA, long EMA, and spike ratio. Allows the agent to track anomaly trends over time."),
            ("latent_motion  (1536 values)",
             "AE channel statistics combined with motion tracking data. Correlates movement patterns "
             "with reconstruction error to distinguish genuine events from artefacts."),
            ("latent_stack  (2560 values)",
             "All pipeline signals combined: AE window, TimeSformer logits, audio features, "
             "tracking features, context vector, and Fusion outputs."),
            ("spatial  (196 values, 14x14 grid)",
             "Spatial heatmap of the frame. Records weapon detection locations and pixel-level "
             "variance per cell. Lets the agent weight spatially localised evidence."),
            ("context  (5 values)",
             "High-level snapshot: normalised AE score, binary classifier confidence, audio score, "
             "weapon flag, and motion intensity."),
        ]:
            if y < 80: break
            c.setFont("Helvetica-Bold", 8.5); c.setFillColor(colors.HexColor(G.C_DARK))
            c.drawString(MX + 8, y, entry_title); y -= 12
            y = G._wrap_text(c, entry_desc, MX + 16, y, BW - 28, "Helvetica", 8.5, 11)
            y -= 5

        if s7.get("interpretation"):
            y -= 4
            y = G._section_title(c, "Interpretation", y, MX, rl_col)
            y = G._wrap_text(c, s7["interpretation"], MX + 8, y, BW - 16, "Helvetica", 9, 13)

        y -= 4; y = G._disclaimer_box(c, y, W); G._footer(c, ts, W)

        # ── Page 10: Top Anomalous Frames ─────────────────────────────────
        frames_data  = s0.get("frames", [])
        scores_list  = s0.get("scores", [])
        samp_indices = s0.get("sampled_indices", [])
        if frames_data and scores_list:
            c.showPage()
            G._page_header(c, "TOP ANOMALOUS FRAMES", G.C_RED, W, H)
            y = G._body_top(H)
            c.setFont("Helvetica-Oblique", 8.5); c.setFillColor(colors.HexColor(G.C_GREY))
            y = G._wrap_text(c,
                "The three frames with the highest AE reconstruction error. High error means the AE "
                "could not reproduce the frame accurately from its learned representation of normal footage.",
                MX, y, BW, "Helvetica-Oblique", 8.5, 12)
            y -= 8
            for i, fd in enumerate(G.extract_top_frames(frames_data, scores_list,
                                    samp_indices or list(range(len(scores_list))), n=3), 1):
                tp    = f"/tmp/af_v3_{i}.jpg"
                Image.fromarray(fd["frame"]).save(tp)
                ratio    = fd["score"] / max(threshold, 1e-8)
                severity = "CRITICAL" if ratio >= 2.0 else "HIGH" if ratio >= 1.0 else "BELOW THRESHOLD"
                c.setFillColor(colors.HexColor(G.C_DARK)); c.setFont("Helvetica-Bold", 9)
                c.drawString(MX, y,
                    f"Frame {i}  |  {ts_of(fd['index'])}  |  "
                    f"Error: {fd['score']:.6f}  |  {ratio:.2f}x threshold  |  Severity: {severity}")
                y -= 14
                img_h = 155
                c.drawImage(tp, MX, y - img_h, width=BW, height=img_h, preserveAspectRatio=True)
                y -= img_h + 12
                if y < 80: break
        G._footer(c, ts, W)

        # ── Page 11: Stage 8 (LLaVA) ──────────────────────────────────────
        if s8 and (s8.get("unified_narrative") or s8.get("frame_descriptions")):
            c.showPage()
            G._page_header(c, "STAGE 8 (LLaVA, EXPERIMENTAL): AI VISUAL ANALYSIS", G.C_PURPLE, W, H)
            y = G._body_top(H)
            y = G._experimental_warning_box(c,
                "EXPERIMENTAL: LLaVA output is a language model generation and may contain "
                "hallucinations. Always verify descriptions against the actual frames.",
                y, W, MX)
            c.setFont("Helvetica-Oblique", 8.5); c.setFillColor(colors.HexColor(G.C_GREY))
            binary_note = s8.get("binary_result", "Unknown")
            y = G._wrap_text(c,
                f"LLaVA receives four key frames with contextual metadata (weapon detections, motion, "
                f"audio) and generates a natural-language description for each. Binary classifier "
                f"result was '{binary_note}'. All four descriptions are synthesised below into a "
                f"single coherent scene narrative. Raw per-frame outputs are in results.json.",
                MX, y, BW, "Helvetica-Oblique", 8.5, 12)
            y -= 10

            ctx = s8.get("context_used", {})
            if ctx:
                y = G._section_title(c, "Context Provided to LLaVA", y, MX, G.C_PURPLE)
                for k, v in [
                    ("Weapon Information",  ctx.get("yolo_summary","?")),
                    ("Motion Behaviours",   ", ".join(ctx.get("behaviors",[])) or "None"),
                    ("Audio Status",        ctx.get("audio_status","?")),
                    ("Audio Score",         f"{ctx.get('audio_score',0):.3f}"),
                    ("Binary Classifier",   binary_note),
                    ("Frames Analysed",     "4 keyframes: clip start, AE onset, AE peak, AE end"),
                ]:
                    y = G._kv_row(c, k, v, MX + 6, y, key_w=185)
                y -= 8

            y = G._section_title(c, "Scene Narrative (Synthesised)", y, MX, G.C_PURPLE)
            c.setFont("Helvetica-Oblique", 8); c.setFillColor(colors.HexColor(G.C_GREY))
            c.drawString(MX + 6, y,
                "Coherent summary derived from all four frame analyses, with redundancy removed.")
            y -= 14

            narrative = s8.get("unified_narrative", "") or " ".join(
                s8.get("frame_descriptions", {}).get(r, "")
                for r in ("baseline","anom_start","peak","anom_end")).strip() or "No description available."

            # Measure narrative height
            import io
            from reportlab.pdfgen.canvas import Canvas as _TC2
            _tc = _TC2(io.BytesIO()); _tc.setFont("Helvetica", 9)
            _words, _line, _nlines = narrative.split(), "", 0
            for _w in _words:
                _test = _line + _w + " "
                if _tc.stringWidth(_test, "Helvetica", 9) < BW - 24: _line = _test
                else: _nlines += 1; _line = _w + " "
            if _line.strip(): _nlines += 1
            narr_box_h = min(_nlines * 13 + 24, y - 90)

            c.setFillColor(colors.HexColor("#f4ecf7"))
            c.setStrokeColor(colors.HexColor("#6c3483")); c.setLineWidth(0.8)
            c.roundRect(MX, y - narr_box_h, BW, narr_box_h, 5, fill=True, stroke=True)
            G._wrap_text(c, narrative, MX + 10, y - 11, BW - 24,
                         "Helvetica", 9, 13, min_y=y - narr_box_h + 10)
            y -= narr_box_h + 14

            # Keyframes panel
            fd_map          = s8.get("frame_descriptions", {})
            frames_d_llava  = s0.get("frames", [])
            peak_idx_llava  = int(s0.get("peak_frame_idx", 0))
            anom_s_llava    = int(s0.get("anomaly_start_frame", 0))
            anom_e_llava    = int(s0.get("anomaly_end_frame", peak_idx_llava))
            n_f_llava       = len(frames_d_llava)
            frame_idx_map   = {
                "baseline":   0,
                "anom_start": min(anom_s_llava, n_f_llava - 1) if n_f_llava else 0,
                "peak":       min(peak_idx_llava, n_f_llava - 1) if n_f_llava else 0,
                "anom_end":   min(anom_e_llava, n_f_llava - 1) if n_f_llava else 0,
            }
            frame_roles = [("baseline","Frame 1: Beginning"),("anom_start","Frame 2: AE Onset"),
                           ("peak","Frame 3: AE Peak"),("anom_end","Frame 4: AE End")]
            if frames_d_llava and y > 130:
                y = G._section_title(c, "Keyframes Used in Scene Narrative", y, MX, G.C_PURPLE)
                c.setFont("Helvetica-Oblique", 8); c.setFillColor(colors.HexColor(G.C_GREY))
                c.drawString(MX + 6, y, "Four frames provided to LLaVA as visual input.")
                y -= 14
                n_kf = len(frame_roles)
                kf_w = (BW - (n_kf - 1) * 6) / n_kf
                kf_h = min(90, kf_w * 0.65)
                if y - kf_h - 36 > 40:
                    for kf_i, (role, label) in enumerate(frame_roles):
                        try:
                            tmp_kf = f"/tmp/llava_kf_{kf_i}.jpg"
                            Image.fromarray(frames_d_llava[frame_idx_map[role]]).save(tmp_kf)
                            cx = MX + kf_i * (kf_w + 6)
                            c.drawImage(tmp_kf, cx, y - kf_h, width=kf_w - 4,
                                        height=kf_h, preserveAspectRatio=True)
                            c.setFont("Helvetica-Bold", 7); c.setFillColor(colors.HexColor(G.C_DARK))
                            c.drawString(cx, y - kf_h - 11, label)
                            raw_desc = fd_map.get(role, "")
                            if raw_desc:
                                max_chars = max(1, int((kf_w - 8) / 3.3))
                                snippet   = raw_desc[:max_chars] + ("…" if len(raw_desc) > max_chars else "")
                                c.setFont("Helvetica-Oblique", 6)
                                c.setFillColor(colors.HexColor(G.C_GREY))
                                c.drawString(cx, y - kf_h - 22, snippet)
                        except Exception:
                            pass
                    y -= kf_h + 36
                y -= 8

            # Signal cross-reference
            if y > 130:
                y = G._section_title(c, "Signal Cross-Reference", y, MX, G.C_PURPLE)
                c.setFont("Helvetica-Oblique", 8); c.setFillColor(colors.HexColor(G.C_GREY))
                y = G._wrap_text(c,
                    "Deterministic system signals used as ground truth context for validation.",
                    MX + 6, y, BW - 12, "Helvetica-Oblique", 8, 11)
                y -= 6
                for k, v in [
                    ("AE Peak Timestamp",       ts_of(s0.get("peak_frame_idx",0))),
                    ("AE Max Error vs Threshold",
                     f"{s0.get('max_error',0):.6f} vs {threshold:.6f} "
                     f"({'exceeds' if s0.get('max_error',0) > threshold else 'below'})"),
                    ("Binary Classification",   f"{s1.get('classification','?')} ({s1.get('confidence',0):.1%})"),
                    ("Anomaly Type",
                     f"{s2.get('class','N/A')} ({s2.get('confidence',0):.1%})"
                     if s2.get("class") not in ("Skipped","Unknown",None) else "N/A"),
                    ("Weapon Detected",         "YES" if s3.get("weapon_flag") else "NO"),
                    ("Audio Status",            s4.get("status","N/A")),
                    ("Motion Behaviours",       ", ".join(s5.get("behaviors",[])) or "None"),
                    ("Final Risk",              f"{s7.get('final_risk', s6.get('fused_risk',0)):.1%}"),
                ]:
                    y = G._kv_row(c, k, v, MX + 6, y, key_w=210)
                    if y < 70: break

            y -= 4; y = G._disclaimer_box(c, y, W); G._footer(c, ts, W)

        c.save()
        return output_path

    @staticmethod
    def generate_all(results: dict) -> dict:
        s0        = results.get("stage0") or {}
        scores    = s0.get("scores", [])
        samp      = s0.get("sampled_indices", [])
        threshold = results.get("threshold", Config.SAFE_THRESHOLD_LEGACY)
        peak_si   = s0.get("peak_score_idx", 0)
        fps       = float(s0.get("fps", 25.0))
        if scores:
            results["graph_path"] = EnhancedReportGenerator.generate_deviation_graph(
                scores, threshold, [peak_si], samp, fps, "/tmp/deviation_graph_v3.png")
        results["pdf_path"] = EnhancedReportGenerator.generate_enhanced_pdf(
            results, "/tmp/forensic_report_v3.pdf")
        return results


# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE — SCENE SOLVER v3
# ══════════════════════════════════════════════════════════════════════════════
class SceneSolverPipeline:
    """
    Stage ordering (enforced):
      0: AE              — primary anomaly signal
      1: TF Binary       — semantic gate (LOGIC-NORMAL: overrides status to SAFE if Normal)
      2: TF Multi-class  — skipped when Stage 1 = Normal
      3: YOLO Weapon     — skipped when Stage 1 = Normal
      4: Audio           — always runs
      5: Tracking        — always runs
      6: Fusion          — always runs
      7: RL3033          — always runs, FINAL stage
      8: LLaVA           — always runs when enable_llava=True, neutral prompts if Stage 1 = Normal
    """

    def __init__(self, config: Optional[Config] = None):
        self.config       = config or Config()
        self.stage0       = Stage0AE(self.config)
        self.stage1       = Stage1TSBinary(self.config)
        self.stage2       = Stage2TSMulti(self.config)
        self.stage3       = Stage3YOLOWeapon(self.config)
        self.stage4       = Stage4Audio(self.config)
        self.stage5       = Stage5Tracking(self.config)
        self.stage6       = Stage6Fusion(self.config)
        self.stage7       = Stage7RL3033(self.config)
        self.stage8       = Stage8LLaVA(self.config)
        self.alert_system = AlertSystem(self.config)
        self.reporter     = EnhancedReportGenerator()

    def _make_output_dir(self, video_path: str, base: str) -> str:
        name = os.path.splitext(os.path.basename(video_path))[0]
        ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
        out  = os.path.join(base, "analysis_results", name, ts)
        os.makedirs(out, exist_ok=True)
        return out

    @staticmethod
    def _skipped_stage2() -> dict:
        return {"class": "Skipped", "confidence": 0.0, "top3": [],
                "all_probabilities": {}, "logits": [0.0] * 7, "sampling_method": "skipped"}

    @staticmethod
    def _skipped_stage3() -> dict:
        return {"weapon_flag": False, "weapon_count": 0, "dominant_class": "None",
                "best_detection": {}, "all_detections": [], "top_frames_annotated": [],
                "spatial_heatmap": np.zeros(196, dtype=np.float32), "summary": "Skipped, Stage 1 = Normal"}

    @staticmethod
    def _disabled_stage5() -> dict:
        return {"motion_intensity": 0.0, "behaviors": ["STATIC"], "track_count": 0,
                "tracking_features": np.zeros(20, dtype=np.float32), "status": "DISABLED"}

    @staticmethod
    def _disabled_stage7(fused_risk: float) -> dict:
        return {"final_risk": fused_risk, "status": "DISABLED",
                "interpretation": "RL agent disabled. Using fused risk from Stage 6."}

    def analyze(self, video_path: str, base_output_dir: Optional[str] = None,
                enable_llava: bool = True, enable_rl: bool = True,
                enable_tracking: bool = True) -> dict:
        base = base_output_dir or self.config.DRIVE_ROOT
        out  = self._make_output_dir(video_path, base)
        log.info("=" * 65)
        log.info(f"  SCENE SOLVER v3  |  {os.path.basename(video_path)}")
        log.info(f"  Output: {out}")
        log.info("=" * 65)

        results: dict = {"video_path": video_path, "output_directory": out}

        # Stage 0
        log.info("\n== STAGE 0: AE ==")
        stage0               = self.stage0.analyze(video_path)
        results["stage0"]    = stage0
        results["threshold"] = stage0["threshold"]
        frames               = stage0["frames"]
        scores               = stage0["scores"]
        sampled_indices      = stage0["sampled_indices"]
        peak_frame_idx       = stage0["peak_frame_idx"]
        peak_score_idx       = stage0.get("peak_score_idx", 0)
        threshold            = stage0["threshold"]
        anomaly_start_frame  = stage0.get("anomaly_start_frame", 0)
        anomaly_end_frame    = stage0.get("anomaly_end_frame", peak_frame_idx)
        ae_signals           = self.stage0.build_ae_signals(scores, peak_score_idx)

        # Stage 1
        log.info("\n== STAGE 1: TimeSformer Binary ==")
        stage1           = self.stage1.analyze(frames, peak_frame_idx)
        results["stage1"]     = stage1
        is_suspicious        = (stage1.get("classification") == "Suspicious")
        ts_binary_logits     = self.stage1.get_logits()
        results["status"]    = stage0["status"] if is_suspicious else "SAFE"
        if not is_suspicious:
            log.info("[Pipeline] Stage 1 = Normal => overall status overridden to SAFE")

        # Stage 2
        log.info("\n== STAGE 2: TimeSformer Multi-class ==")
        stage2 = (self.stage2.analyze(frames, peak_frame_idx, scores, sampled_indices)
                  if is_suspicious else self._skipped_stage2())
        if not is_suspicious:
            log.info("[Stage2-TSMulti] Skipped (Stage 1 = Normal)")
        results["stage2"] = stage2
        ts_multi_logits   = self.stage2.get_logits()

        # Stage 3
        log.info("\n== STAGE 3: YOLO Weapon ==")
        stage3 = (self.stage3.analyze(frames, anomaly_start_frame, anomaly_end_frame)
                  if is_suspicious else self._skipped_stage3())
        if not is_suspicious:
            log.info("[Stage3-YOLO] Skipped (Stage 1 = Normal)")
        results["stage3"] = stage3

        # Stage 4
        log.info("\n== STAGE 4: Audio ==")
        stage4            = self.stage4.analyze(video_path)
        results["stage4"] = stage4

        # Stage 5
        if enable_tracking:
            log.info("\n== STAGE 5: Tracking ==")
            stage5 = self.stage5.analyze(frames, anomaly_start_frame, n_frames=32)
        else:
            log.info("\n== STAGE 5: Tracking DISABLED ==")
            stage5 = self._disabled_stage5()
        results["stage5"] = stage5

        # Stage 6
        log.info("\n== STAGE 6: Fusion ==")
        stage6            = self.stage6.fuse(stage0, stage1, stage2, stage3, stage4, stage5)
        results["stage6"] = stage6

        # Stage 7
        if enable_rl:
            log.info("\n== STAGE 7: RL3033 ==")
            stage7 = self.stage7.analyze(ae_signals, ts_binary_logits, ts_multi_logits,
                                          stage3, stage4, stage5, stage6, frames, peak_frame_idx)
        else:
            log.info("\n== STAGE 7: RL3033 DISABLED ==")
            stage7 = self._disabled_stage7(float(stage6.get("fused_risk", 0.0)))
        results["stage7"] = stage7

        # Sync risk level from Stage 7
        _fr = float(stage7.get("final_risk", stage6.get("fused_risk", 0.0)))
        stage6["risk_level"] = ("CRITICAL" if _fr >= 0.85 else "HIGH" if _fr >= 0.70
                                else "MODERATE" if _fr >= 0.40 else "LOW")
        results["risk_label_final"] = stage6["risk_level"]
        # Keep "status" in sync with the final fused/RL risk, not just the AE/Stage-1 gate
        results["status"] = "CRITICAL" if stage6["risk_level"] in ("CRITICAL", "HIGH") else "SAFE"

        # Stage 8
        if enable_llava:
            log.info("\n== STAGE 8: LLaVA ==")
            stage8 = self.stage8.analyze(frames, peak_frame_idx, anomaly_start_frame,
                                          anomaly_end_frame, stage3, stage4, stage5,
                                          binary_result=stage1.get("classification", "Unknown"))
        else:
            stage8 = None
        results["stage8"] = stage8

        # Alert
        alert            = self.alert_system.evaluate_and_fire(results)
        results["alert"] = alert

        # Visualisations + PDF
        results = EnhancedReportGenerator.generate_all(results)

        self._save_outputs(results, out)
        self._print_summary(results)
        return results

    def _save_outputs(self, results: dict, out_dir: str):
        def _default(o):
            if isinstance(o, np.ndarray): return o.tolist()
            return str(o)

        # Sanitise for JSON serialisation
        save = dict(results)
        if save.get("stage0"):
            save["stage0"] = {k: v for k, v in save["stage0"].items() if k != "frames"}
        for skey in ("stage3", "stage4", "stage5", "stage6"):
            if save.get(skey):
                sx = dict(save[skey])
                for arr_key in ("spatial_heatmap", "audio_features_vec",
                                "tracking_features", "context_vector"):
                    if arr_key in sx and isinstance(sx[arr_key], np.ndarray):
                        sx[arr_key] = sx[arr_key].tolist()
                if skey == "stage3":
                    sx.pop("all_detections", None)
                    sx.pop("top_frames_annotated", None)
                save[skey] = sx

        with open(os.path.join(out_dir, "results.json"), "w") as f:
            json.dump(save, f, indent=2, default=_default)

        # Summary text
        s0  = results.get("stage0") or {}
        fps = float(s0.get("fps", 25.0))
        ts  = lambda fi: frame_to_timestamp(int(fi), fps)
        s1  = results.get("stage1") or {}
        s2  = results.get("stage2") or {}
        s3  = results.get("stage3") or {}
        s4  = results.get("stage4") or {}
        s5  = results.get("stage5") or {}
        s6  = results.get("stage6") or {}
        s7  = results.get("stage7") or {}
        s8  = results.get("stage8") or {}

        with open(os.path.join(out_dir, "summary.txt"), "w") as f:
            f.write(f"SCENE SOLVER v3  |  {os.path.basename(results.get('video_path',''))}\n")
            f.write("=" * 65 + "\n\n")
            f.write(f"Status              : {results.get('status')}\n")
            f.write(f"Threshold           : {results.get('threshold',0):.6f}\n")
            f.write(f"AE Max Error        : {s0.get('max_error',0):.6f}\n")
            f.write(f"Peak Timestamp      : {ts(s0.get('peak_frame_idx',0))}\n")
            if s1: f.write(f"TF Binary           : {s1.get('classification')} ({s1.get('confidence',0):.1%})\n")
            if s2:
                note = " [Skipped]" if s2.get("class") == "Skipped" else f" ({s2.get('confidence',0):.1%}) [{s2.get('sampling_method','?')}]"
                f.write(f"Anomaly Type        : {s2.get('class','?')}{note}\n")
            if s3: f.write(f"Weapon Detection    : {'Detected' if s3.get('weapon_flag') else 'None'}\n")
            if s4:
                ok = s4.get("audio_present", True)
                f.write(f"Audio               : {'No audio' if not ok else s4.get('status','?')}"
                        + (f"  score={s4.get('audio_score',0):.3f}" if ok else "") + "\n")
            if s5: f.write(f"Tracking            : motion={s5.get('motion_intensity',0):.3f} behaviors={s5.get('behaviors',[])} [{s5.get('status')}]\n")
            if s6:
                f.write(f"Fusion Risk         : {s6.get('fused_risk',0):.1%} [{s6.get('risk_level')}]\n")
                for line in s6.get("reasoning_trace", []):
                    f.write(f"  {line}\n")
            if s7: f.write(f"RL3033 Final        : {s7.get('final_risk',0):.1%} [{s7.get('status')}]\n")
            if results.get("alert"):
                al = results["alert"]
                f.write(f"ALERT               : {al['level']} risk={al['final_risk']:.1%}\n")
            if s8 and s8.get("unified_narrative"):
                f.write(f"\nAI Visual Narrative:\n{s8['unified_narrative']}\n")
            elif s8 and s8.get("description"):
                f.write(f"\nAI Visual Description:\n{s8['description']}\n")

        # PDF copy
        pdf_src = results.get("pdf_path")
        if pdf_src and os.path.exists(pdf_src):
            dest = os.path.join(out_dir, "forensic_report.pdf")
            shutil.copy2(pdf_src, dest)
            results["pdf_path_local"] = dest
            colab_dest = "/content/scene_solver_report.pdf"
            try:
                shutil.copy2(pdf_src, colab_dest)
                log.info(f"[Output] PDF saved to {colab_dest}")
                try:
                    from google.colab import files as colab_files
                    colab_files.download(colab_dest)
                except Exception:
                    pass
            except Exception:
                pass
        log.info(f"[Output] Saved to {out_dir}")

    def _print_summary(self, results: dict):
        s0  = results.get("stage0") or {}
        fps = float(s0.get("fps", 25.0))
        ts  = lambda fi: frame_to_timestamp(int(fi), fps)
        s1  = results.get("stage1") or {}
        s2  = results.get("stage2") or {}
        s3  = results.get("stage3") or {}
        s4  = results.get("stage4") or {}
        s5  = results.get("stage5") or {}
        s6  = results.get("stage6") or {}
        s7  = results.get("stage7") or {}
        s8  = results.get("stage8") or {}
        al  = results.get("alert")

        print("\n" + "=" * 65)
        print("  SCENE SOLVER v3, SUMMARY")
        print("=" * 65)
        print(f"  Overall Status        : {results.get('status')}")
        print(f"  Dynamic Threshold     : {results.get('threshold',0):.6f}")
        print(f"\n  [Stage 0 (AE)]  max={s0.get('max_error',0):.6f}  avg={s0.get('avg_error',0):.6f}  peak={ts(s0.get('peak_frame_idx',0))}")
        print(f"  [Stage 1 (TF Binary)] {s1.get('classification','?')} ({s1.get('confidence',0):.2%})")
        if s2: print(f"  [Stage 2 (TF Multi)]  {s2.get('class','?')} ({s2.get('confidence',0):.2%}) [{s2.get('sampling_method','?')}]")
        if s3: print(f"  [Stage 3 (YOLO)]      {'Detected' if s3.get('weapon_flag') else 'No weapons detected'}")
        if s4:
            ok = s4.get("audio_present", True)
            print(f"  [Stage 4 (Audio)]     {'No audio' if not ok else s4.get('status','?')}" +
                  (f"  score={s4.get('audio_score',0):.3f}" if ok else ""))
        if s5: print(f"  [Stage 5 (Tracking)]  motion={s5.get('motion_intensity',0):.3f}  {s5.get('behaviors',[])}  [{s5.get('status')}]")
        if s6: print(f"  [Stage 6 (Fusion)]    {s6.get('fused_risk',0):.2%}  [{s6.get('risk_level','?')}]")
        if s7: print(f"  [Stage 7 (RL3033)]    final={s7.get('final_risk',0):.2%}  [{s7.get('status','?')}]")
        if al: print(f"\n  ALERT: {al['level']}  |  final_risk={al['final_risk']:.2%}")
        if s8:
            narr = s8.get("unified_narrative") or s8.get("description","")
            if narr: print(f"\n  [Stage 8 (LLaVA)]\n    {narr[:300]}{'...' if len(narr) > 300 else ''}")
        print("=" * 65 + "\n")

    def start_stream(self, source: str, window_frames: int = 50, on_alert: Any = None) -> RealTimeStream:
        stream = RealTimeStream(self)
        stream.start(source, window_frames=window_frames, on_alert=on_alert)
        return stream


# ══════════════════════════════════════════════════════════════════════════════
# READY MESSAGE
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("[STATUS] SCENE SOLVER v3, REASONING SYSTEM READY")
print("=" * 65)
print(f"Device             : {Config.DEVICE}")
print(f"RL3033 Available   : {RL_AVAILABLE}")
print(f"YOLO Available     : {YOLO_AVAILABLE}")
print(f"Librosa Available  : {LIBROSA_AVAILABLE}")
print()
print("Usage:")
print("  pipeline = SceneSolverPipeline()")
print("  results  = pipeline.analyze('/path/to/video.mp4')")
print("=" * 65)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

[HF] All SceneSolver assets are available.
[STATUS] SCENE SOLVER v3, REASONING SYSTEM READY
Device             : cuda
RL3033 Available   : True
YOLO Available     : True
Librosa Available  : True

Usage:
  pipeline = SceneSolverPipeline()
  results  = pipeline.analyze('/path/to/video.mp4')


In [17]:
#from google.colab import drive
#drive.mount('/content/drive')

pipeline = SceneSolverPipeline()
# results = pipeline.analyze('/content/drive/MyDrive/solver1/AnomalyDetectionDataset/Explosion/Explosion001_x264.mp4')
# Without LLaVA (faster):
# results = pipeline.analyze('/path/to/video.mp4', enable_llava=False)

In [18]:
!pip install -q gradio

In [19]:
# ══════════════════════════════════════════════════════════════════════════════
# GRADIO WEB APP  : CrimeSceneSolver  (refactored)
#
# Run AFTER Cell 2 (pipeline) and Cell 4 (gradio install).
# Agent Escalation Layer below this cell for the full experience.
# ══════════════════════════════════════════════════════════════════════════════

import gradio as gr
import numpy as np
import json, os, traceback, smtplib, ssl, re, base64
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from PIL import Image

# ─────────────────────────────────────────────────────────────────────────────
# SHARED CONSTANTS  (used by both the web app and the agent layer)
# ─────────────────────────────────────────────────────────────────────────────
ADMIN_EMAIL  = "garoldwerner2512@gmail.com"
SMTP_SERVER  = "smtp.gmail.com"
SMTP_PORT    = 465
SENDER_EMAIL = os.environ.get("SENDER_EMAIL", "")
SENDER_PASS  = os.environ.get("SENDER_PASS",  "")

# Design tokens — pitch black / white / orange-saffron
C_BG     = "#000000"
C_ORANGE = "#FF8C00"
C_RED    = "#FF4500"
C_GREEN  = "#4CAF50"

# Shared HTML blocks
NAV_HTML = """
<div style="background:#000;border-bottom:1px solid #1a1a1a;
            padding:16px 36px;display:flex;justify-content:space-between;align-items:center;">
  <span style="font-family:'Roboto Mono',monospace;font-size:.92rem;
               letter-spacing:.15em;color:#fff;text-transform:uppercase;
               font-weight:700;-webkit-font-smoothing:antialiased;">
    CrimeSceneSolver
  </span>
  <span style="font-family:'Roboto Mono',monospace;font-size:.6rem;
               letter-spacing:.1em;color:#555;text-transform:uppercase;">
    Scene Solver v3
  </span>
</div>
"""

FOOTER_HTML = """
<div style="border-top:1px solid #0d0d0d;margin-top:16px;">
  <div style="background:#0d0d0d;border-bottom:1px solid #1a1a1a;padding:14px 32px;">
    <div style="max-width:860px;margin:0 auto;display:flex;align-items:flex-start;gap:12px;">
      <svg style="flex-shrink:0;margin-top:2px;" width="14" height="14" viewBox="0 0 24 24"
           fill="none" stroke="#FF8C00" stroke-width="2">
        <circle cx="12" cy="12" r="10"/>
        <line x1="12" y1="8" x2="12" y2="12"/>
        <line x1="12" y1="16" x2="12.01" y2="16"/>
      </svg>
      <p style="font-family:'Roboto Mono',monospace;font-size:.62rem;
                letter-spacing:.04em;color:#888;line-height:1.7;margin:0;font-weight:500;">
        <span style="color:#FF8C00;font-weight:700;">DISCLAIMER &mdash; </span>
        This report is produced by machine learning models and should be treated as an
        investigative aid, not a definitive conclusion. Human expert review is strongly
        recommended before any operational action is taken.
      </p>
    </div>
  </div>
  <div style="text-align:center;padding:18px;font-family:'Roboto Mono',monospace;
              font-size:.6rem;letter-spacing:.1em;color:#444;text-transform:uppercase;">
    CrimeSceneSolver &nbsp;&middot;&nbsp; Scene Solver v3 &nbsp;&middot;&nbsp;
    Confidential &nbsp;&middot;&nbsp; ML outputs require expert review
  </div>
</div>
"""


def sec_label(text: str) -> str:
    """Shared section-label helper used across the web app and agent layer."""
    return (f"<div style='font-family:\"Roboto Mono\",monospace;font-size:.62rem;"
            f"letter-spacing:.14em;text-transform:uppercase;color:#ddd;"
            f"margin-bottom:10px;font-weight:800;-webkit-font-smoothing:antialiased;"
            f"border-left:3px solid #FF8C00;padding-left:10px;'>{text}</div>")


def risk_color(r: float) -> str:
    if r >= 0.70: return C_RED
    if r >= 0.40: return C_ORANGE
    return C_GREEN


# ─────────────────────────────────────────────────────────────────────────────
# EMAIL
# ─────────────────────────────────────────────────────────────────────────────
def send_contact_email(name: str, email: str, phone: str, message: str) -> bool:
    if not SENDER_EMAIL or not SENDER_PASS:
        with open("/tmp/css_contact.log", "a") as f:
            f.write(f"[CONTACT] {name} | {email} | {phone}\n")
        return False
    try:
        msg = MIMEMultipart("alternative")
        msg["Subject"] = f"[CrimeSceneSolver] Contact: {name}"
        msg["From"]    = SENDER_EMAIL
        msg["To"]      = ADMIN_EMAIL
        msg.attach(MIMEText(f"Name:{name}\nEmail:{email}\nPhone:{phone}\nMessage:\n{message}\n", "plain"))
        ctx = ssl.create_default_context()
        with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT, context=ctx) as srv:
            srv.login(SENDER_EMAIL, SENDER_PASS)
            srv.sendmail(SENDER_EMAIL, ADMIN_EMAIL, msg.as_string())
        return True
    except Exception:
        return False


# ─────────────────────────────────────────────────────────────────────────────
# PIPELINE WRAPPER
# ─────────────────────────────────────────────────────────────────────────────
def run_pipeline(video_path: str) -> dict:
    pipeline = SceneSolverPipeline()
    results  = pipeline.analyze(
        video_path=video_path, base_output_dir="/tmp/css_webapp",
        enable_llava=True, enable_rl=True, enable_tracking=True,
    )
    s0 = results.get("stage0") or {}
    s1 = results.get("stage1") or {}
    s2 = results.get("stage2") or {}
    s3 = results.get("stage3") or {}
    s4 = results.get("stage4") or {}
    s5 = results.get("stage5") or {}
    s6 = results.get("stage6") or {}
    s7 = results.get("stage7") or {}
    s8 = results.get("stage8") or {}
    fps = float(s0.get("fps", 25.0))

    def ts(fi): return frame_to_timestamp(int(fi), fps)

    risk_score = float(s7.get("final_risk", s6.get("fused_risk", 0.0)))
    rl_status  = s7.get("status", "")
    _rl_map    = {"CRITICAL":"CRITICAL","HIGH_RISK":"HIGH","MODERATE":"MODERATE","LOW_RISK":"LOW"}
    risk_label = (_rl_map.get(rl_status, "") or
                  results.get("risk_label_final") or
                  s6.get("risk_level", "LOW"))
    status    = results.get("status", "UNKNOWN")
    narrative = (s8 or {}).get("unified_narrative") or (s8 or {}).get("description") or ""

    report = {
        "overall_status":   status,
        "final_risk_score": f"{risk_score:.1%}",
        "risk_level":       risk_label,
        "dynamic_threshold": f"{results.get('threshold',0):.6f}",
        "ae_max_error":     f"{s0.get('max_error',0):.6f}",
        "ae_avg_error":     f"{s0.get('avg_error',0):.6f}",
        "peak_timestamp":   ts(s0.get("peak_frame_idx",0)),
        "anomaly_window":   f"{ts(s0.get('anomaly_start_frame',0))} \u2192 {ts(s0.get('anomaly_end_frame',0))}",
        "stage1_tf_binary": f"{s1.get('classification','?')}  ({s1.get('confidence',0):.1%})",
        "stage2_anomaly_type": f"{s2.get('class','N/A')}  ({s2.get('confidence',0):.1%})",
        "stage2_top3":      s2.get("top3", []),
        "stage2_all_probs": s2.get("all_probabilities", {}),
        "stage3_weapon":    ("YES — " + str(s3.get("dominant_class","?"))
                             if s3.get("weapon_flag") else "None detected"),
        "stage4_audio":     f"{s4.get('status','N/A')}  (score: {s4.get('audio_score',0):.4f})",
        "stage5_motion":    f"{s5.get('motion_intensity',0):.3f}  |  {', '.join(s5.get('behaviors',[]) or ['STATIC'])}",
        "stage6_fusion_risk": f"{s6.get('fused_risk',0):.1%}  [{s6.get('risk_level','?')}]",
        "stage6_reasoning": s6.get("reasoning_trace", []),
        "stage7_rl_final":  f"{s7.get('final_risk',0):.1%}  [{rl_status}]",
        "stage7_interpretation": s7.get("interpretation",""),
        "llava_narrative":  narrative,
        # raw values for charts
        "_stage1_conf":     float(s1.get("confidence", 0)),
        "_stage2_conf":     float(s2.get("confidence", 0)),
        "_stage2_top3_raw": s2.get("top3", []),
        "_stage2_all_probs": s2.get("all_probabilities", {}),
        "_risk_score":      risk_score,
        "_fused_risk":      float(s6.get("fused_risk", 0)),
        "_audio_score":     float(s4.get("audio_score", 0)),
        "_motion":          float(s5.get("motion_intensity", 0)),
    }

    frames_l  = s0.get("frames", [])
    scores_l  = s0.get("scores", [])
    samp_idx  = s0.get("sampled_indices", [])
    top_frame = None
    if frames_l and scores_l:
        psi       = int(s0.get("peak_score_idx", 0))
        pfi       = min(samp_idx[psi] if psi < len(samp_idx) else 0, len(frames_l) - 1)
        top_frame = frames_l[pfi]

    def _serial(o):
        if isinstance(o, np.ndarray): return o.tolist()
        if isinstance(o, np.integer): return int(o)
        if isinstance(o, np.floating): return float(o)
        return str(o)

    jd = {
        "status": status, "risk_score": round(risk_score, 4), "risk_level": risk_label,
        "threshold": round(float(results.get("threshold", 0)), 6),
        "stage0": {"max_error": round(float(s0.get("max_error",0)),6),
                   "avg_error": round(float(s0.get("avg_error",0)),6),
                   "peak_timestamp": ts(s0.get("peak_frame_idx",0)),
                   "anomaly_start": ts(s0.get("anomaly_start_frame",0)),
                   "anomaly_end":   ts(s0.get("anomaly_end_frame",0))},
        "stage1": {"classification": s1.get("classification","?"),
                   "confidence": round(float(s1.get("confidence",0)),4)},
        "stage2": {"class": s2.get("class","N/A"),
                   "confidence": round(float(s2.get("confidence",0)),4),
                   "top3": s2.get("top3",[])},
        "stage3": {"weapon_flag": bool(s3.get("weapon_flag",False)),
                   "dominant_class": s3.get("dominant_class","None"),
                   "weapon_count": int(s3.get("weapon_count",0))},
        "stage4": {"status": s4.get("status","N/A"),
                   "audio_score": round(float(s4.get("audio_score",0)),4)},
        "stage5": {"motion_intensity": round(float(s5.get("motion_intensity",0)),4),
                   "behaviors": s5.get("behaviors",[])},
        "stage6": {"fused_risk": round(float(s6.get("fused_risk",0)),4),
                   "risk_level": s6.get("risk_level","?"),
                   "reasoning": s6.get("reasoning_trace",[])},
        "stage7": {"final_risk": round(risk_score,4), "status": rl_status,
                   "interpretation": s7.get("interpretation","")},
        "llava_narrative": narrative,
    }

    pdf = (results.get("pdf_path_local") or results.get("pdf_path")
           or "/tmp/forensic_report_v3.pdf")
    if not os.path.exists(pdf): pdf = None

    graph_path = results.get("graph_path", "/tmp/deviation_graph_v3.png")
    if not os.path.exists(graph_path): graph_path = None

    return {"risk_score": risk_score, "risk_label": risk_label, "status": status,
            "report": report, "top_frame": top_frame, "json_data": jd,
            "pdf_path": pdf, "graph_path": graph_path}


# ─────────────────────────────────────────────────────────────────────────────
# HTML HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def img_to_data_uri(path: str) -> str:
    """Encode an image file as a base64 data-URI for inline HTML embedding."""
    if not path or not os.path.exists(path):
        return ""
    ext  = os.path.splitext(path)[1].lower().lstrip(".")
    mime = {"png":"image/png","jpg":"image/jpeg","jpeg":"image/jpeg"}.get(ext,"image/png")
    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime};base64,{encoded}"


def spinner_html() -> str:
    return """
<div style="text-align:center;padding:18px 0;">
  <div style="display:inline-block;width:40px;height:40px;
    border:3px solid rgba(255,255,255,0.07);border-top-color:#FF8C00;
    border-right-color:rgba(255,140,0,0.28);border-radius:50%;
    animation:arc-spin .75s cubic-bezier(0.6,0.2,0.4,0.8) infinite;"></div>
  <div style="font-family:'Roboto Mono',monospace;font-size:.64rem;
    letter-spacing:.12em;color:#FF8C00;margin-top:11px;text-transform:uppercase;font-weight:600;">
    Processing&hellip; this may take several minutes
  </div>
</div>"""


def no_result_html() -> str:
    return ("<div style='text-align:center;padding:60px 20px;color:#333;"
            "font-family:Charter,serif;font-size:1rem;letter-spacing:.04em;'>"
            "Upload footage and press ANALYSE to begin.</div>")


# ─────────────────────────────────────────────────────────────────────────────
# REPORT HTML BUILDER
# ─────────────────────────────────────────────────────────────────────────────
def build_report_html(report: dict, risk_score: float, risk_label: str,
                      status: str, graph_path: str = None) -> str:
    col = risk_color(risk_score)

    def numfmt(v: str) -> str:
        return re.sub(r'([\d]+\.?[\d]*%?)',
                      r"<span style='font-family:\"Times New Roman\",Times,serif;'>\1</span>", v)

    def row(label: str, value: str, mono: bool = False) -> str:
        vd    = numfmt(str(value))
        vfont = "font-family:'Roboto Mono',monospace;font-size:.8rem;" if mono else "font-size:.82rem;"
        return (f"<tr>"
                f"<td style='padding:9px 14px;color:#aaa;font-size:.78rem;"
                f"border-bottom:1px solid #222;white-space:nowrap;width:38%;'>{label}</td>"
                f"<td style='padding:9px 14px;color:#eee;{vfont}border-bottom:1px solid #222;'>{vd}</td>"
                f"</tr>")

    def section_hdr(title: str) -> str:
        return (f"<div style='font-size:.65rem;letter-spacing:.16em;text-transform:uppercase;"
                f"color:#FF8C00;padding:14px 0 6px 2px;border-bottom:2px solid #FF8C00;"
                f"margin-bottom:0;font-family:\"Roboto Mono\",monospace;font-weight:800;'>{title}</div>")

    def section(title: str, rows_html: str, extra: str = "") -> str:
        return (f"<div style='margin-bottom:26px;'>"
                + section_hdr(title)
                + f"<table style='width:100%;border-collapse:collapse;'>{rows_html}</table>"
                + extra + "</div>")

    # Risk badge
    is_crit    = (status == "CRITICAL")
    badge_text = f"{'ANOMALY DETECTED' if is_crit else 'NO ANOMALY DETECTED'} ({status})"
    badge = (f"<div style='text-align:center;padding:28px 20px;margin-bottom:28px;"
             f"background:{col};border-radius:4px;'>"
             f"<div style='font-size:3.8rem;font-weight:700;color:#fff;"
             f"font-family:\"Times New Roman\",Times,serif;line-height:1;'>"
             f"{risk_score:.1%}</div>"
             f"<div style='color:rgba(255,255,255,.85);font-size:.75rem;margin-top:10px;"
             f"letter-spacing:.14em;text-transform:uppercase;font-family:\"Roboto Mono\",monospace;"
             f"font-weight:600;'>{badge_text} &nbsp;|&nbsp; {risk_label} RISK</div>"
             f"</div>")

    # Raw chart values
    stage1_conf = float(report.get("_stage1_conf", 0))
    stage2_conf = float(report.get("_stage2_conf", 0))
    top3_raw    = report.get("_stage2_top3_raw", []) or []
    all_probs   = report.get("_stage2_all_probs", {}) or {}
    fused_risk  = float(report.get("_fused_risk", risk_score))
    audio_score = float(report.get("_audio_score", 0))
    motion      = float(report.get("_motion", 0))
    PIE_COLORS  = ["#c0392b","#2471a3","#d35400","#6c3483","#117a65","#1a5276","#7d6608"]

    # ── Donut pie (SVG) ──────────────────────────────────────────────────────
    def donut_pie_svg(probs: dict) -> str:
        if not probs: return ""
        import math
        items  = sorted(probs.items(), key=lambda x: x[1], reverse=True)
        labels = [i[0] for i in items]
        sizes  = [max(float(i[1]), 1e-6) for i in items]
        total  = sum(sizes)
        cx = cy = R = 90; R = 70; r = 36
        wedges = ""
        start  = -90
        for idx, (label, val) in enumerate(zip(labels, sizes)):
            sweep = (val / total) * 360
            end   = start + sweep
            x1o, y1o = cx + R * math.cos(math.radians(start)), cy + R * math.sin(math.radians(start))
            x2o, y2o = cx + R * math.cos(math.radians(end)),   cy + R * math.sin(math.radians(end))
            x1i, y1i = cx + r * math.cos(math.radians(end)),   cy + r * math.sin(math.radians(end))
            x2i, y2i = cx + r * math.cos(math.radians(start)), cy + r * math.sin(math.radians(start))
            lg   = 1 if sweep > 180 else 0
            fill = PIE_COLORS[idx % len(PIE_COLORS)]
            mid_a = start + sweep / 2
            lx = cx + (R + r) / 2 * math.cos(math.radians(mid_a))
            ly = cy + (R + r) / 2 * math.sin(math.radians(mid_a))
            num_lbl = (f"<text x='{lx:.1f}' y='{ly:.1f}' text-anchor='middle' "
                       f"dominant-baseline='middle' fill='#fff' font-size='9' font-weight='bold'>{idx+1}</text>"
                       if sweep > 15 else "")
            wedges += (f"<path d='M {x1o:.2f} {y1o:.2f} A {R} {R} 0 {lg} 1 {x2o:.2f} {y2o:.2f} "
                       f"L {x1i:.2f} {y1i:.2f} A {r} {r} 0 {lg} 0 {x2i:.2f} {y2i:.2f} Z' "
                       f"fill='{fill}' stroke='#000' stroke-width='1'/>{num_lbl}")
            start = end
        legend = "".join(
            f"<div style='display:flex;align-items:center;gap:7px;margin-bottom:5px;'>"
            f"<div style='width:11px;height:11px;min-width:11px;background:{PIE_COLORS[i%len(PIE_COLORS)]};border-radius:2px;'></div>"
            f"<span style='font-family:\"Roboto Mono\",monospace;font-size:.62rem;color:#ccc;'>"
            f"{i+1}. {cls}&nbsp;<span style='font-family:\"Times New Roman\",Times,serif;color:#fff;font-weight:700;'>"
            f"({v/total*100:.1f}%)</span></span></div>"
            for i, (cls, v) in enumerate(zip(labels, sizes))
        )
        svg = (f"<svg width='180' height='180' viewBox='0 0 180 180'>{wedges}"
               f"<circle cx='{cx}' cy='{cy}' r='{r-2}' fill='#050505'/>"
               f"<text x='{cx}' y='{cy}' text-anchor='middle' dominant-baseline='middle' "
               f"fill='#666' font-size='8'>ANOMALY</text>"
               f"<text x='{cx}' y='{cy+11}' text-anchor='middle' dominant-baseline='middle' "
               f"fill='#666' font-size='8'>CLASSES</text></svg>")
        return (f"<div style='display:flex;gap:16px;align-items:flex-start;padding:16px;"
                f"background:#050505;border:1px solid #1e1e1e;border-radius:3px;margin-top:6px;'>"
                f"<div style='flex-shrink:0;'>{svg}</div><div style='flex:1;'>{legend}</div></div>")

    # ── Horizontal bar chart ──────────────────────────────────────────────────
    def conf_bar_chart(top3: list) -> str:
        if not top3: return ""
        BAR_COLORS = ["#c0392b","#d35400","#5d6d7e"]
        rows = "".join(
            f"<div style='margin-bottom:10px;'>"
            f"<div style='display:flex;justify-content:space-between;align-items:center;margin-bottom:4px;'>"
            f"<span style='font-family:\"Roboto Mono\",monospace;font-size:.68rem;color:#ccc;'>{cls}</span>"
            f"<span style='font-family:\"Times New Roman\",Times,serif;color:#fff;font-weight:700;font-size:.82rem;margin-left:8px;'>{prob*100:.1f}%</span>"
            f"</div><div style='background:#1a1a1a;border-radius:2px;height:10px;overflow:hidden;'>"
            f"<div style='background:{BAR_COLORS[idx]};height:10px;width:{prob*100:.1f}%;border-radius:2px;'></div>"
            f"</div></div>"
            for idx, (cls, prob) in enumerate(top3[:3])
        )
        return (f"<div style='padding:14px 16px;background:#050505;border:1px solid #1e1e1e;"
                f"border-radius:3px;margin-top:6px;'>"
                f"<div style='font-family:\"Roboto Mono\",monospace;font-size:.6rem;"
                f"letter-spacing:.14em;color:#FF8C00;text-transform:uppercase;margin-bottom:12px;'>"
                f"Top-3 Anomaly Predictions</div>{rows}</div>")

    # ── Signal gauge bars ─────────────────────────────────────────────────────
    def risk_gauge(val: float, label: str, color: str) -> str:
        pct = min(100, int(val * 100))
        return (f"<div style='margin-bottom:10px;'>"
                f"<div style='display:flex;justify-content:space-between;align-items:center;margin-bottom:3px;'>"
                f"<span style='font-family:\"Roboto Mono\",monospace;font-size:.65rem;color:#aaa;'>{label}</span>"
                f"<span style='font-family:\"Times New Roman\",Times,serif;color:#fff;font-weight:700;font-size:.8rem;'>{val:.1%}</span>"
                f"</div><div style='background:#1a1a1a;border-radius:2px;height:8px;overflow:hidden;'>"
                f"<div style='background:{color};height:8px;width:{pct}%;border-radius:2px;'></div>"
                f"</div></div>")

    # ── Small donut pies for Stage 1 / Stage 2 ───────────────────────────────
    def small_donut(pct: float, color: str, label: str, size: int = 84) -> str:
        import math
        cx = cy = size // 2
        R  = cx - 8
        r  = R - 12
        sweep = pct * 360
        end_a = -90 + sweep
        x1,  y1  = cx + R*math.cos(math.radians(-90)),    cy + R*math.sin(math.radians(-90))
        x2,  y2  = cx + R*math.cos(math.radians(end_a)),  cy + R*math.sin(math.radians(end_a))
        xi1, yi1 = cx + r*math.cos(math.radians(end_a)),  cy + r*math.sin(math.radians(end_a))
        xi2, yi2 = cx + r*math.cos(math.radians(-90)),    cy + r*math.sin(math.radians(-90))
        lg = 1 if sweep > 180 else 0
        if pct >= 0.999:
            arc = (f"<circle cx='{cx}' cy='{cy}' r='{R}' fill='none' stroke='{color}' stroke-width='12'/>"
                   f"<circle cx='{cx}' cy='{cy}' r='{r}' fill='#050505'/>")
        else:
            arc = (f"<circle cx='{cx}' cy='{cy}' r='{R}' fill='none' stroke='#1a1a1a' stroke-width='12'/>"
                   f"<path d='M {x1:.2f} {y1:.2f} A {R} {R} 0 {lg} 1 {x2:.2f} {y2:.2f} "
                   f"L {xi1:.2f} {yi1:.2f} A {r} {r} 0 {lg} 0 {xi2:.2f} {yi2:.2f} Z' fill='{color}'/>"
                   f"<circle cx='{cx}' cy='{cy}' r='{r}' fill='#050505'/>")
        return (f"<div style='text-align:center;'>"
                f"<svg width='{size}' height='{size}' viewBox='0 0 {size} {size}'>"
                f"{arc}"
                f"<text x='{cx}' y='{cy}' text-anchor='middle' dominant-baseline='middle' "
                f"fill='#fff' font-family='Times New Roman,Times,serif' font-size='11' font-weight='700'>"
                f"{pct:.0%}</text></svg>"
                f"<div style='font-family:\"Roboto Mono\",monospace;font-size:.55rem;color:#666;margin-top:2px;'>"
                f"{label}</div></div>")

    charts_cls = (
        f"<div style='display:flex;gap:20px;padding:14px 16px;background:#050505;"
        f"border:1px solid #1e1e1e;border-radius:3px;margin-top:6px;align-items:flex-start;flex-wrap:wrap;'>"
        + small_donut(stage1_conf, '#FF8C00', 'TF BINARY · Stage 1')
        + small_donut(stage2_conf, '#c0392b', 'ANOMALY · Stage 2')
        + f"<div style='flex:1;min-width:200px;'>{conf_bar_chart(top3_raw)}</div>"
        + (f"<div style='width:100%;margin-top:10px;'>{donut_pie_svg(all_probs)}</div>" if all_probs else "")
        + "</div>"
    )
    charts_sensing = (
        f"<div style='padding:14px 16px;background:#050505;border:1px solid #1e1e1e;"
        f"border-radius:3px;margin-top:6px;'>"
        f"<div style='font-family:\"Roboto Mono\",monospace;font-size:.6rem;letter-spacing:.14em;"
        f"color:#FF8C00;text-transform:uppercase;margin-bottom:12px;'>Signal Intensities</div>"
        + risk_gauge(fused_risk,        "Fusion Risk (Stage 6)", risk_color(fused_risk))
        + risk_gauge(audio_score,       "Audio Score (Stage 4)", "#2471a3")
        + risk_gauge(min(motion, 1.0),  "Motion Intensity (Stage 5)", "#117a65")
        + "</div>"
    )

    # ── Table rows ────────────────────────────────────────────────────────────
    ae_rows = (row("Overall Status",    report.get("overall_status",""))
             + row("Dynamic Threshold", report.get("dynamic_threshold",""), mono=True)
             + row("AE Max Error",      report.get("ae_max_error",""), mono=True)
             + row("AE Avg Error",      report.get("ae_avg_error",""), mono=True)
             + row("Peak Timestamp",    report.get("peak_timestamp",""), mono=True)
             + row("Anomaly Window",    report.get("anomaly_window",""), mono=True))

    cl_rows = (row("TF Binary (Stage 1)",   report.get("stage1_tf_binary",""))
             + row("Anomaly Type (Stage 2)", report.get("stage2_anomaly_type","")))
    for i, (cls, prob) in enumerate(report.get("stage2_top3",[]), 1):
        cl_rows += row(f"  Top-{i}", f"{cls}  ({prob:.2%})", mono=True)

    det_rows = (row("Weapon Detection (Stage 3)", report.get("stage3_weapon",""))
              + row("Audio Status (Stage 4)",      report.get("stage4_audio",""))
              + row("Motion (Stage 5)",            report.get("stage5_motion","")))

    fus_rows = row("Fusion Risk (Stage 6)", report.get("stage6_fusion_risk",""))
    for i, line in enumerate(report.get("stage6_reasoning",[]), 1):
        fus_rows += row(f"  Step {i}", line)

    rl_rows = (row("RL3033 Final Risk (Stage 7)", report.get("stage7_rl_final",""), mono=True)
             + row("Interpretation",              report.get("stage7_interpretation","")))

    narr = report.get("llava_narrative","")
    narr_html = (
        f"<div style='margin-bottom:26px;'>"
        + section_hdr("AI Visual Narrative (LLaVA — Stage 8)")
        + f"<div style='color:#ccc;font-size:.85rem;line-height:1.78;padding:16px 18px;"
        f"background:#050505;border:1px solid #1e1e1e;border-radius:3px;margin-top:6px;'>{narr}</div>"
        + "</div>"
    ) if narr else ""

    ae_graph_html = ""
    if graph_path:
        data_uri = img_to_data_uri(graph_path)
        if data_uri:
            ae_graph_html = (
                f"<div style='margin-bottom:26px;'>"
                + section_hdr("AE Deviation Graph — Temporal Reconstruction Error")
                + f"<div style='padding:14px;background:#050505;border:1px solid #1e1e1e;"
                f"border-radius:3px;margin-top:6px;'>"
                f"<img src='{data_uri}' alt='AE Deviation Graph' "
                f"style='width:100%;border-radius:2px;display:block;'/>"
                f"<div style='font-family:\"Roboto Mono\",monospace;font-size:.58rem;color:#555;"
                f"margin-top:8px;text-align:center;'>"
                f"Red = reconstruction error &nbsp;&middot;&nbsp; Green dashed = dynamic threshold "
                f"&nbsp;&middot;&nbsp; Star = peak anomaly frame"
                f"</div></div></div>"
            )

    return (
        f"<div style='background:#0a0a0a;padding:28px 26px;border:1px solid #1e1e1e;border-radius:4px;'>"
        + badge + ae_graph_html
        + section("Autoencoder — Stage 0",            ae_rows)
        + section("Classification — Stages 1 & 2",    cl_rows, charts_cls)
        + section("Detection & Sensing — Stages 3-5", det_rows, charts_sensing)
        + section("Fusion Layer — Stage 6",            fus_rows)
        + section("RL3033 Temporal Reasoning — Stage 7", rl_rows)
        + narr_html
        + "</div>"
    )


def process_video(video):
    if video is None:
        return (no_result_html(), None, None, None)
    try:
        out         = run_pipeline(video)
        r           = out["risk_score"]
        report_html = build_report_html(out["report"], r, out["risk_label"],
                                         out["status"], graph_path=out.get("graph_path"))
        fi = Image.fromarray(out["top_frame"]) if out["top_frame"] is not None else None
        return (report_html, fi, json.dumps(out["json_data"], indent=2, default=str), out["pdf_path"])
    except Exception:
        tb = traceback.format_exc()
        return (f"<div style='color:#FF4500;font-family:\"Roboto Mono\",monospace;"
                f"font-size:.8rem;padding:20px;background:#050505;'>Analysis failed:\n\n{tb}</div>",
                None, None, None)


def handle_contact(name: str, email: str, phone: str, msg: str) -> str:
    """Shared contact form handler — used in both the base app and agent layer tabs."""
    if not name.strip() or not email.strip() or not msg.strip():
        return ("<div style='font-family:\"Roboto Mono\",monospace;font-size:.72rem;"
                "color:#FF4500;padding:10px 0;font-weight:600;'>"
                "Name, email, and message are required.</div>")
    if not re.match(r"^[^@]+@[^@]+\.[^@]+$", email.strip()):
        return ("<div style='font-family:\"Roboto Mono\",monospace;font-size:.72rem;"
                "color:#FF4500;padding:10px 0;font-weight:600;'>"
                "Please enter a valid email address.</div>")
    send_contact_email(name.strip(), email.strip(), phone.strip(), msg.strip())
    return ("<div style='font-family:\"Roboto Mono\",monospace;font-size:.72rem;"
            "color:#4CAF50;padding:10px 0;letter-spacing:.06em;font-weight:600;'>"
            "&#10003; Message received. We will respond at the provided email.</div>")


# ─────────────────────────────────────────────────────────────────────────────
# CSS — pitch black / white / orange palette
# ─────────────────────────────────────────────────────────────────────────────
APP_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Roboto+Mono:wght@300;400;500;600;700&display=swap');

:root {
  --bg:     #000000;
  --surface:#080808;
  --border: #1e1e1e;
  --white:  #FFFFFF;
  --dim:    #888888;
  --orange: #FF8C00;
  --red:    #FF4500;
  --mono:   'Roboto Mono', monospace;
  --serif:  Charter, 'Bitstream Charter', Georgia, serif;
}

*, *::before, *::after { box-sizing:border-box; margin:0; padding:0; }
html, body, .gradio-container, .gradio-container > * {
  background:var(--bg) !important; color:var(--white) !important;
  font-family:var(--serif) !important;
}
.gradio-container {
  zoom: 1.3 !important;
}
* { -webkit-font-smoothing:antialiased; }

::-webkit-scrollbar { width:4px; height:4px; }
::-webkit-scrollbar-track { background:var(--bg); }
::-webkit-scrollbar-thumb { background:#333; border-radius:2px; }

@keyframes arc-spin {
  from { transform:rotate(0deg); } to { transform:rotate(360deg); }
}

/* TABS */
.tab-nav {
  background:var(--bg) !important; border-bottom:1px solid var(--border) !important;
  padding:0 24px !important; display:flex !important; gap:0 !important;
  align-items:center !important; justify-content:flex-start !important;
}
.tab-nav button {
  font-family:var(--mono) !important; font-size:.72rem !important;
  letter-spacing:.13em !important; color:#999 !important;
  background:transparent !important; border:none !important;
  border-bottom:2px solid transparent !important; padding:14px 0 !important;
  min-width:160px !important; width:160px !important; text-align:center !important;
  text-transform:uppercase !important; transition:color .18s, border-color .18s !important;
  font-weight:600 !important; white-space:nowrap !important; flex:0 0 160px !important;
}
.tab-nav button:hover  { color:#ddd !important; }
.tab-nav button.selected {
  color:#FFFFFF !important; border-bottom-color:var(--orange) !important; font-weight:700 !important;
}

/* GRADIO INTERNALS */
.gradio-container .block, .gradio-container .panel,
.gradio-container .form, .gradio-container fieldset,
.gradio-container .wrap, .gradio-container .container,
.gradio-container > div { background:var(--bg) !important; border:none !important; }

/* INPUTS */
input, textarea, select {
  background:#0d0d0d !important; border:1px solid #2a2a2a !important;
  border-radius:3px !important; color:#f0f0f0 !important;
  font-family:var(--serif) !important; font-size:.88rem !important;
  padding:12px 14px !important; transition:border-color .18s !important; width:100% !important;
}
input:focus, textarea:focus {
  border-color:var(--orange) !important; outline:none !important;
  box-shadow:0 0 0 1px var(--orange) !important;
}
label, .label-wrap span {
  font-family:var(--mono) !important; font-size:.65rem !important;
  letter-spacing:.1em !important; text-transform:uppercase !important;
  color:#ccc !important; font-weight:600 !important;
}

/* BUTTONS */
.gr-button {
  background:var(--white) !important; color:var(--bg) !important;
  border-radius:3px !important; font-family:var(--mono) !important;
  font-size:.72rem !important; letter-spacing:.12em !important; font-weight:700 !important;
}
.gr-button:hover { background:var(--orange) !important; }
#ana-btn, #ana-btn button {
  width:auto !important; max-width:260px !important;
  min-height:44px !important; height:44px !important; padding:0 30px !important;
}

/* VIDEO UPLOAD */
#vid-up { min-height:150px !important; max-height:200px !important; }
#vid-up .upload-container, #vid-up .wrap {
  background:#060606 !important; border:1px dashed #2a2a2a !important;
  border-radius:3px !important; min-height:150px !important; max-height:200px !important;
}
#vid-up .wrap:hover { border-color:var(--orange) !important; }

/* OUTPUT AREAS */
#report-html { background:var(--bg) !important; }
#kf-img, #kf-img img { border-radius:3px !important; border:1px solid var(--border) !important; width:100% !important; }
#json-out, #json-out textarea {
  background:#060606 !important; color:#bbb !important;
  font-family:var(--mono) !important; font-size:.75rem !important;
  line-height:1.7 !important; border:1px solid var(--border) !important; border-radius:3px !important;
}
#pdf-out { border:1px solid var(--border) !important; border-radius:3px !important; }
#pdf-out .file-preview { background:#060606 !important; }

.rule { border:none; border-top:1px solid #111; margin:28px 0; }
.contact-card {
  background:#080808; border:1px solid #1e1e1e; border-radius:4px; padding:28px 28px 20px;
}
.contact-info-block {
  background:#060606; border:1px solid #222; border-radius:3px; padding:18px 20px; margin-top:16px;
}
footer { display:none !important; }
"""

HOME_HTML = """
<div style="max-width:900px;margin:0 auto;padding:0 28px 72px;">

  <!-- HERO -->
  <div style="padding:72px 0 56px;border-bottom:1px solid #111;">
    <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;letter-spacing:.18em;
                color:#FF8C00;text-transform:uppercase;margin-bottom:16px;font-weight:700;">
      Forensic AI Platform &nbsp;&middot;&nbsp; Scene Solver v3
    </div>
    <h1 style="font-family:Charter,Georgia,serif;font-size:3.6rem;font-weight:400;
               color:#fff;line-height:1.08;margin-bottom:22px;letter-spacing:-.01em;">
      Uncover the Fact.
    </h1>
    <p style="font-family:Charter,Georgia,serif;font-size:1rem;color:#666;
              max-width:560px;line-height:1.78;margin-bottom:40px;">
      CrimeSceneSolver is an end-to-end forensic intelligence platform.
      Upload CCTV footage and a nine-stage AI pipeline analyses it, then
      Sherl𝟬ck, the agentic escalation layer, briefs you by voice or text,
      dispatches evidence to all channels, and generates a full PDF dossier.
    </p>
    <button onclick="(function(){
        var b=document.querySelectorAll('.tab-nav button,[role=tab]');
        for(var i=0;i<b.length;i++){var t=(b[i].textContent||'').trim().toLowerCase();
          if(t.indexOf('analyse')!==-1){b[i].click();return;}}
        if(b.length>1)b[1].click();
      })()"
      style="background:#fff;color:#000;font-family:'Roboto Mono',monospace;font-size:.74rem;
             letter-spacing:.14em;text-transform:uppercase;padding:14px 36px;border-radius:3px;
             border:none;cursor:pointer;font-weight:700;transition:background .2s;"
      onmouseover="this.style.background='#FF8C00'" onmouseout="this.style.background='#fff'">
      Analyse Footage &rarr;
    </button>
  </div>

  <!-- NINE-STAGE PIPELINE -->
  <div style="padding:56px 0 48px;border-bottom:1px solid #111;">
    <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;letter-spacing:.16em;
                color:#FF8C00;text-transform:uppercase;margin-bottom:28px;font-weight:700;">
      Nine-Stage Forensic Pipeline
    </div>
    <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:10px;">
      <div style="padding:20px 18px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:'Roboto Mono',monospace;font-size:.55rem;color:#FF8C00;margin-bottom:6px;font-weight:700;">STAGE 0</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.9rem;color:#fff;margin-bottom:6px;">Autoencoder (AE)</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.76rem;color:#555;line-height:1.6;">Primary anomaly signal. Reconstruction error over a dynamic threshold triggers the pipeline.</div>
      </div>
      <div style="padding:20px 18px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:'Roboto Mono',monospace;font-size:.55rem;color:#FF8C00;margin-bottom:6px;font-weight:700;">STAGE 1</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.9rem;color:#fff;margin-bottom:6px;">TimeSformer Binary</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.76rem;color:#555;line-height:1.6;">Semantic gate: classifies the clip as Normal or Suspicious. Normal overrides the AE signal to SAFE.</div>
      </div>
      <div style="padding:20px 18px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:'Roboto Mono',monospace;font-size:.55rem;color:#FF8C00;margin-bottom:6px;font-weight:700;">STAGE 2</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.9rem;color:#fff;margin-bottom:6px;">TimeSformer Multi-class</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.76rem;color:#555;line-height:1.6;">Fine-grained anomaly type across 7 categories using AE-weighted temporal sampling.</div>
      </div>
      <div style="padding:20px 18px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:'Roboto Mono',monospace;font-size:.55rem;color:#FF8C00;margin-bottom:6px;font-weight:700;">STAGE 3</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.9rem;color:#fff;margin-bottom:6px;">YOLO Weapon Detection</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.76rem;color:#555;line-height:1.6;">Hard safety trigger. Melee and firearm detection within the anomaly window. Forces 0.90 risk floor.</div>
      </div>
      <div style="padding:20px 18px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:'Roboto Mono',monospace;font-size:.55rem;color:#FF8C00;margin-bottom:6px;font-weight:700;">STAGE 4</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.9rem;color:#fff;margin-bottom:6px;">Audio Feature Extraction</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.76rem;color:#555;line-height:1.6;">ZCR, MFCC variance, spectral flux, RMS, and centroid scored against global thresholds.</div>
      </div>
      <div style="padding:20px 18px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:'Roboto Mono',monospace;font-size:.55rem;color:#FF8C00;margin-bottom:6px;font-weight:700;">STAGE 5</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.9rem;color:#fff;margin-bottom:6px;">Tracking &amp; Motion</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.76rem;color:#555;line-height:1.6;">YOLO tracking with Farneback optical flow fallback. Detects FAST_MOVEMENT, CLUSTERING, STATIC.</div>
      </div>
      <div style="padding:20px 18px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:'Roboto Mono',monospace;font-size:.55rem;color:#FF8C00;margin-bottom:6px;font-weight:700;">STAGE 6</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.9rem;color:#fff;margin-bottom:6px;">Fusion Layer</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.76rem;color:#555;line-height:1.6;">Explainable rule-based reasoning combining all stage outputs into a single risk score with a full trace.</div>
      </div>
      <div style="padding:20px 18px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:'Roboto Mono',monospace;font-size:.55rem;color:#FF8C00;margin-bottom:6px;font-weight:700;">STAGE 7</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.9rem;color:#fff;margin-bottom:6px;">RL3033 Agent</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.76rem;color:#555;line-height:1.6;">PPO reinforcement learning agent. Final risk blends TFB, TFM, Fusion, and RL scores <span style="font-family:'Times New Roman',Times,serif;">(70/20/5/5%)</span>.</div>
      </div>
      <div style="padding:20px 18px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:'Roboto Mono',monospace;font-size:.55rem;color:#FF8C00;margin-bottom:6px;font-weight:700;">STAGE 8</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.9rem;color:#fff;margin-bottom:6px;">LLaVA Narrative</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.76rem;color:#555;line-height:1.6;">Four keyframes described in natural language and synthesised into a coherent scene narrative.</div>
      </div>
    </div>
  </div>

  <!-- SHERL0CK AGENT -->
  <div style="padding:56px 0 48px;border-bottom:1px solid #111;">
    <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;letter-spacing:.16em;
                color:#FF8C00;text-transform:uppercase;margin-bottom:28px;font-weight:700;">
      Sherl𝟬ck — Agentic Escalation Layer
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="padding:26px 22px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:Charter,Georgia,serif;font-size:.95rem;color:#fff;margin-bottom:8px;">Voice Call Interface</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.8rem;color:#555;line-height:1.7;">
          When a CRITICAL or HIGH risk is detected, Sherl𝟬ck initiates an incoming call.
          Accept it to speak directly with the agent, Whisper transcribes your speech,
          Cohere generates an analytical response, and XTTS v2 delivers it as a voice reply.
          Server-side VAD handles turn detection automatically.
        </div>
      </div>
      <div style="padding:26px 22px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:Charter,Georgia,serif;font-size:.95rem;color:#fff;margin-bottom:8px;">Text Chat Interface</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.8rem;color:#555;line-height:1.7;">
          A persistent text chat powered by Cohere command-a-05-2026 with the full pipeline
          JSON pre-loaded as context. Ask about any stage output, request reasoning traces,
          or say <em style="color:#FF8C00;font-style:normal;">"send report"</em> to dispatch evidence.
        </div>
      </div>
      <div style="padding:26px 22px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:Charter,Georgia,serif;font-size:.95rem;color:#fff;margin-bottom:8px;">Evidence Dispatch</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.8rem;color:#555;line-height:1.7;">
          One click dispatches the full forensic dossier simultaneously to Telegram,
          Discord, and Email, including the peak anomaly frame and the 11-page PDF report.
          CRITICAL alerts auto-dispatch without user action.
        </div>
      </div>
      <div style="padding:26px 22px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:Charter,Georgia,serif;font-size:.95rem;color:#fff;margin-bottom:8px;">11-Page PDF Dossier</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.8rem;color:#555;line-height:1.7;">
          Auto-generated PDF covering every stage: AE temporal graph, classification charts,
          weapon detections with bounding boxes, audio feature breakdown, motion tracking,
          fusion reasoning trace, RL3033 observation space, anomalous frames, and LLaVA narrative.
        </div>
      </div>
    </div>
  </div>

  <!-- WORKFLOW -->
  <div style="padding:56px 0 48px;border-bottom:1px solid #111;">
    <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;letter-spacing:.16em;
                color:#FF8C00;text-transform:uppercase;margin-bottom:28px;font-weight:700;">End-to-End Workflow</div>
    <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:24px;">
      <div>
        <div style="font-family:'Roboto Mono',monospace;font-size:.6rem;color:#333;margin-bottom:8px;font-weight:700;">01</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.92rem;color:#fff;margin-bottom:6px;">Upload Footage</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.78rem;color:#555;line-height:1.65;">Drop CCTV footage on the Analyse tab and press Analyse Footage.</div>
      </div>
      <div>
        <div style="font-family:'Roboto Mono',monospace;font-size:.6rem;color:#333;margin-bottom:8px;font-weight:700;">02</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.92rem;color:#fff;margin-bottom:6px;">Pipeline Runs</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.78rem;color:#555;line-height:1.65;">All nine stages execute sequentially. Report, frame, JSON, and PDF are produced.</div>
      </div>
      <div>
        <div style="font-family:'Roboto Mono',monospace;font-size:.6rem;color:#333;margin-bottom:8px;font-weight:700;">03</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.92rem;color:#fff;margin-bottom:6px;">Sherl0ck Calls</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.78rem;color:#555;line-height:1.65;">On HIGH or CRITICAL risk, an incoming call overlay appears. Accept to begin the briefing.</div>
      </div>
      <div>
        <div style="font-family:'Roboto Mono',monospace;font-size:.6rem;color:#333;margin-bottom:8px;font-weight:700;">04</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.92rem;color:#fff;margin-bottom:6px;">Dispatch &amp; Close</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.78rem;color:#555;line-height:1.65;">Dispatch evidence to all channels, download the PDF dossier, and end the session.</div>
      </div>
    </div>
  </div>

  <!-- PRICING -->
  <div style="padding:56px 0 48px;border-bottom:1px solid #111;">
    <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;letter-spacing:.16em;
                color:#FF8C00;text-transform:uppercase;margin-bottom:28px;font-weight:700;">Pricing</div>
    <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:12px;">
      <div style="padding:26px 22px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;color:#555;margin-bottom:12px;font-weight:700;">BASIC</div>
        <div style="font-size:2rem;color:#fff;margin-bottom:14px;font-family:'Times New Roman',Times,serif;">Free</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.8rem;color:#555;line-height:2.1;">
          9-stage pipeline<br>5 analyses / month<br>PDF dossier<br>JSON output
        </div>
      </div>
      <div style="padding:26px 22px;background:#080808;border:1px solid #FF8C00;border-radius:3px;">
        <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;color:#FF8C00;margin-bottom:12px;font-weight:700;">PRO</div>
        <div style="font-size:2rem;color:#fff;margin-bottom:14px;font-family:'Times New Roman',Times,serif;">&#8377;49/mo</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.8rem;color:#888;line-height:2.1;">
          Unlimited analyses<br>Sherl0ck voice &amp; text agent<br>Auto-dispatch (Telegram, Discord, Email)<br>LLaVA narratives<br>Priority support
        </div>
      </div>
      <div style="padding:26px 22px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;color:#555;margin-bottom:12px;font-weight:700;">ENTERPRISE</div>
        <div style="font-size:2rem;color:#fff;margin-bottom:14px;font-family:'Times New Roman',Times,serif;">Custom</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.8rem;color:#555;line-height:2.1;">
          Custom model fine-tuning<br>On-premise deployment<br>Dedicated SLA<br>API access
        </div>
      </div>
    </div>
  </div>

  <!-- TESTIMONIAL -->
  <div style="padding:56px 0 24px;">
    <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;letter-spacing:.16em;
                color:#FF8C00;text-transform:uppercase;margin-bottom:26px;font-weight:700;">From the Field</div>
    <blockquote style="font-family:Charter,Georgia,serif;font-size:1.08rem;color:#aaa;line-height:1.76;
                       border-left:2px solid #FF8C00;padding-left:22px;font-style:italic;margin:0 0 14px;">
      &ldquo;We closed a complex six-month investigation in under two weeks after deploying
      SceneSolver. The nine-stage pipeline, Sherl0ck voice briefings, and automated
      evidence dispatch changed how our forensics team operates entirely.&rdquo;
    </blockquote>
    <div style="font-family:'Roboto Mono',monospace;font-size:.66rem;color:#555;font-weight:500;">
      &mdash; Detective Lara Monroe, Forensics Division
    </div>
  </div>

</div>
"""


# ─────────────────────────────────────────────────────────────────────────────
# BUILD APP
# ─────────────────────────────────────────────────────────────────────────────
with gr.Blocks(css=APP_CSS, title="CrimeSceneSolver") as demo:

    gr.HTML(NAV_HTML)

    with gr.Tabs():

        with gr.TabItem("\u00a0\u00a0\u00a0 Home \u00a0\u00a0\u00a0"):
            gr.HTML(HOME_HTML)
            gr.HTML(FOOTER_HTML)

        with gr.TabItem("\u00a0\u00a0 Analyse \u00a0\u00a0"):
            gr.HTML("""
            <div style="max-width:780px;margin:0 auto;padding:48px 28px 0;">
              <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;letter-spacing:.16em;
                          color:#FF8C00;text-transform:uppercase;margin-bottom:12px;font-weight:600;">
                Forensic Analysis</div>
              <h1 style="font-family:Charter,Georgia,serif;font-size:3rem;font-weight:400;
                         color:#fff;margin-bottom:8px;">Case File Analysis</h1>
              <p style="font-family:Charter,Georgia,serif;font-size:.88rem;color:#555;margin-bottom:28px;">
                Upload CCTV footage to run the full nine-stage forensic pipeline.
              </p>
              <hr class="rule">
            </div>
            """)

            with gr.Column():
                gr.HTML('<div style="max-width:780px;margin:0 auto;padding:0 28px;">')
                gr.HTML(sec_label("Upload Footage"))
                vid_in = gr.Video(label="", elem_id="vid-up", sources=["upload"],
                                  show_label=False, height=175)
                gr.HTML('<div style="height:14px;"></div>')
                gr.HTML('<div style="display:flex;justify-content:flex-start;">')
                ana_btn = gr.Button("Analyse Footage", elem_id="ana-btn", elem_classes="gr-button")
                gr.HTML('</div>')
                spinner_out = gr.HTML("")
                gr.HTML('<div style="height:28px;"></div>')
                gr.HTML(sec_label("Risk Assessment &amp; Report"))
                report_out = gr.HTML(
                    value=("<div style='text-align:center;padding:60px 0;"
                           "font-family:Charter,Georgia,serif;font-size:.9rem;"
                           "color:#2a2a2a;'>Upload footage and press Analyse.</div>"),
                    elem_id="report-html")
                gr.HTML('<div style="height:28px;"></div>')
                gr.HTML(sec_label("Peak Anomaly Frame"))
                frame_out = gr.Image(label="", elem_id="kf-img", show_label=False)
                gr.HTML('<div style="height:28px;"></div>')
                gr.HTML(sec_label("Download Forensic PDF Report"))
                pdf_out = gr.File(label="", elem_id="pdf-out", show_label=False,
                                  file_types=[".pdf"], interactive=False)
                gr.HTML("""
                <div style="font-family:'Roboto Mono',monospace;font-size:.6rem;color:#444;
                            line-height:1.7;padding:10px 0;border-top:1px solid #0d0d0d;margin-top:6px;">
                  11-page forensic dossier: AE temporal graph &middot; classification charts
                  &middot; weapon detections &middot; audio features &middot; motion tracking
                  &middot; fusion reasoning &middot; RL3033 output &middot; anomalous frames
                  &middot; LLaVA narrative
                </div>""")
                gr.HTML('<div style="height:28px;"></div>')
                gr.HTML(sec_label("Structured JSON Output"))
                json_out = gr.Textbox(label="", elem_id="json-out", show_label=False,
                                      lines=26, placeholder="JSON output will appear here…")
                gr.HTML('</div>')

            def show_spinner(): return gr.update(value=spinner_html())
            def hide_spinner(): return gr.update(value="")

            ana_btn.click(fn=show_spinner, inputs=None, outputs=spinner_out, queue=True
            ).then(fn=process_video, inputs=[vid_in],
                   outputs=[report_out, frame_out, json_out, pdf_out]
            ).then(fn=hide_spinner, inputs=None, outputs=spinner_out, queue=True)

            gr.HTML(FOOTER_HTML)

        with gr.TabItem("\u00a0\u00a0 Contact \u00a0\u00a0"):
            gr.HTML("""
            <div style="max-width:600px;margin:0 auto;padding:52px 28px 0;">
              <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;letter-spacing:.16em;
                          color:#FF8C00;text-transform:uppercase;margin-bottom:12px;font-weight:600;">
                Get In Touch</div>
              <h2 style="font-family:Charter,Georgia,serif;font-size:3.4rem;font-weight:400;
                  color:#fff;line-height:1.1;margin-bottom:20px;">Contact Us</h2>
              <p style="font-family:Charter,Georgia,serif;font-size:.88rem;color:#555;margin-bottom:28px;">
                Reach out with questions, partnership inquiries, or case-specific requests.
              </p>
            </div>
            """)

            with gr.Column():
                gr.HTML('<div style="max-width:600px;margin:0 auto;padding:0 28px 48px;">')
                gr.HTML('<div class="contact-card">')
                ct_n = gr.Textbox(label="Name",         placeholder="Your full name")
                gr.HTML('<div style="height:12px;"></div>')
                ct_e = gr.Textbox(label="Email",        placeholder="your@email.com")
                gr.HTML('<div style="height:12px;"></div>')
                ct_p = gr.Textbox(label="Phone Number", placeholder="+91 XXXXXXXXXX")
                gr.HTML('<div style="height:12px;"></div>')
                ct_m = gr.Textbox(label="Message",      placeholder="Your message…", lines=5)
                gr.HTML('<div style="height:18px;"></div>')
                gr.HTML('<div style="display:flex;justify-content:flex-start;">')
                ct_b = gr.Button("Send Message", elem_classes="gr-button", elem_id="send-btn")
                gr.HTML('</div>')
                ct_r = gr.HTML("")
                gr.HTML("""
                <div class="contact-info-block">
                  <div style="font-family:'Roboto Mono',monospace;font-size:.6rem;letter-spacing:.14em;
                              text-transform:uppercase;color:#FF8C00;margin-bottom:14px;font-weight:600;">
                    Direct Contact</div>
                  <div style="display:flex;flex-direction:column;gap:12px;">
                    <div style="display:flex;align-items:center;gap:12px;">
                      <a href="mailto:anumulasamarth008@gmail.com"
                         style="font-family:Charter,Georgia,serif;font-size:.86rem;color:#bbb;
                                text-decoration:none;"
                         onmouseover="this.style.color='#FF8C00'" onmouseout="this.style.color='#bbb'">
                        anumulasamarth008@gmail.com
                      </a>
                    </div>
                    <div style="display:flex;align-items:center;gap:12px;">
                      <span style="font-family:Charter,Georgia,serif;font-size:.86rem;color:#bbb;">
                        +91 8374982517
                      </span>
                    </div>
                  </div>
                </div>
                """)
                gr.HTML('</div></div>')

            gr.HTML("""
            <style>
            #send-btn, #send-btn button {
              width:auto !important; max-width:200px !important;
              min-height:44px !important; height:44px !important; padding:0 28px !important;
              font-size:.72rem !important; letter-spacing:.12em !important;
            }
            </style>""")

            ct_b.click(handle_contact, [ct_n, ct_e, ct_p, ct_m], ct_r)
            gr.HTML(FOOTER_HTML)
#demo.launch(share=True,debug=True) #--> for faster inference uncomment and use the link this cell provides.

/tmp/ipykernel_246/544474527.py:875: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=APP_CSS, title="CrimeSceneSolver") as demo:


In [20]:
# ============================================================
# AGENTIC ESCALATION LAYER
# DEPENDENCIES / ENVIRONMENT
# Run this cell ONCE before the Agentic Escalation Layer cell.
# ============================================================

import subprocess
import sys
import os

def pip_install(package):
    print(f"[Setup] Installing {package} ...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", package],
        check=False,
    )
    if result.returncode != 0:
        print(f"[Setup] WARNING: {package} installation returned {result.returncode}")

def apt_install(package):
    print(f"[Setup] Installing apt package: {package} ...")
    result = subprocess.run(
        ["apt-get", "install", "-y", "-qq", package],
        check=False,
    )
    if result.returncode != 0:
        print(f"[Setup] WARNING: {package} installation returned {result.returncode}")


# ------------------------------------------------------------
# Required for the current voice architecture
# ------------------------------------------------------------

pip_install("cohere>=5.0.0")
pip_install("liquid-audio")
pip_install("soundfile")

# Browser-recorded WebM/Opus -> WAV conversion
apt_install("ffmpeg")


# ------------------------------------------------------------
# Optional / already used elsewhere in SceneSolver
# Install only if your earlier cells do not already provide them.
# ------------------------------------------------------------

# Uncomment only if your notebook actually needs these independently.
# pip_install("torch")
# pip_install("torchaudio")


# ------------------------------------------------------------
# Cohere API key
# ------------------------------------------------------------
# Better: set this before running the Agentic cell.
# Do NOT put the real key directly into the Agentic Escalation Layer.

os.environ["COHERE_API_KEY"] = "e6Eo0JkKqPzpH7kUeAFpU0km6Dk73a8yJ6zXXtRN"

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("\n============================================================")
print("AGENTIC ESCALATION LAYER DEPENDENCIES READY")
print("============================================================")

print("COHERE_API_KEY configured:",
      bool(os.environ.get("COHERE_API_KEY", "").strip()))

try:
    import cohere
    print("Cohere          : OK")
except Exception as e:
    print("Cohere          : FAILED", e)

try:
    import soundfile
    print("SoundFile       : OK")
except Exception as e:
    print("SoundFile       : FAILED", e)

try:
    import liquid_audio
    print("liquid-audio    : OK")
except Exception as e:
    print("liquid-audio    : FAILED", e)

print("FFmpeg          :", end=" ")

ffmpeg_check = subprocess.run(
    ["ffmpeg", "-version"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=False,
)

print("OK" if ffmpeg_check.returncode == 0 else "FAILED")

[Setup] Installing cohere>=5.0.0 ...
[Setup] Installing liquid-audio ...
[Setup] Installing soundfile ...
[Setup] Installing apt package: ffmpeg ...

AGENTIC ESCALATION LAYER DEPENDENCIES READY
COHERE_API_KEY configured: True
Cohere          : OK
SoundFile       : OK
liquid-audio    : OK
FFmpeg          : OK


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# AGENTIC ESCALATION LAYER  :  Sherl0ck Agent
# Run AFTER Cell 2 (pipeline), Cell 5 (web app), and Cell 6 (dependencies).
# Reuses shared constants/helpers from Cell 5: NAV_HTML, FOOTER_HTML, APP_CSS,
# sec_label, risk_color, img_to_data_uri, spinner_html, no_result_html,
# run_pipeline, build_report_html, process_video, handle_contact, HOME_HTML.
#
# VOICE SUBSYSTEM (this revision):
#   Continuous hands-free voice after Accept Call.
#   Browser MediaStream + Web-Audio VAD + MediaRecorder
#   → same-origin POST /ssvoice/turn (binary WebM bytes)
#   → FFmpeg → mono 24 kHz WAV
#   → LFM2.5-Audio-1.5B ASR
#   → Cohere / Sherl0ck reasoning
#   → LFM2.5-Audio-1.5B US male TTS
#   → browser playback + live waveforms + barge-in
#   → automatic next turn until End Call.
#
# PRIMARY BUG FIXED:
#   Previous Gradio textbox bridge delivered size=0 to Python while the browser
#   held a real nonzero Blob. Transport is now raw HTTP binary on the same
#   Gradio FastAPI app — BROWSER BYTES == SERVER BYTES.
# ══════════════════════════════════════════════════════════════════════════════

import os, re, json, smtplib, ssl, base64, traceback, threading, subprocess, sys, io, wave
import struct, math, time, uuid, tempfile, shutil
from pathlib import Path as _Path
from datetime import datetime, timezone
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from typing import Optional, Dict, Any, List, Tuple

import requests
import numpy as np
import gradio as gr

try:
    import liquid_audio  # noqa: F401 — ensure dependency cell installed it
except Exception:
    pass
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "liquid-audio"], check=False)
except Exception:
    pass

# ── Suppress non-critical log noise ───────────────────────────────────────────
import logging as _logging, warnings as _warnings
_warnings.filterwarnings("ignore", category=DeprecationWarning)
_warnings.filterwarnings("ignore", category=UserWarning, module="gradio")
_warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
for _noisy in ("gradio.routes", "gradio.queueing", "starlette", "uvicorn.error",
               "httpx", "httpcore", "urllib3", "transformers.modeling_utils",
               "transformers.generation.configuration_utils"):
    _logging.getLogger(_noisy).setLevel(_logging.ERROR)
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

# ── Config ────────────────────────────────────────────────────────────────────
TELEGRAM_BOT_TOKEN  = os.environ.get("TELEGRAM_BOT_TOKEN", "")
TELEGRAM_CHAT_ID    = os.environ.get("TELEGRAM_CHAT_ID", "")
DISCORD_WEBHOOK_URL = os.environ.get("DISCORD_WEBHOOK_URL", "")
COHERE_API_KEY      = os.environ.get("COHERE_API_KEY", "")
ALERT_EMAIL_FROM    = os.environ.get("ALERT_EMAIL_FROM", "")
ALERT_EMAIL_PASS    = os.environ.get("ALERT_EMAIL_PASS", "")
ALERT_EMAIL_TO      = os.environ.get("ALERT_EMAIL_TO", "")
ALERT_SMTP_SERVER   = "smtp.gmail.com"
ALERT_SMTP_PORT     = 465
CALLER_TUNE_PATH    = os.environ.get("CALLER_TUNE_PATH", AUDIOPATHS.get("CALLER_TUNE", "") if isinstance(AUDIOPATHS, dict) else "")
THRESH_CRITICAL     = 0.90
THRESH_HIGH         = 0.70
COHERE_MODEL        = "command-a-plus-05-2026"
def _cohere_extract_text(resp):
    """Extract text from Cohere response, handling thinking blocks in command-a-plus."""
    try:
        for item in resp.message.content:
            if hasattr(item, 'type') and item.type == "text":
                return item.text.strip()
        # Fallback: try content[0].text (older models)
        return resp.message.content[0].text.strip()
    except Exception:
        return ""


# ── Optional deps ─────────────────────────────────────────────────────────────
try:
    import cohere as _cohere
    COHERE_AVAILABLE = True
    print("[AgentLayer] Cohere OK")
except ImportError:
    COHERE_AVAILABLE = False
    print("[AgentLayer] Cohere unavailable")

try:
    import torch
except ImportError:
    torch = None
    print("[AgentLayer] torch unavailable")

try:
    import torchaudio
except ImportError:
    torchaudio = None
    print("[AgentLayer] torchaudio unavailable")

HF_LFM_AUDIO_REPO   = "LiquidAI/LFM2.5-Audio-1.5B"
LFM_AUDIO_AVAILABLE = False
_lfm_processor = None
_lfm_model     = None
_lfm_lock      = threading.Lock()

try:
    from liquid_audio import LFM2AudioModel, LFM2AudioProcessor, ChatState, LFMModality
    LFM_AUDIO_AVAILABLE = True
    print("[AgentLayer] liquid-audio import OK")
except ImportError as e:
    LFM_AUDIO_AVAILABLE = False
    print(f"[AgentLayer] liquid-audio unavailable: {e}")

# ══════════════════════════════════════════════════════════════════════════════
# SVG ICONS
# ══════════════════════════════════════════════════════════════════════════════
def _icon(body: str, size: int = 18, col: str = "currentColor") -> str:
    return (f'<svg width="{size}" height="{size}" viewBox="0 0 24 24" fill="none" '
            f'stroke="{col}" stroke-width="1.8" stroke-linecap="round" '
            f'stroke-linejoin="round" style="display:inline-block;vertical-align:'
            f'middle;flex-shrink:0;">{body}</svg>')

ICON_SHIELD   = _icon('<path d="M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10z"/>', 18)
ICON_DISPATCH = _icon('<polyline points="22 2 11 13"/><polygon points="22 2 15 22 11 13 2 9 22 2"/>', 16)

# ══════════════════════════════════════════════════════════════════════════════
# PIPELINE DATA HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def _has_pipeline_data(jd: dict) -> bool:
    return bool(jd) and isinstance(jd, dict) and len(jd) > 0


def _build_plain_context(jd: dict, risk_pct: int) -> str:
    lines = ["=== FORENSIC PIPELINE SUMMARY ==="]
    lines.append(f"Overall status   : {jd.get('status', 'UNKNOWN')}")
    lines.append(f"Risk level label : {jd.get('risk_level', 'UNKNOWN')}")
    lines.append(f"Final risk score : {risk_pct}%")
    s1 = jd.get("stage1") or {}
    if s1:
        lines.append(f"Stage 1 TF-Binary: {s1.get('classification','?')} @ "
                     f"{round(float(s1.get('confidence',0))*100)}% confidence")
    s2   = jd.get("stage2") or {}
    cls2 = s2.get("class", "")
    if s2 and cls2 and cls2 != "Skipped":
        lines.append(f"Stage 2 Anomaly  : {cls2} @ {round(float(s2.get('confidence',0))*100)}% confidence")
    else:
        lines.append("Stage 2 Anomaly  : Skipped / Normal scene")
    s3 = jd.get("stage3") or {}
    if s3:
        lines.append(f"Stage 3 Weapon   : detected={'YES' if s3.get('weapon_flag') else 'NO'}, "
                     f"class={s3.get('dominant_class','?')}, count={s3.get('weapon_count',0)}")
    s4 = jd.get("stage4") or {}
    if s4:
        lines.append(f"Stage 4 Audio    : {s4.get('status','?')}, "
                     f"score={round(float(s4.get('audio_score',0))*100)}%")
    s5 = jd.get("stage5") or {}
    if s5:
        bhv = ", ".join(s5.get("behaviors") or ["STATIC"])
        lines.append(f"Stage 5 Motion   : intensity={round(float(s5.get('motion_intensity',0))*100)}%, behaviors=[{bhv}]")
    s6 = jd.get("stage6") or {}
    if s6:
        lines.append(f"Stage 6 Fusion   : fused_risk={round(float(s6.get('fused_risk',0))*100)}%, level={s6.get('risk_level','?')}")
        if s6.get("reasoning"):
            lines.append(f"  Fusion reasoning: {s6['reasoning']}")
    s7 = jd.get("stage7") or {}
    if s7:
        lines.append(f"Stage 7 RL3033   : final_risk={round(float(s7.get('final_risk',0))*100)}%, status={s7.get('status','?')}")
        if s7.get("interpretation"):
            lines.append(f"  Interpretation: {s7['interpretation']}")
    s0 = jd.get("stage0") or {}
    if s0:
        lines.append(f"Anomaly window   : {s0.get('anomaly_start','?')} -> {s0.get('anomaly_end','?')}, peak={s0.get('peak_timestamp','?')}")
    narr = jd.get("llava_narrative", "")
    if narr:
        lines.append(f"LLaVA narrative  : {str(narr)[:600]}")
    lines.append("=== END SUMMARY ===")
    return "\n".join(lines)


def _build_priming_messages(payload: dict, level: str) -> list:
    risk_pct = round(float(payload.get("risk", 0.0)) * 100)
    jd       = payload.get("json_data") or {}
    if not _has_pipeline_data(jd):
        return [
            {"role": "user",      "content": "Confirm your data status."},
            {"role": "assistant", "content": "No pipeline data is currently loaded. I cannot answer analysis questions yet."},
        ]
    plain = _build_plain_context(jd, risk_pct)
    return [
        {"role": "user", "content": (
            f"Loading the forensic pipeline context for this session. "
            f"Alert level: {level}. Risk score: {risk_pct}%. "
            f"Here is the full summary — confirm you have received it:\n\n{plain}"
        )},
        {"role": "assistant", "content": (
            f"Confirmed. I have received and processed the forensic pipeline context. "
            f"This session is locked to alert level {level} with a final risk score of {risk_pct}%. "
            f"I am ready to answer your questions about the findings."
        )},
    ]


def _build_text_system_prompt(payload: dict, level: str) -> str:
    risk_pct = round(float(payload.get("risk", 0.0)) * 100)
    jd       = payload.get("json_data") or {}
    has_data = _has_pipeline_data(jd)
    try:
        full_json = json.dumps(jd, indent=2, default=str)
    except Exception:
        full_json = str(jd)
    plain_ctx = _build_plain_context(jd, risk_pct) if has_data else "(no pipeline data)"
    no_data_block = ""
    if not has_data:
        no_data_block = (
            "\n\nDATA STATUS: No pipeline data is loaded yet.\n"
            "Tell the analyst: 'No forensic analysis has been loaded yet. "
            "Please run the pipeline on the Analyse tab first.'\nDo NOT fabricate findings.\n"
        )
    return (
        f"You are Sherl0ck, a senior forensic surveillance analyst AI integrated into the "
        f"SceneSolver platform. Communicate with precision, depth, and professional clarity.\n\n"
        f"SESSION CONSTANTS — NEVER CHANGE:\n"
        f"  Alert Level : {level}\n"
        f"  Risk Score  : {risk_pct}%  (ALWAYS use {risk_pct}%, never any other value)\n"
        f"{no_data_block}\n"
        f"COMMUNICATION RULES:\n"
        f"1. You are Sherl0ck. You may confirm this identity if asked.\n"
        f"2. Respond with full analytical depth. Be thorough when warranted, concise otherwise.\n"
        f"3. No markdown headers. No leading dashes or bullets. Use numbered lists for sequences, "
        f"   plain prose for explanations. Bold key terms with **term** only when essential.\n"
        f"4. All risk scores as whole-number percentages (e.g. {risk_pct}%).\n"
        f"5. Risk score and alert level are FROZEN. Never alter them.\n"
        f"6. If the analyst says 'send report', 'dispatch', 'forward evidence', etc. — confirm "
        f"   report dispatched to all channels with delivery status.\n"
        f"7. Answer ONLY from the pipeline data below. Do not fabricate.\n"
        f"8. The investigative-aid disclaimer was stated at session open. Do not repeat it.\n\n"
        f"PRE-PARSED CONTEXT (primary reference):\n{plain_ctx}\n\n"
        f"FULL PIPELINE JSON:\n{full_json}\n"
    )


def _build_voice_system_prompt(payload: dict, level: str) -> str:
    risk_pct  = round(float(payload.get("risk", 0.0)) * 100)
    jd        = payload.get("json_data") or {}
    plain_ctx = _build_plain_context(jd, risk_pct) if _has_pipeline_data(jd) else "(no data)"
    try:
        full_json = json.dumps(jd, indent=2, default=str)
    except Exception:
        full_json = str(jd)
    return (
        f"You are Sherl0ck, a forensic surveillance AI agent speaking aloud to an analyst.\n\n"
        f"PERSONA: male, mature, calm, confident, warm, steady, reassuring, precise, experienced. "
        f"If the analyst sounds frightened, acknowledge briefly, reduce fear, establish control, "
        f"explain what can be determined. Do not amplify fear, become theatrical, or sound patronizing.\n\n"
        f"RULES:\n"
        f"1. You are Sherl0ck. You may confirm this if asked.\n"
        f"2. Plain words only. No punctuation except percent signs. No markdown, no bullets.\n"
        f"3. Short sentences. 2-4 sentences max per response.\n"
        f"4. All scores as whole-number percentages e.g. {risk_pct}%.\n"
        f"5. Risk score FIXED at {risk_pct}%. Alert level FIXED at {level}.\n"
        f"6. No disclaimer repetition.\n"
        f"7. If user says send report — confirm dispatch to all channels.\n"
        f"8. Answer ONLY from pipeline data.\n\n"
        f"CONTEXT SUMMARY:\n{plain_ctx}\n\nFULL JSON:\n{full_json}\n"
    )

# ══════════════════════════════════════════════════════════════════════════════
# CONVERSATION STATE
# ══════════════════════════════════════════════════════════════════════════════
_text_conversation_history:  list = []
_text_active_payload:        dict = {}
_voice_conversation_history: list = []
_voice_active_payload:       dict = {}

# ══════════════════════════════════════════════════════════════════════════════
# TEXT AGENT
# ══════════════════════════════════════════════════════════════════════════════
def text_agent_init(payload: dict, level: str) -> str:
    global _text_conversation_history, _text_active_payload
    _text_conversation_history = []
    _text_active_payload       = payload
    risk_pct = round(float(payload.get("risk", 0.0)) * 100)
    jd       = payload.get("json_data") or {}
    has_data = _has_pipeline_data(jd)
    print(f"[AgentLayer][Text] init: has_data={has_data} jd_keys={list(jd.keys())[:8]}")
    if not has_data:
        reply = (
            "No forensic analysis has been loaded yet. Please run the pipeline on the "
            "**Analyse** tab first — upload your footage and press Analyse. "
            "Once complete, click **Start Chat** to begin your briefing."
        )
        _text_conversation_history.append({"role": "assistant", "content": reply})
        return reply
    fallback = (
        f"**Alert Level: {level} | Risk Score: {risk_pct}%**\n\n"
        "Forensic pipeline analysis complete. I am ready to answer your questions.\n\n"
        "_This is an investigative aid — not a definitive conclusion. "
        "Human expert review is required before any operational action._"
    )
    if not COHERE_AVAILABLE:
        _text_conversation_history.append({"role": "assistant", "content": fallback})
        return fallback
    try:
        co   = _cohere.ClientV2(COHERE_API_KEY)
        msgs = [{"role": "system", "content": _build_text_system_prompt(payload, level)}]
        msgs.extend(_build_priming_messages(payload, level))
        msgs.append({"role": "user", "content": (
            f"The analysis is complete. Alert level is {level}, risk score is {risk_pct}%. "
            f"Provide a structured opening briefing: headline finding, key contributing factors "
            f"with actual values, high-confidence detections above 70%, temporal context, and "
            f"the LLaVA narrative in your own words. Plain prose, no markdown headers. "
            f"End with the standard investigative-aid disclaimer."
        )})
        resp  = co.chat(model=COHERE_MODEL, messages=msgs)
        reply = _cohere_extract_text(resp)
    except Exception as e:
        print(f"[AgentLayer][Text] Cohere init FAIL: {e}")
        reply = fallback
    _text_conversation_history.append({"role": "assistant", "content": reply})
    return reply


def _build_dispatch_reply(payload: dict, level: str) -> str:
    results  = dispatch_all_channels(payload, level)
    ts       = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    risk_pct = round(float(payload.get("risk", 0.0)) * 100)
    lines = [
        f"**Forensic dossier dispatched.** Timestamp: {ts}",
        f"Alert Level: **{level}** | Risk Score: **{risk_pct}%**", "",
    ]
    all_ok = True
    for ch, ok in results.items():
        lines.append(f"{'OK' if ok else 'FAIL'} **{ch.capitalize()}**: {'Delivered' if ok else 'Failed'}")
        if not ok: all_ok = False
    lines += ["", "**Dossier contents transmitted:**", str(payload.get("summary", ""))]
    if payload.get("image"): lines.append("Peak anomaly frame: attached (JPEG)")
    if payload.get("pdf"):   lines.append("Forensic PDF report: attached")
    lines.append("")
    lines.append("All channels confirmed delivery." if all_ok else "One or more channels failed — verify manually.")
    return "\n".join(lines)

_DISPATCH_HOOKS = (
    "send report", "send pdf", "share evidence", "dispatch report",
    "forward report", "share report", "forward evidence", "send dossier",
    "transmit report", "transmit dossier", "send findings", "escalate", "send alert",
)


def text_agent_respond(user_text: str) -> str:
    global _text_conversation_history, _text_active_payload
    if not user_text or not user_text.strip():
        return "Please enter a message."
    _text_conversation_history.append({"role": "user", "content": user_text.strip()})
    if any(h in user_text.lower() for h in _DISPATCH_HOOKS):
        level = determine_alert_level(float(_text_active_payload.get("risk", 0.0)))
        reply = _build_dispatch_reply(_text_active_payload, level)
        _text_conversation_history.append({"role": "assistant", "content": reply})
        return reply
    if not COHERE_AVAILABLE:
        reply = f"AI offline. You said: {user_text.strip()}"
        _text_conversation_history.append({"role": "assistant", "content": reply})
        return reply
    try:
        co    = _cohere.ClientV2(COHERE_API_KEY)
        level = determine_alert_level(float(_text_active_payload.get("risk", 0.0)))
        msgs  = [{"role": "system", "content": _build_text_system_prompt(_text_active_payload, level)}]
        msgs.extend(_build_priming_messages(_text_active_payload, level))
        real_history = _text_conversation_history[:-1]
        for m in real_history[-14:]:
            msgs.append({"role": m["role"], "content": m["content"]})
        msgs.append({"role": "user", "content": user_text.strip()})
        resp  = co.chat(model=COHERE_MODEL, messages=msgs)
        reply = _cohere_extract_text(resp)
    except Exception as e:
        print(f"[AgentLayer][Text] Cohere respond FAIL: {e}")
        reply = "I encountered a processing error. Please repeat your question."
    _text_conversation_history.append({"role": "assistant", "content": reply})
    return reply


def handle_text_input(user_text: str):
    if not user_text or not user_text.strip():
        return gr.update(), ""
    text_agent_respond(user_text)
    return _render_chat_html(_text_conversation_history), ""

# ══════════════════════════════════════════════════════════════════════════════
# VOICE AGENT (Cohere reasoning; LFM is speech bridge)
# ══════════════════════════════════════════════════════════════════════════════
def _sanitise_for_tts(text: str) -> str:
    text = re.sub(r'\bSherl[0o]ck\b', 'Sherlock', text, flags=re.IGNORECASE)
    text = re.sub(r'[^\w\s%]', ' ', text)
    return re.sub(r' {2,}', ' ', text).strip()


def voice_agent_init(payload: dict, level: str) -> str:
    global _voice_conversation_history, _voice_active_payload
    _voice_conversation_history = []
    _voice_active_payload       = payload
    risk_pct = round(float(payload.get("risk", 0.0)) * 100)
    suffix   = (
        "  This is an investigative AI assessment not a definitive conclusion"
        "  Speak your question and pause briefly  I will respond automatically"
        "  Say send report to dispatch evidence to all channels"
    )
    fallback = f"Alert level {level}  risk score {risk_pct}%" + suffix
    if not COHERE_AVAILABLE:
        _voice_conversation_history.append({"role": "CHATBOT", "content": fallback})
        return fallback
    try:
        co   = _cohere.ClientV2(COHERE_API_KEY)
        msgs = [{"role": "system", "content": _build_voice_system_prompt(payload, level)}]
        msgs.extend(_build_priming_messages(payload, level))
        msgs.append({"role": "user", "content": (
            f"State alert level is {level} and risk score is {risk_pct}% in one short sentence. "
            "No punctuation except percent. Under 80 characters."
        )})
        resp  = co.chat(model=COHERE_MODEL, messages=msgs)
        reply = _sanitise_for_tts(_cohere_extract_text(resp)) + suffix
    except Exception as e:
        print(f"[AgentLayer][Voice] Cohere init FAIL: {e}")
        reply = fallback
    reply = _sanitise_for_tts(reply)
    _voice_conversation_history.append({"role": "CHATBOT", "content": reply})
    return reply


def voice_agent_respond(user_text: str):
    global _voice_conversation_history, _voice_active_payload
    if not user_text or not user_text.strip():
        return "Please say that again", None
    _voice_conversation_history.append({"role": "USER", "content": user_text.strip()})
    if any(h in user_text.lower() for h in
           ("send report","send pdf","share evidence","dispatch report","forward report")):
        level   = determine_alert_level(float(_voice_active_payload.get("risk", 0.0)))
        results = dispatch_all_channels(_voice_active_payload, level)
        ok      = all(results.values())
        reply   = _sanitise_for_tts(
            f"Report dispatched to all channels  "
            f"{'Delivery confirmed' if ok else 'Some channels may have failed'}"
        )
        _voice_conversation_history.append({"role": "CHATBOT", "content": reply})
        return reply, "report_sent"
    if not COHERE_AVAILABLE:
        reply = _sanitise_for_tts(f"AI offline  You said {user_text.strip()[:60]}")
        _voice_conversation_history.append({"role": "CHATBOT", "content": reply})
        return reply, None
    try:
        co    = _cohere.ClientV2(COHERE_API_KEY)
        level = determine_alert_level(float(_voice_active_payload.get("risk", 0.0)))
        msgs  = [{"role": "system", "content": _build_voice_system_prompt(_voice_active_payload, level)}]
        msgs.extend(_build_priming_messages(_voice_active_payload, level))
        tail = _voice_conversation_history[-12:]
        start_idx = 0
        for i, m in enumerate(tail):
            if m.get("role", "USER").upper() in ("USER", "user"):
                start_idx = i
                break
        for m in tail[start_idx:]:
            role = m.get("role", "USER").upper()
            msgs.append({
                "role": "user" if role == "USER" else "assistant",
                "content": m.get("content", "")
            })
        if not msgs or msgs[-1]["role"] != "user":
            msgs.append({"role": "user", "content": user_text.strip()})
        resp  = co.chat(model=COHERE_MODEL, messages=msgs)
        reply = _sanitise_for_tts(_cohere_extract_text(resp))
    except Exception as e:
        print(f"[AgentLayer][Voice] Cohere respond FAIL: {e}")
        reply = "Processing error  Please repeat your question"
    _voice_conversation_history.append({"role": "CHATBOT", "content": reply})
    return reply, None

# ══════════════════════════════════════════════════════════════════════════════
# NOTIFICATION / DISPATCH
# ══════════════════════════════════════════════════════════════════════════════
def _fmt_msg(payload: dict, level: str) -> str:
    ts   = datetime.now(timezone.utc).strftime("%Y-%m-%d  %H:%M:%S UTC")
    risk = float(payload.get("risk", 0.0))
    jd   = payload.get("json_data") or {}
    summ = str(payload.get("summary", "No summary available."))
    sep, line = "=" * 56, "-" * 56
    header = (f"{sep}\n  SCENESOLVER  |  FORENSIC ALERT DOSSIER\n"
              f"  Level      :  {level}\n  Risk Score :  {round(risk*100)}%\n"
              f"  Timestamp  :  {ts}\n{sep}")
    body = ["", "DETECTION SUMMARY", line]
    body += [f"  {r.strip()}" for r in summ.split("\n") if r.strip()]
    if _has_pipeline_data(jd):
        body += ["", "STAGE-BY-STAGE BREAKDOWN", line]
        body += [f"  {r}" for r in _build_plain_context(jd, round(risk*100)).split("\n")]
    footer = (
        f"\n{line}\n  ACTION REQUIRED\n  Automated escalation triggered.\n"
        f"{line}\n  DISCLAIMER: ML investigative aid. Human review required.\n{sep}"
        if level == "CRITICAL" else
        f"\n{line}\n  DISCLAIMER: Investigative aid only. Human review required.\n{sep}"
    )
    return "\n".join([header] + body + [footer])


def _fmt_discord(payload: dict, level: str) -> str:
    ts   = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    risk = float(payload.get("risk", 0.0))
    jd   = payload.get("json_data") or {}
    summ = str(payload.get("summary", "No summary available."))
    col  = ":red_circle:" if level == "CRITICAL" else ":orange_circle:"
    lines = [
        f"**SCENESOLVER FORENSIC ALERT DOSSIER**  {col}",
        f"**Level:** `{level}`  |  **Risk:** `{round(risk*100)}%`  |  **Time:** `{ts}`", "```",
    ]
    lines += [f"  {r.strip()}" for r in summ.split("\n") if r.strip()]
    if _has_pipeline_data(jd):
        lines.append("")
        lines += [f"  {r}" for r in _build_plain_context(jd, round(risk*100)).split("\n")]
    lines += ["```"]
    if level == "CRITICAL":
        lines.append("**ACTION REQUIRED** — automated escalation triggered.")
    lines.append("*Investigative aid only — human review required before any action.*")
    return "\n".join(lines)


def build_alert_payload(pipeline_output: dict) -> dict:
    risk = float(pipeline_output.get("risk_score", 0.0))
    jd   = pipeline_output.get("json_data") or {}
    parts = [
        f"Status     : {jd.get('status','UNKNOWN')}",
        f"Risk Level : {jd.get('risk_level','?')}",
        f"Risk Score : {round(risk*100)}%",
    ]
    for key, label in [("stage1","TF Binary"),("stage2","Anomaly Type"),("stage3","Weapon"),
                        ("stage4","Audio"),("stage5","Motion"),("stage6","Fusion"),("stage7","RL3033")]:
        s = jd.get(key) or {}
        if not s: continue
        if key == "stage1":
            parts.append(f"{label:<12s}: {s.get('classification','?')} ({round(float(s.get('confidence',0))*100)}%)")
        elif key == "stage2" and s.get("class") not in ("Skipped", None):
            parts.append(f"{label:<12s}: {s.get('class','?')} ({round(float(s.get('confidence',0))*100)}%)")
        elif key == "stage3" and s.get("weapon_flag"):
            parts.append(f"{label:<12s}: {s.get('dominant_class','?')} (count={s.get('weapon_count',0)})")
        elif key == "stage4":
            parts.append(f"{label:<12s}: {s.get('status','?')} score={round(float(s.get('audio_score',0))*100)}%")
        elif key == "stage5":
            bhv = ", ".join(s.get("behaviors") or ["STATIC"])
            parts.append(f"{label:<12s}: intensity={round(float(s.get('motion_intensity',0))*100)}% [{bhv}]")
        elif key == "stage6":
            parts.append(f"{label:<12s}: {round(float(s.get('fused_risk',0))*100)}% [{s.get('risk_level','?')}]")
        elif key == "stage7":
            parts.append(f"{label:<12s}: {round(float(s.get('final_risk',0))*100)}% [{s.get('status','?')}]")
    narr = jd.get("llava_narrative","")
    if narr: parts.append(f"Narrative  : {str(narr)[:300]}")
    top_frame_path = None
    frame = pipeline_output.get("top_frame")
    if frame is not None:
        try:
            from PIL import Image as _I
            top_frame_path = "/tmp/agent_top_frame.jpg"
            _I.fromarray(frame).save(top_frame_path)
        except Exception:
            pass
    pdf_path = pipeline_output.get("pdf_path")
    if pdf_path and not os.path.exists(str(pdf_path)):
        pdf_path = None
    print(f"[AgentLayer] Payload risk={risk:.3f} jd_keys={list(jd.keys())[:8]}")
    return {"risk": risk, "summary": "\n".join(parts), "image": top_frame_path,
            "pdf": pdf_path, "json_data": jd}


def determine_alert_level(risk: float) -> str:
    if risk >= THRESH_CRITICAL: return "CRITICAL"
    if risk >= THRESH_HIGH:     return "HIGH"
    return "LOW"


def send_telegram_text(text: str) -> bool:
    url, ok = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage", True
    for chunk in [text[i:i+4000] for i in range(0, len(text), 4000)]:
        try:
            r = requests.post(url, data={"chat_id": TELEGRAM_CHAT_ID, "text": chunk, "parse_mode": ""}, timeout=15)
            if r.status_code != 200: ok = False
        except Exception: ok = False
    return ok

def send_telegram_image(path: str, caption: str = "") -> bool:
    if not path or not os.path.exists(path): return False
    try:
        with open(path, "rb") as f:
            r = requests.post(f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendPhoto",
                data={"chat_id": TELEGRAM_CHAT_ID, "caption": caption[:1000]}, files={"photo": f}, timeout=30)
        return r.status_code == 200
    except Exception: return False

def send_telegram_file(path: str, caption: str = "") -> bool:
    if not path or not os.path.exists(path): return False
    try:
        with open(path, "rb") as f:
            r = requests.post(f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendDocument",
                data={"chat_id": TELEGRAM_CHAT_ID, "caption": caption[:1000]}, files={"document": f}, timeout=60)
        return r.status_code == 200
    except Exception: return False

def send_telegram_alert(payload: dict, level: str) -> bool:
    ok = send_telegram_text(_fmt_msg(payload, level))
    if payload.get("image"): send_telegram_image(payload["image"], f"Peak anomaly frame [{level}]")
    if payload.get("pdf"):   send_telegram_file(payload["pdf"],   f"Forensic report [{level}]")
    return ok

def send_discord_text(content: str) -> bool:
    if not DISCORD_WEBHOOK_URL or DISCORD_WEBHOOK_URL.startswith("YOUR_"): return False
    ok = True
    for chunk in [content[i:i+1900] for i in range(0, len(content), 1900)]:
        try:
            r = requests.post(DISCORD_WEBHOOK_URL, json={"content": chunk}, timeout=15)
            if r.status_code not in (200, 204): ok = False
        except Exception: ok = False
    return ok

def send_discord_file(path: str, fname: str = None) -> bool:
    if not path or not os.path.exists(path): return False
    if not DISCORD_WEBHOOK_URL or DISCORD_WEBHOOK_URL.startswith("YOUR_"): return False
    try:
        with open(path, "rb") as f:
            r = requests.post(DISCORD_WEBHOOK_URL, files={"file": (fname or os.path.basename(path), f)}, timeout=60)
        return r.status_code in (200, 204)
    except Exception: return False

def send_discord_alert(payload: dict, level: str) -> bool:
    ok = send_discord_text(_fmt_discord(payload, level))
    if payload.get("image"): send_discord_file(payload["image"], "peak_frame.jpg")
    if payload.get("pdf"):   send_discord_file(payload["pdf"],   "forensic_report.pdf")
    return ok

def send_email_alert(payload: dict, level: str) -> bool:
    if not ALERT_EMAIL_FROM or not ALERT_EMAIL_PASS: return False
    try:
        msg = MIMEMultipart()
        risk = float(payload.get("risk", 0.0))
        msg["Subject"] = f"SceneSolver Forensic Alert [{level}] Risk {round(risk*100)}%"
        msg["From"], msg["To"] = ALERT_EMAIL_FROM, ALERT_EMAIL_TO
        msg.attach(MIMEText(_fmt_msg(payload, level), "plain", "utf-8"))
        for fp, fn in [(payload.get("image"), "peak_frame.jpg"), (payload.get("pdf"), "forensic_report.pdf")]:
            if fp and os.path.exists(str(fp)):
                with open(fp, "rb") as f:
                    p = MIMEBase("application", "octet-stream"); p.set_payload(f.read())
                encoders.encode_base64(p)
                p.add_header("Content-Disposition", f"attachment; filename={fn}")
                msg.attach(p)
        ctx = ssl.create_default_context()
        with smtplib.SMTP_SSL(ALERT_SMTP_SERVER, ALERT_SMTP_PORT, context=ctx) as srv:
            srv.login(ALERT_EMAIL_FROM, ALERT_EMAIL_PASS)
            srv.sendmail(ALERT_EMAIL_FROM, ALERT_EMAIL_TO, msg.as_string())
        return True
    except Exception as e:
        print(f"[Email] FAIL: {e}"); return False

def dispatch_all_channels(payload: dict, level: str) -> dict:
    print(f"[AgentLayer] Dispatching [{level}]...")
    return {
        "telegram": send_telegram_alert(payload, level),
        "discord":  send_discord_alert(payload, level),
        "email":    send_email_alert(payload, level),
    }

# ══════════════════════════════════════════════════════════════════════════════
# LFM2.5-Audio-1.5B — ASR + TTS
# ══════════════════════════════════════════════════════════════════════════════
_SPECIAL_TOKEN_RE = re.compile(r'<\|[a-z_]+\|>')

def _strip_special_tokens(text: str) -> str:
    return _SPECIAL_TOKEN_RE.sub('', text or '').strip()


def _looks_like_english(text: str) -> bool:
    """Check if text looks like usable speech. Very lenient — only reject CJK/cyrillic."""
    if not text or not text.strip():
        return False
    stripped = text.strip()
    # Only reject if predominantly CJK/cyrillic/thai characters
    if re.search(r'[\u3040-\u30ff\u4e00-\u9fff\uac00-\ud7af\u0e00-\u0e7f\u0400-\u04FF]', stripped):
        cjk = sum(1 for ch in stripped if '\u3040' <= ch <= '\u30ff' or '\u4e00' <= ch <= '\u9fff' or '\uac00' <= ch <= '\ud7af')
        if cjk / max(len(stripped), 1) > 0.3:
            return False
    return True


def _load_lfm_audio():
    global _lfm_processor, _lfm_model
    if not LFM_AUDIO_AVAILABLE:
        return None, None
    if _lfm_model is not None and _lfm_processor is not None:
        return _lfm_processor, _lfm_model
    with _lfm_lock:
        if _lfm_model is None:
            print("[AgentLayer] Loading LFM2.5-Audio-1.5B (first use, ~3.65GB)...")
            _lfm_processor = LFM2AudioProcessor.from_pretrained(HF_LFM_AUDIO_REPO).eval()
            _lfm_model     = LFM2AudioModel.from_pretrained(HF_LFM_AUDIO_REPO).eval()
            if torch is not None and torch.cuda.is_available():
                try:
                    _lfm_model = _lfm_model.to("cuda")
                    print("[AgentLayer] LFM2.5-Audio moved to CUDA")
                except Exception as e:
                    print(f"[AgentLayer] CUDA move skipped: {e}")
            print("[AgentLayer] LFM2.5-Audio-1.5B loaded")
    return _lfm_processor, _lfm_model


def _lfm_transcribe(audio_path: str) -> str:
    if not audio_path or not os.path.exists(str(audio_path)):
        return ""
    processor, model = _load_lfm_audio()
    if model is None or torchaudio is None or torch is None:
        print("[LFM-ASR] model or torch/torchaudio unavailable")
        return ""
    try:
        wav, sr = torchaudio.load(audio_path)
        with _lfm_lock, torch.inference_mode():
            chat = ChatState(processor)
            chat.new_turn("system")
            chat.add_text("Perform ASR.")
            chat.end_turn()
            chat.new_turn("user")
            chat.add_audio(wav, sr)
            chat.end_turn()
            chat.new_turn("assistant")
            chunks = []
            for t in model.generate_sequential(**chat, max_new_tokens=256):
                if t.numel() == 1:
                    chunks.append(processor.text.decode(t))
            text = "".join(chunks)
        text = _strip_special_tokens(text)
        if not _looks_like_english(text):
            print(f"[LFM-ASR] Rejected non-English/garbled output: '{text}'")
            return ""
        print(f"[LFM-ASR] {text}")
        return text
    except Exception as e:
        print(f"[LFM-ASR] FAIL: {e}")
        return ""


_LFM_VOICE_SYSTEM_PROMPT = (
    "Perform TTS. Use the US male voice. "
    "Speak calmly, confidently, warmly, and reassuringly. "
    "Use a steady, mature male delivery."
)

def lfm_text_to_speech(text: str, out_path: str = "/tmp/sherl0ck_reply.wav"):
    if not text or not text.strip():
        return None
    if torchaudio is None or torch is None:
        print("[LFM-TTS] torch/torchaudio unavailable")
        return None
    text = _sanitise_for_tts(text)
    processor, model = _load_lfm_audio()
    if model is None:
        print("[LFM-TTS] model unavailable")
        return None
    try:
        with _lfm_lock, torch.inference_mode():
            chat = ChatState(processor)
            chat.new_turn("system")
            chat.add_text(_LFM_VOICE_SYSTEM_PROMPT)
            chat.end_turn()
            chat.new_turn("user")
            chat.add_text(text)
            chat.end_turn()
            chat.new_turn("assistant")
            audio_out = []
            for token in model.generate_sequential(
                **chat, max_new_tokens=512, audio_temperature=0.8, audio_top_k=64,
            ):
                if token.numel() > 1:
                    audio_out.append(token)
            if not audio_out:
                print("[LFM-TTS] no audio tokens generated")
                return None
            codes_list = audio_out[:-1] if len(audio_out) > 1 else audio_out
            if not codes_list:
                print("[LFM-TTS] no audio codes after trimming EOA")
                return None
            audio_codes = torch.stack(codes_list, 1).unsqueeze(0)
            waveform = processor.decode(audio_codes)
        wf = waveform.detach().float().cpu()
        if wf.numel() < 240:
            print(f"[LFM-TTS] waveform too short numel={wf.numel()}")
            return None
        if not torch.isfinite(wf).all():
            print("[LFM-TTS] waveform has non-finite values")
            return None
        if wf.dim() == 1:
            wf = wf.unsqueeze(0)
        torchaudio.save(out_path, wf, 24000)
        try:
            if not os.path.exists(out_path) or os.path.getsize(out_path) < 44:
                print("[LFM-TTS] saved file missing/too small")
                return None
            with open(out_path, "rb") as _bf:
                _hdr = _bf.read(12)
            if _hdr[0:4] != b"RIFF" or _hdr[8:12] != b"WAVE":
                print("[LFM-TTS] saved file not RIFF/WAVE")
                return None
            print(f"[LFM-TTS] saved ok bytes={os.path.getsize(out_path)}")
        except Exception as _ve:
            print(f"[LFM-TTS] saved file check fail: {_ve}")
            return None
        return out_path
    except Exception as e:
        print(f"[LFM-TTS] FAIL: {e}")
        return None


def _caller_tune_uri() -> str:
    path = (CALLER_TUNE_PATH or "").strip()
    if not path or not os.path.exists(path): return ""
    try:
        ext  = os.path.splitext(path)[1].lower()
        mime = "audio/mpeg" if ext == ".mp3" else "audio/wav"
        with open(path, "rb") as f:
            data = base64.b64encode(f.read()).decode()
        return f"data:{mime};base64,{data}"
    except Exception as e:
        print(f"[CallerTune] FAIL: {e}"); return ""


def _make_synth_ring_bytes() -> bytes:
    sr = 22050
    duration = 2.4
    freq = 480
    n = int(sr * duration)
    buf = io.BytesIO()
    with wave.open(buf, 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        for i in range(n):
            t = i / sr
            in_burst = (t < 0.40) or (0.45 <= t < 0.85)
            amp = int(28000 * math.sin(2 * math.pi * freq * t)) if in_burst else 0
            wf.writeframes(struct.pack('<h', amp))
    return buf.getvalue()


def _stream_ringtone():
    CHUNK   = 16384
    MAX_SEC = 60
    start   = time.time()
    path = (CALLER_TUNE_PATH or "").strip()
    if path and os.path.exists(path):
        print(f"[Ringtone] Streaming file: {path}")
        try:
            with open(path, "rb") as f:
                raw = f.read()
        except Exception as e:
            print(f"[Ringtone] File read failed: {e}")
            raw = _make_synth_ring_bytes()
    else:
        print("[Ringtone] File not found, using synth ring")
        raw = _make_synth_ring_bytes()
    header_yielded = False
    while (time.time() - start) < MAX_SEC:
        if _AGENT_STATE.get("ring_stopped", False):
            print("[Ringtone] Ring stopped, stopping stream")
            break
        if not header_yielded:
            yield raw
            header_yielded = True
        else:
            yield raw
        time.sleep(2.5)


def _stop_ringtone():
    return b""


# ══════════════════════════════════════════════════════════════════════════════
# AGENT STATE + TTS EXECUTOR
# ══════════════════════════════════════════════════════════════════════════════
_AGENT_STATE = {
    "payload": {},
    "level": "LOW",
    "voice_live": False,
    "text_live": False,
    "ring_stopped": False,
    "voice_session_id": "",
    "voice_turn": 0,
}

import concurrent.futures as _cf
_tts_executor = _cf.ThreadPoolExecutor(max_workers=4, thread_name_prefix="tts")


def _tts_blocking(text: str, out_path: str = "/tmp/sherl0ck_reply.wav"):
    try:
        future = _tts_executor.submit(lfm_text_to_speech, text, out_path)
        return future.result(timeout=90)
    except _cf.TimeoutError:
        print("[TTS] Timeout after 90s")
        return None
    except Exception as e:
        print(f"[TTS] Thread error: {e}")
        return None


def _ffmpeg_to_wav(src_path: str, dst_path: str, sr: int = 24000) -> bool:
    try:
        cmd = [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", src_path,
            "-ac", "1", "-ar", str(sr),
            "-c:a", "pcm_s16le",
            dst_path,
        ]
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        if r.returncode != 0:
            print(f"[VoiceTransport] ffmpeg FAIL rc={r.returncode} stderr={(r.stderr or '')[:400]}")
            return False
        if not os.path.exists(dst_path) or os.path.getsize(dst_path) < 44:
            print("[VoiceTransport] ffmpeg produced empty/missing wav")
            return False
        return True
    except Exception as e:
        print(f"[VoiceTransport] ffmpeg exception: {e}")
        return False


def _wav_duration_sec(path: str) -> float:
    try:
        with wave.open(path, "rb") as wf:
            return wf.getnframes() / float(wf.getframerate() or 1)
    except Exception:
        return 0.0


def _validate_wav_file(path: str, expect_sr: int = 24000) -> tuple:
    """Return (ok, meta_or_error). Rejects non-RIFF / empty / absurd files."""
    try:
        if not path or not os.path.exists(path):
            return False, "missing"
        sz = os.path.getsize(path)
        if sz < 44:
            return False, "too_small"
        with open(path, "rb") as f:
            hdr = f.read(12)
        if len(hdr) < 12 or hdr[0:4] != b"RIFF" or hdr[8:12] != b"WAVE":
            return False, "not_riff_wave"
        with wave.open(path, "rb") as wf:
            ch = wf.getnchannels()
            sw = wf.getsampwidth()
            sr = wf.getframerate()
            nframes = wf.getnframes()
            dur = nframes / float(sr or 1)
        if ch < 1 or sw < 1 or sr < 8000 or nframes < 1:
            return False, f"bad_params ch={ch} sw={sw} sr={sr} n={nframes}"
        # Allow nearby sample rates; ffmpeg should have forced 24k
        if abs(int(sr) - int(expect_sr)) > 100 and int(sr) not in (16000, 22050, 24000, 44100, 48000):
            return False, f"unexpected_sr={sr}"
        return True, {"duration": float(dur), "sr": int(sr), "channels": int(ch),
                      "sampwidth": int(sw), "nframes": int(nframes), "bytes": int(sz)}
    except Exception as e:
        return False, f"exc:{e}"


# Serialize voice turns so opening / barge-in / double-submit cannot overlap LFM+Cohere
_voice_turn_lock = threading.Lock()
_VOICE_MAX_BODY_BYTES = 3 * 1024 * 1024  # 3 MB raw WebM upper bound
_processed_turns: set = set()
_processed_turns_lock = threading.Lock()


def process_voice_bytes(raw: bytes, session_id: str = "", turn_id: str = "",
                        is_opening: bool = False) -> dict:
    t0 = time.time()
    result = {
        "ok": False,
        "status": "error",
        "transcript": "",
        "reply": "",
        "audio_b64": "",
        "audio_mime": "audio/wav",
        "turn_id": turn_id,
        "session_id": session_id,
        "action": None,
        "error": "",
        "timings_ms": {},
    }
    nbytes = len(raw) if raw else 0
    print(f"[VoiceTransport] RECEIVED bytes={nbytes} "
          f"session={session_id[:8] if session_id else '-'} turn={turn_id}")

    # Reject oversized bodies
    if nbytes > _VOICE_MAX_BODY_BYTES:
        result["error"] = "body_too_large"
        result["status"] = "body_too_large"
        print(f"[VoiceTransport] REJECT body_too_large bytes={nbytes}")
        return result

    # Duplicate turn guard (same session+turn)
    turn_key = f"{session_id}:{turn_id}:{int(bool(is_opening))}"
    with _processed_turns_lock:
        if turn_id and turn_key in _processed_turns and not is_opening:
            result["error"] = "duplicate_turn"
            result["status"] = "duplicate_turn"
            print(f"[VoiceTransport] REJECT duplicate_turn {turn_key}")
            return result
        if turn_id:
            _processed_turns.add(turn_key)
            # bound memory
            if len(_processed_turns) > 500:
                _processed_turns.clear()

    # Must hold voice_live for non-opening (session active)
    if not is_opening and not _AGENT_STATE.get("voice_live"):
        result["error"] = "call_not_active"
        result["status"] = "call_not_active"
        print("[VoiceTransport] REJECT call_not_active")
        return result

    # Serialize entire turn (LFM lock alone is not enough across ASR+Cohere+TTS)
    if not _voice_turn_lock.acquire(blocking=True, timeout=120):
        result["error"] = "turn_busy"
        result["status"] = "turn_busy"
        print("[VoiceTransport] REJECT turn_busy")
        return result
    try:
        return _process_voice_bytes_locked(raw, session_id, turn_id, is_opening, result, t0)
    finally:
        _voice_turn_lock.release()


def _process_voice_bytes_locked(raw, session_id, turn_id, is_opening, result, t0):
    if is_opening:
        try:
            payload = _AGENT_STATE.get("payload", {}) or _voice_active_payload
            level   = _AGENT_STATE.get("level", "LOW")
            print("[Voice] OPENING START")
            opening = voice_agent_init(payload, level)
            print(f"[Voice] OPENING TEXT chars={len(opening)}")
            out_path = f"/tmp/sherl0ck_opening_{int(time.time()*1000)}.wav"
            print("[Voice] TTS_START (opening)")
            tts_path = _tts_blocking(opening, out_path)
            print(f"[Voice] TTS_DONE path={tts_path}")
            if tts_path and os.path.exists(tts_path):
                vok, vmeta = _validate_wav_file(tts_path, expect_sr=24000)
                if not vok:
                    print(f"[Voice] OPENING TTS_INVALID_WAV {vmeta}")
                    result.update({"ok": True, "status": "ok_no_audio", "reply": opening,
                                   "error": f"tts_invalid_wav:{vmeta}"})
                else:
                    with open(tts_path, "rb") as f:
                        ab = f.read()
                    result.update({
                        "ok": True,
                        "status": "ok",
                        "transcript": "",
                        "reply": opening,
                        "audio_b64": base64.b64encode(ab).decode("ascii"),
                        "audio_mime": "audio/wav",
                    })
                    print(f"[Voice] AUDIO_RESPONSE bytes={len(ab)} (opening)")
            else:
                print("[Voice] TTS_FAILURE (opening)")
                result.update({"ok": True, "status": "ok_no_audio", "reply": opening,
                               "error": "tts_failed"})
            result["timings_ms"]["total"] = int((time.time() - t0) * 1000)
            return result
        except Exception as e:
            print(f"[Voice] OPENING FAIL: {e}")
            traceback.print_exc()
            result["error"] = str(e)
            return result

    if not raw or len(raw) < 64:
        print("[VoiceTransport] Empty or tiny packet — rejecting")
        result["error"] = "empty_packet"
        result["status"] = "empty"
        return result

    stamp = int(time.time() * 1000)
    webm_path = f"/tmp/ssvoice_in_{stamp}.webm"
    wav_path  = f"/tmp/ssvoice_in_{stamp}.wav"
    try:
        with open(webm_path, "wb") as f:
            f.write(raw)
        print(f"[VoiceTransport] WEBM bytes={os.path.getsize(webm_path)} path={webm_path}")
    except Exception as e:
        result["error"] = f"write_webm_failed:{e}"
        return result

    print("[Voice] FFMPEG_START")
    ok_conv = _ffmpeg_to_wav(webm_path, wav_path, sr=24000)
    if not ok_conv:
        # HARD FAILURE — never rename/copy WebM to .wav
        result["error"] = "ffmpeg_failed"
        result["status"] = "convert_error"
        print("[Voice] FFMPEG_FAILURE")
        try:
            if os.path.exists(webm_path):
                os.remove(webm_path)
        except Exception:
            pass
        return result
    print("[Voice] FFMPEG_SUCCESS")

    wav_sz = os.path.getsize(wav_path)
    # Validate real WAV structure before LFM
    wav_ok, wav_meta = _validate_wav_file(wav_path, expect_sr=24000)
    if not wav_ok:
        result["error"] = f"invalid_wav:{wav_meta}"
        result["status"] = "invalid_wav"
        print(f"[Voice] WAV_INVALID {wav_meta}")
        for p in (webm_path, wav_path):
            try:
                if os.path.exists(p):
                    os.remove(p)
            except Exception:
                pass
        return result

    dur = float(wav_meta.get("duration", 0.0))
    print(f"[VoiceTransport] WAV bytes={wav_sz} duration={dur:.3f}s sr={wav_meta.get('sr')} ch={wav_meta.get('channels')}")
    result["timings_ms"]["convert"] = int((time.time() - t0) * 1000)

    if dur < 0.15 or wav_sz < 1000:
        print("[VoiceTransport] utterance too short — skip ASR")
        result["status"] = "too_short"
        result["error"] = "utterance_too_short"
        result["ok"] = True
        for p in (webm_path, wav_path):
            try:
                if os.path.exists(p):
                    os.remove(p)
            except Exception:
                pass
        return result

    print("[Voice] ASR_START")
    t_asr = time.time()
    transcript = _lfm_transcribe(wav_path)
    result["timings_ms"]["asr"] = int((time.time() - t_asr) * 1000)
    print(f"[Voice] ASR_DONE transcript={transcript!r}")
    result["transcript"] = transcript or ""

    if not transcript or len(transcript.strip()) < 2:
        result["status"] = "no_speech"
        result["error"] = "no_speech"
        result["ok"] = True
        print("[Voice] No usable speech — returning to LISTENING")
        return result

    print("[Voice] COHERE_START")
    t_co = time.time()
    reply, action = voice_agent_respond(transcript)
    result["timings_ms"]["cohere"] = int((time.time() - t_co) * 1000)
    print(f"[Voice] COHERE_DONE reply={reply[:120]!r} action={action}")
    result["reply"] = reply
    result["action"] = action

    print("[Voice] TTS_START")
    t_tts = time.time()
    out_path = f"/tmp/sherl0ck_reply_{stamp}.wav"
    tts_path = _tts_blocking(reply, out_path)
    result["timings_ms"]["tts"] = int((time.time() - t_tts) * 1000)
    print(f"[Voice] TTS_DONE path={tts_path}")

    if tts_path and os.path.exists(tts_path):
        vok, vmeta = _validate_wav_file(tts_path, expect_sr=24000)
        if not vok:
            print(f"[Voice] TTS_INVALID_WAV {vmeta}")
            result["ok"] = True
            result["status"] = "ok_no_audio"
            result["error"] = f"tts_invalid_wav:{vmeta}"
        else:
            with open(tts_path, "rb") as f:
                ab = f.read()
            result["audio_b64"] = base64.b64encode(ab).decode("ascii")
            result["audio_mime"] = "audio/wav"
            print(f"[Voice] AUDIO_RESPONSE bytes={len(ab)} turn={turn_id}")
            result["ok"] = True
            result["status"] = "ok"
    else:
        result["ok"] = True
        result["status"] = "ok_no_audio"
        result["error"] = "tts_failed"
        print("[Voice] TTS_FAILURE")

    result["timings_ms"]["total"] = int((time.time() - t0) * 1000)
    for p in (webm_path, wav_path):
        try:
            if os.path.exists(p):
                os.remove(p)
        except Exception:
            pass
    return result


_SSVOICE_ROUTES_REGISTERED = False


def _register_ssvoice_routes(fastapi_app):
    global _SSVOICE_ROUTES_REGISTERED
    if _SSVOICE_ROUTES_REGISTERED:
        return
    try:
        from fastapi import Request
        from fastapi.responses import JSONResponse
        from fastapi.middleware.cors import CORSMiddleware
    except ImportError:
        print("[VoiceTransport] fastapi import failed — routes not registered")
        return

    try:
        fastapi_app.add_middleware(
            CORSMiddleware,
            allow_origins=["*"],
            allow_credentials=True,
            allow_methods=["*"],
            allow_headers=["*"],
        )
    except Exception as e:
        print(f"[VoiceTransport] CORS middleware note: {e}")

    @fastapi_app.post("/ssvoice/turn")
    async def ssvoice_turn(request: Request):
        import anyio
        try:
            session_id = request.headers.get("x-ss-session", "") or ""
            turn_id    = request.headers.get("x-ss-turn", "") or ""
            is_opening = request.headers.get("x-ss-opening", "0") == "1"
            # Reject absurd Content-Length early when present
            cl = request.headers.get("content-length")
            if cl is not None:
                try:
                    if int(cl) > _VOICE_MAX_BODY_BYTES:
                        return JSONResponse({"ok": False, "status": "body_too_large",
                                             "error": "body_too_large"}, status_code=413)
                except Exception:
                    pass
            raw = await request.body()
            print(f"[VoiceTransport] REQUEST_ENTERED "
                  f"session_id={session_id} body_bytes={len(raw)} "
                  f"opening={is_opening} turn={turn_id}")
            if len(raw) > _VOICE_MAX_BODY_BYTES:
                return JSONResponse({"ok": False, "status": "body_too_large",
                                     "error": "body_too_large"}, status_code=413)
            active = _AGENT_STATE.get("voice_session_id", "")
            if active and session_id and session_id != active and not is_opening:
                print(f"[VoiceTransport] stale session {session_id[:8]} != {active[:8]}")
                return JSONResponse({"ok": False, "status": "stale_session",
                                     "error": "stale_session", "turn_id": turn_id}, status_code=409)
            # Run blocking ASR/LLM/TTS off the event loop
            out = await anyio.to_thread.run_sync(
                process_voice_bytes, raw, session_id, turn_id, is_opening
            )
            return JSONResponse(out)
        except Exception as e:
            print(f"[VoiceTransport] /ssvoice/turn EXCEPTION: {e}")
            traceback.print_exc()
            return JSONResponse({"ok": False, "status": "error", "error": str(e)}, status_code=500)

    @fastapi_app.get("/ssvoice/health")
    async def ssvoice_health():
        return JSONResponse({
            "ok": True,
            "lfm": LFM_AUDIO_AVAILABLE,
            "cohere": COHERE_AVAILABLE,
            "voice_live": bool(_AGENT_STATE.get("voice_live")),
            "session": (_AGENT_STATE.get("voice_session_id") or "")[:8],
        })

    @fastapi_app.post("/ssvoice/end")
    async def ssvoice_end(request: Request):
        sid = request.headers.get("x-ss-session", "")
        print(f"[VoiceTransport] END session={sid[:8] if sid else '-'}")
        _AGENT_STATE["voice_live"] = False
        if sid and sid == _AGENT_STATE.get("voice_session_id"):
            _AGENT_STATE["voice_session_id"] = ""
        return JSONResponse({"ok": True})

    @fastapi_app.post("/ssvoice/accept")
    async def ssvoice_accept(request: Request):
        """Mint/return the authoritative voice session. Browser calls this from the Accept gesture."""
        global _voice_active_payload
        try:
            payload = _AGENT_STATE.get("payload", {}) or {}
            level = _AGENT_STATE.get("level", "LOW")
            # Reuse existing live session if already minted by Gradio accept_call
            sid = _AGENT_STATE.get("voice_session_id") or ""
            if not sid or not _AGENT_STATE.get("voice_live"):
                sid = uuid.uuid4().hex
                _AGENT_STATE["voice_session_id"] = sid
                _AGENT_STATE["voice_turn"] = 0
            _AGENT_STATE["voice_live"] = True
            _AGENT_STATE["ring_stopped"] = True
            _voice_active_payload = payload
            print(f"[VoiceTransport] ACCEPT session_id={sid} level={level} "
                  f"has_payload={bool(payload)}")
            return JSONResponse({
                "ok": True,
                "session_id": sid,
                "level": level,
                "risk": float(payload.get("risk", 0.0) or 0.0),
            })
        except Exception as e:
            print(f"[VoiceTransport] ACCEPT FAIL: {e}")
            traceback.print_exc()
            return JSONResponse({"ok": False, "error": str(e)}, status_code=500)


    @fastapi_app.get("/ssvoice/ringtone")
    async def ssvoice_ringtone():
        """Serve the ringtone audio file."""
        from fastapi.responses import FileResponse
        path = (CALLER_TUNE_PATH or "").strip()
        if path and os.path.exists(path):
            return FileResponse(path, media_type="audio/mpeg")
        # Fallback: generate synth ring
        import tempfile
        synth = _make_synth_ring_bytes()
        tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        tmp.write(synth)
        tmp.close()
        return FileResponse(tmp.name, media_type="audio/wav")




    _SSVOICE_ROUTES_REGISTERED = True
    print("[VoiceTransport] Routes registered: POST /ssvoice/turn GET /ssvoice/health "
          "POST /ssvoice/end POST /ssvoice/accept GET /ssvoice/ringtone")


AGENT_CSS = APP_CSS + """
@keyframes phone-shake {
  0%,100% { transform:rotate(0deg) scale(1); }
  10%,30%,50%,70% { transform:rotate(-14deg) scale(1.08); }
  20%,40%,60%,80% { transform:rotate(14deg) scale(1.08); }
  90% { transform:rotate(-6deg) scale(1); }
}
@keyframes ring-pulse {
  0%,100% { box-shadow:0 0 0 0 rgba(255,69,0,0); }
  50% { box-shadow:0 0 0 18px rgba(255,69,0,0); }
}
@keyframes fade-in-down {
  from { opacity:0; transform:translateY(-18px) scale(.97); }
  to { opacity:1; transform:translateY(0) scale(1); }
}
@keyframes pulse-dot {
  0%,100% { opacity:1; transform:scale(1); } 50% { opacity:.25; transform:scale(.65); }
}
@keyframes scan-line {
  0% { transform:translateY(-100%); } 100% { transform:translateY(100vh); }
}
@keyframes progress-bar {
  0% { transform:scaleX(0); } 100% { transform:scaleX(1); }
}
@keyframes glow-border {
  0%,100% { border-color:rgba(255,69,0,.5); }
  50% { border-color:rgba(255,69,0,1); box-shadow:0 0 20px rgba(255,69,0,.18); }
}
@keyframes status-flash {
  0%,100% { opacity:1; } 50% { opacity:.55; }
}
#ss-ringtone-out { position:fixed !important; bottom:20px !important; right:20px !important; z-index:2147483646 !important; opacity:0.9 !important; max-width:200px !important; }
#ss-overlay-root {
  position:fixed !important; top:0 !important; left:0 !important;
  width:0 !important; height:0 !important; overflow:visible !important;
  pointer-events:none !important; z-index:2147483647 !important;
}
#ss-call-card {
  position:fixed !important; top:20px !important; right:20px !important;
  width:360px !important; pointer-events:all !important;
  background:rgba(6,3,1,.97) !important; backdrop-filter:blur(16px) !important;
  border:1px solid rgba(255,69,0,.7) !important; border-radius:4px !important;
  overflow:hidden !important;
  animation:fade-in-down .3s cubic-bezier(.2,.8,.3,1), ring-pulse 2.4s ease-in-out infinite !important;
  box-shadow:0 24px 64px rgba(0,0,0,.95) !important;
}
#ss-call-card .cc-scan {
  position:absolute !important; top:0 !important; left:0 !important;
  width:100% !important; height:2px !important;
  background:linear-gradient(90deg,transparent,rgba(255,69,0,.5),transparent) !important;
  animation:scan-line 3s linear infinite !important; pointer-events:none !important;
}
#ss-call-card .cc-header {
  display:flex !important; align-items:center !important; justify-content:space-between !important;
  padding:6px 14px !important; background:rgba(255,69,0,.07) !important;
  border-bottom:1px solid rgba(255,69,0,.18) !important;
}
#ss-call-card .cc-header-label {
  font-family:var(--mono) !important; font-size:.5rem !important; letter-spacing:.2em !important;
  text-transform:uppercase !important; color:rgba(255,69,0,.85) !important; font-weight:700 !important;
}
#ss-call-card .cc-header-dot {
  width:6px !important; height:6px !important; border-radius:50% !important;
  background:#FF4500 !important; animation:pulse-dot 1s ease-in-out infinite !important;
}
#ss-call-card .cc-body {
  padding:18px 18px 14px !important; display:flex !important; align-items:center !important; gap:16px !important;
}
#ss-call-card .cc-phone-icon {
  width:52px !important; height:52px !important; border-radius:50% !important;
  background:rgba(255,69,0,.1) !important; border:1px solid rgba(255,69,0,.3) !important;
  display:flex !important; align-items:center !important; justify-content:center !important;
  animation:phone-shake 1.2s ease-in-out infinite !important; flex-shrink:0 !important;
}
#ss-call-card .cc-phone-icon.stopped { animation:none !important; }
#ss-call-card .cc-info-title {
  font-family:var(--serif) !important; font-size:.95rem !important; font-weight:700 !important;
  color:#fff !important; line-height:1.3 !important;
}
#ss-call-card .cc-info-sub {
  font-family:var(--mono) !important; font-size:.6rem !important;
  color:rgba(255,255,255,.45) !important; margin-top:4px !important;
}
#ss-call-card .cc-risk-badge {
  font-family:var(--mono) !important; font-size:.65rem !important; font-weight:700 !important;
  color:#FF4500 !important; background:rgba(255,69,0,.09) !important;
  border:1px solid rgba(255,69,0,.3) !important; border-radius:3px !important;
  padding:2px 8px !important; margin-top:5px !important; display:inline-block !important;
  animation:status-flash 2s ease-in-out infinite !important;
}
#ss-call-card .cc-actions { display:flex !important; gap:8px !important; padding:0 18px 18px !important; }
.cc-btn-accept {
  flex:1 !important; background:linear-gradient(135deg,#2e7d32,#388e3c) !important;
  color:#fff !important; font-family:var(--mono) !important; font-size:.6rem !important;
  letter-spacing:.1em !important; font-weight:700 !important; text-transform:uppercase !important;
  border:none !important; border-radius:3px !important; padding:11px 0 !important; cursor:pointer !important;
}
.cc-btn-accept:hover { background:linear-gradient(135deg,#388e3c,#43a047) !important; }
.cc-btn-dismiss {
  background:rgba(255,255,255,.05) !important; color:rgba(255,255,255,.4) !important;
  font-family:var(--mono) !important; font-size:.6rem !important; letter-spacing:.1em !important;
  font-weight:600 !important; text-transform:uppercase !important;
  border:1px solid rgba(255,255,255,.1) !important; border-radius:3px !important;
  padding:11px 14px !important; cursor:pointer !important;
}
.cc-btn-dismiss:hover { background:rgba(255,255,255,.09) !important; }
#ss-call-card .cc-progress { height:2px !important; background:rgba(255,69,0,.1) !important; }
#ss-call-card .cc-progress-fill {
  height:100% !important; width:100% !important;
  background:linear-gradient(90deg,#FF4500,#FF8C00) !important;
  transform-origin:left !important; animation:progress-bar 4s linear infinite !important;
}
.agent-page { max-width:860px; margin:0 auto; padding:0 28px 80px; font-family:var(--serif); }
.agent-sec-label {
  font-family:var(--mono); font-size:.62rem; letter-spacing:.14em; text-transform:uppercase;
  color:#ddd; margin-bottom:10px; font-weight:800; border-left:3px solid var(--orange); padding-left:10px;
}
.agent-status-card {
  background:var(--surface); border:1px solid var(--border); border-radius:3px; padding:20px 22px;
  display:flex; align-items:flex-start; justify-content:space-between; gap:16px;
  position:relative; overflow:hidden;
}
.agent-status-card.critical { border-color:rgba(255,69,0,.4); background:rgba(255,69,0,.04); animation:glow-border 2s ease-in-out infinite; }
.agent-status-card.high { border-color:rgba(255,140,0,.35); background:rgba(255,140,0,.03); }
.agent-dep-chips { display:flex; flex-wrap:wrap; gap:5px; margin-top:10px; }
.agent-dep-chip {
  font-family:var(--mono); font-size:.5rem; letter-spacing:.08em;
  background:rgba(255,255,255,.05); border:1px solid rgba(255,255,255,.1);
  border-radius:3px; padding:3px 9px; display:flex; align-items:center; gap:5px;
}
.agent-call-strip {
  border-radius:3px; padding:14px 18px; display:flex; align-items:center; gap:14px;
  background:rgba(255,69,0,.06); border:1px solid rgba(255,69,0,.28);
  margin:12px 0 0; cursor:pointer; user-select:none;
}
.strip-phone { animation:phone-shake 1.2s ease-in-out infinite; flex-shrink:0; }
.agent-dual-panel { display:grid; grid-template-columns:1fr 1fr; gap:20px; margin-top:8px; align-items:start; }
@media (max-width:700px) { .agent-dual-panel { grid-template-columns:1fr; } }
.agent-panel { background:var(--surface); border:1px solid var(--border); border-radius:3px; padding:18px 18px 16px; min-height:420px; }
.agent-panel.voice-panel { border-color:rgba(76,175,80,.15); background:rgba(76,175,80,.02); }
.agent-panel.text-panel { border-color:rgba(100,149,237,.18); background:rgba(100,149,237,.02); }
.panel-header {
  font-family:var(--mono); font-size:.52rem; letter-spacing:.16em; text-transform:uppercase;
  font-weight:700; margin-bottom:10px; display:flex; align-items:center; gap:8px;
}
.panel-header.voice-header { color:rgba(76,175,80,.85); }
.panel-header.text-header { color:rgba(100,149,237,.85); }
.experimental-badge {
  font-family:var(--mono); font-size:.42rem; letter-spacing:.1em; text-transform:uppercase;
  font-weight:700; color:rgba(255,193,7,.9); background:rgba(255,193,7,.08);
  border:1px solid rgba(255,193,7,.25); border-radius:3px; padding:2px 7px; margin-left:4px;
}
.panel-desc {
  font-family:var(--mono); font-size:.52rem; letter-spacing:.05em; color:rgba(255,255,255,.25);
  margin-bottom:12px; line-height:1.85;
}
.chat-display {
  background:#050505; border:1px solid var(--border); border-radius:3px; padding:16px 18px;
  min-height:280px; max-height:440px; overflow-y:auto; display:flex; flex-direction:column; gap:16px;
}
.chat-display .msg-agent { display:flex; flex-direction:column; gap:5px; align-self:flex-start; max-width:96%; }
.chat-display .msg-agent .msg-lbl {
  font-family:var(--mono); font-size:.55rem; letter-spacing:.1em; text-transform:uppercase;
  color:var(--orange); font-weight:600; display:flex; align-items:center; gap:6px;
}
.chat-display .msg-agent .msg-lbl .lbl-dot {
  width:6px; height:6px; border-radius:50%; background:var(--orange); flex-shrink:0; display:inline-block;
}
.chat-display .msg-agent .msg-lbl .brand-name { display:inline-flex; align-items:baseline; }
.chat-display .msg-agent .msg-lbl .brand-zero { font-family:'Times New Roman',Times,serif; font-weight:900; font-size:1.1em; }
.chat-display .msg-agent .msg-bubble {
  background:#080808; border:1px solid var(--border); border-left:3px solid rgba(255,140,0,.5);
  border-radius:0 3px 3px 3px; padding:14px 16px; font-family:var(--serif); font-size:.84rem;
  line-height:1.8; color:#e0ddd8;
}
.chat-display .msg-you { display:flex; flex-direction:column; gap:5px; align-self:flex-end; max-width:78%; }
.chat-display .msg-you .msg-lbl {
  font-family:var(--mono); font-size:.55rem; letter-spacing:.1em; text-transform:uppercase;
  color:rgba(100,149,237,.7); font-weight:600; text-align:right;
  display:flex; align-items:center; justify-content:flex-end; gap:6px;
}
.chat-display .msg-you .msg-lbl .lbl-dot {
  width:6px; height:6px; border-radius:50%; background:rgba(100,149,237,.7); flex-shrink:0; display:inline-block;
}
.chat-display .msg-you .msg-bubble {
  background:#0d0d0d; border:1px solid #2a2a2a; border-right:3px solid rgba(100,149,237,.4);
  border-radius:3px 0 3px 3px; padding:12px 16px; font-family:var(--serif); font-size:.84rem;
  line-height:1.75; color:rgba(220,230,255,.85);
}
.chat-display .msg-system {
  align-self:center; font-family:var(--mono); font-size:.6rem; font-weight:600; letter-spacing:.06em;
  color:rgba(76,175,80,.6); background:rgba(76,175,80,.05); border:1px solid rgba(76,175,80,.15);
  border-radius:3px; padding:6px 12px; text-align:center;
}
.chat-display .msg-empty {
  flex:1; display:flex; flex-direction:column; align-items:center; justify-content:center;
  font-family:var(--serif); font-size:.88rem; color:#333; text-align:center; padding:48px 24px;
}
#text-chat-input textarea, #text-chat-input input {
  background:#0d0d0d !important; color:#e0ddd8 !important; font-family:var(--serif) !important;
  font-size:.86rem !important; border:1px solid #2a2a2a !important; border-radius:3px !important;
  padding:12px 14px !important; min-height:52px !important; height:52px !important;
  resize:none !important; overflow:hidden !important;
}
#voice-conv-box textarea {
  background:#060606 !important; color:rgba(200,200,200,.7) !important; font-family:var(--mono) !important;
  font-size:.65rem !important; line-height:1.9 !important; border:1px solid rgba(76,175,80,.12) !important;
  border-radius:3px !important;
}
#dispatch-status textarea {
  background:#060606 !important; color:rgba(255,255,255,.35) !important; font-family:var(--mono) !important;
  font-size:.63rem !important; border:1px solid var(--border) !important; border-radius:3px !important;
}
.agent-btn, .agent-btn button {
  height:40px !important; min-height:40px !important; padding:0 20px !important;
  font-family:var(--mono) !important; font-size:.65rem !important; letter-spacing:.12em !important;
  font-weight:700 !important; text-transform:uppercase !important; border-radius:3px !important;
  border:none !important; cursor:pointer !important; background:#ffffff !important; color:#000000 !important;
}
.agent-btn:hover, .agent-btn button:hover { background:var(--orange) !important; }
.btn-accept, .btn-accept button { background:linear-gradient(135deg,#1b5e20,#2e7d32) !important; color:#fff !important; }
.btn-end, .btn-end button { background:linear-gradient(135deg,#7f0000,#b71c1c) !important; color:#fff !important; }
.btn-dispatch, .btn-dispatch button { background:#ffffff !important; color:#000 !important; width:100% !important; height:44px !important; }
.btn-send, .btn-send button { background:#ffffff !important; color:#000 !important; min-height:44px !important; width:100% !important; }
.btn-start, .btn-start button { background:#ffffff !important; color:#000 !important; width:100% !important; height:40px !important; }
.live-indicator {
  display:inline-flex; align-items:center; gap:8px; background:rgba(76,175,80,.08);
  border:1px solid rgba(76,175,80,.25); border-radius:3px; padding:5px 12px; margin:10px 0 6px;
}
.live-dot { display:inline-block; width:6px; height:6px; border-radius:50%; background:#4CAF50; animation:pulse-dot 1.2s ease-in-out infinite; }
.live-label { font-family:var(--mono); font-size:.55rem; letter-spacing:.1em; color:rgba(76,175,80,.9); font-weight:700; text-transform:uppercase; }
#ss-voice-ui {
  margin-top: 8px; border: 1px solid rgba(76,175,80,.18); border-radius: 3px;
  background: #050505; padding: 12px;
}
#ss-wave-canvas {
  width: 100%; height: 72px; display: block; background: #0a0a0a;
  border-radius: 2px; border: 1px solid #1a1a1a;
}
#ss-voice-status-line {
  font-family: var(--mono); font-size: .56rem; letter-spacing: .1em; text-transform: uppercase;
  color: rgba(76,175,80,.85); margin-top: 8px; min-height: 16px;
}
#ss-voice-status-line.mode-listening { color: rgba(76,175,80,.9); }
#ss-voice-status-line.mode-recording { color: #FF4500; }
#ss-voice-status-line.mode-processing { color: #FF8C00; }
#ss-voice-status-line.mode-speaking { color: #64B5F6; }
#ss-voice-status-line.mode-error { color: #FF4500; }
#ss-playback-audio {
  position: fixed !important; left: -9999px !important; width: 1px !important; height: 1px !important;
  opacity: 0 !important; pointer-events: none !important;
}
#ss-ringtone-out {
  position:fixed !important; bottom:-200px !important; left:-200px !important;
  width:1px !important; height:1px !important; overflow:hidden !important;
  pointer-events:none !important; opacity:0 !important;
}
"""


SSVOICE_CONTROLLER_JS = r"""
(function(){
  console.log('[SSVoice] CONTROLLER_BOOT');
  if (window.__SSVoiceV2) {
    console.log('[SSVoice] CONTROLLER_ALREADY_INSTALLED');
    return;
  }
  window.__SSVoiceV2 = true;
  console.log('[SSVoice] VERSION=2.1.0');
  console.log('[SSVoice] CONTROLLER_READY');

  var CFG = {
    noiseCalibMs: 600,
    speechStartMul: 3.2,
    speechStopMul: 1.8,
    minSpeechMs: 500,
    silenceHangMs: 1800,
    maxUtterMs: 15000,
    minPacketBytes: 800,
    turnCooldownMs: 500,
    bargeInMul: 4.0,
    bargeInMs: 220
  };

  var S = {
    mode: 'IDLE',
    sessionId: '',
    turn: 0,
    active: false,
    stream: null,
    audioCtx: null,
    micSource: null,
    micAnalyser: null,
    playAnalyser: null,
    playSource: null,
    mediaRecorder: null,
    recChunks: [],
    recording: false,
    speechOn: false,
    speechStartAt: 0,
    silenceStartAt: 0,
    noiseFloor: 0.008,
    raf: 0,
    submitting: false,
    playbackTurn: 0,
    bargeMs: 0,
    lastSubmitAt: 0,
    mime: '',
    vadTimer: 0,
    openingDone: false,
    ending: false
  };

  /* ── Idempotent browser ringtone controller ── */
  var Ring = {
    el: null,
    timer: null,
    playing: false,
    ensure: function(){
      if (this.el) return this.el;
      var a = document.getElementById('ss-browser-ringtone');
      if (!a) {
        a = document.createElement('audio');
        a.id = 'ss-browser-ringtone';
        a.loop = true;
        a.preload = 'auto';
        a.style.cssText = 'position:fixed;left:-9999px;width:1px;height:1px;opacity:0;pointer-events:none;';
        document.body.appendChild(a);
      }
      this.el = a;
      return a;
    },
    start: function(dataUri){
      try {
        var a = this.ensure();
        // Use /ssvoice/ringtone URL as primary source, dataUri as fallback
        if (!a.src || a.src === '' || a.src === window.location.href) {
          a.src = '...';
        }
        if (dataUri && dataUri.length > 10) a.src = dataUri;
        if (!a.src) return;
        this.playing = true;
        var self = this;
        a.play().then(function(){ console.log('[SSVoice] RINGTONE_START'); }).catch(function(e){
          console.log('[SSVoice] RINGTONE_PLAY_BLOCKED', e.name);
          // Autoplay blocked — retry on next user gesture
          var resume = function() {
            a.play().catch(function(){});
            document.removeEventListener('click', resume);
            document.removeEventListener('touchstart', resume);
          };
          document.addEventListener('click', resume, {once:true});
          document.addEventListener('touchstart', resume, {once:true});
        });
      } catch(e){ console.log('[SSVoice] RINGTONE_START_FAIL', e); }
    },
    stop: function(){
      try {
        this.playing = false;
        if (this.timer) { clearInterval(this.timer); this.timer = null; }
        var a = this.el || document.getElementById('ss-browser-ringtone');
        if (a) {
          try { a.pause(); } catch(e){}
          try { a.currentTime = 0; } catch(e){}
          try { a.removeAttribute('src'); a.load(); } catch(e){}
        }
        // Also mute any Gradio streaming ringtone audio elements
        try {
          var nodes = document.querySelectorAll('#ss-ringtone-out audio, audio');
          for (var i=0;i<nodes.length;i++){
            var n = nodes[i];
            if (n.id === 'ss-playback-audio') continue;
            // only stop elements that look like ringtone / streaming
            try {
              if (n.id === 'ss-browser-ringtone' || (n.closest && n.closest('#ss-ringtone-out'))) {
                n.pause(); n.currentTime = 0;
              }
            } catch(e){}
          }
          // Force-stop Gradio ringtone component audio inside #ss-ringtone-out
          var root = document.getElementById('ss-ringtone-out');
          if (root) {
            root.querySelectorAll('audio').forEach(function(au){
              try { au.pause(); au.currentTime=0; au.removeAttribute('src'); au.load(); } catch(e){}
            });
          }
        } catch(e){}
        console.log('[SSVoice] RINGTONE_STOP');
      } catch(e){ console.log('[SSVoice] RINGTONE_STOP_FAIL', e); }
    }
  };
  // Canonical global used by overlay Accept/Dismiss/End
  window._ssRingStop = function(){ Ring.stop(); };
  window._ssRingStart = function(uri){ Ring.start(uri || ''); };

  function log(){
    try {
      var a = ['[SSVoice]'].concat([].slice.call(arguments));
      console.log.apply(console, a);
    } catch(e){}
  }

  function setStatus(mode, msg){
    S.mode = mode;
    var el = document.getElementById('ss-voice-status-line');
    if (!el) return;
    el.className = 'mode-' + String(mode||'').toLowerCase();
    el.textContent = msg || mode || '';
    log('STATE', mode, msg || '');
  }

  function ensureCanvas(){
    var host = document.getElementById('ss-voice-ui');
    if (!host) return null;
    var c = document.getElementById('ss-wave-canvas');
    if (!c) {
      c = document.createElement('canvas');
      c.id = 'ss-wave-canvas';
      c.width = 640; c.height = 72;
      host.insertBefore(c, host.firstChild);
    }
    var st = document.getElementById('ss-voice-status-line');
    if (!st) {
      st = document.createElement('div');
      st.id = 'ss-voice-status-line';
      host.appendChild(st);
    }
    return c;
  }

  function rmsFromAnalyser(an){
    if (!an) return 0;
    var buf = new Uint8Array(an.fftSize);
    an.getByteTimeDomainData(buf);
    var sum = 0;
    for (var i=0;i<buf.length;i++){
      var v = (buf[i]-128)/128;
      sum += v*v;
    }
    return Math.sqrt(sum / buf.length);
  }

  function drawWave(){
    var c = ensureCanvas();
    if (!c) { S.raf = requestAnimationFrame(drawWave); return; }
    var ctx = c.getContext('2d');
    var w = c.width, h = c.height;
    ctx.fillStyle = '#0a0a0a';
    ctx.fillRect(0,0,w,h);
    ctx.strokeStyle = '#1a1a1a';
    ctx.beginPath();
    ctx.moveTo(0, h/2); ctx.lineTo(w, h/2); ctx.stroke();

    var an = null;
    var col = '#4CAF50';
    if (S.mode === 'SPEAKING' && S.playAnalyser) { an = S.playAnalyser; col = '#64B5F6'; }
    else if (S.micAnalyser) { an = S.micAnalyser; col = (S.mode==='RECORDING') ? '#FF4500' : (S.mode==='PROCESSING' ? '#FF8C00' : '#4CAF50'); }

    ctx.strokeStyle = col;
    ctx.lineWidth = 2;
    ctx.beginPath();
    if (an) {
      var data = new Uint8Array(an.fftSize);
      an.getByteTimeDomainData(data);
      var step = Math.max(1, Math.floor(data.length / w));
      for (var x=0; x<w; x++){
        var v = data[x*step] || 128;
        var y = (v/255) * h;
        if (x===0) ctx.moveTo(x,y); else ctx.lineTo(x,y);
      }
    } else {
      // baseline ambient
      for (var x2=0; x2<w; x2++){
        var y2 = h/2 + Math.sin(x2/18 + Date.now()/400)*2;
        if (x2===0) ctx.moveTo(x2,y2); else ctx.lineTo(x2,y2);
      }
    }
    ctx.stroke();
    if (S.active) S.raf = requestAnimationFrame(drawWave);
  }

  function pickMime(){
    var cands = [
      'audio/webm;codecs=opus',
      'audio/webm',
      'audio/ogg;codecs=opus',
      'audio/mp4'
    ];
    for (var i=0;i<cands.length;i++){
      try {
        if (window.MediaRecorder && MediaRecorder.isTypeSupported && MediaRecorder.isTypeSupported(cands[i])) {
          return cands[i];
        }
      } catch(e){}
    }
    return '';
  }

  async function unlockAudio(){
    try {
      var AC = window.AudioContext || window.webkitAudioContext;
      if (!S.audioCtx) S.audioCtx = new AC();
      if (S.audioCtx.state === 'suspended') await S.audioCtx.resume();
      // unlock element play
      var a = document.getElementById('ss-playback-audio');
      if (a) {
        a.muted = true;
        try { await a.play(); a.pause(); a.currentTime=0; } catch(e){}
        a.muted = false;
      }
      log('AudioContext unlocked', S.audioCtx.state);
    } catch(e){ log('unlock fail', e); }
  }

  async function startMic(){
    log('mediaDevices:', !!(navigator.mediaDevices && navigator.mediaDevices.getUserMedia));
    if (!navigator.mediaDevices || !navigator.mediaDevices.getUserMedia) {
      log('MIC_API_UNAVAILABLE');
      setStatus('ERROR', 'Microphone API unavailable');
      throw new Error('no mediaDevices');
    }
    // Check permission state if available
    try {
      if (navigator.permissions && navigator.permissions.query) {
        var perm = await navigator.permissions.query({name: 'microphone'});
        log('MIC_PERMISSION_STATE=' + perm.state);
      }
    } catch(pe) { log('MIC_PERMISSION_STATE=unknown'); }
    log('MIC_REQUEST_START');
    S.stream = await navigator.mediaDevices.getUserMedia({
      audio: {
        channelCount: 1,
        echoCancellation: true,
        noiseSuppression: true,
        autoGainControl: true
      },
      video: false
    });
    log('MIC_REQUEST_RESOLVED');
    var tracks = S.stream.getTracks();
    log('MIC_STREAM_OK tracks=' + tracks.length);
    if (tracks.length > 0) {
      log('MIC_TRACK_KIND=' + tracks[0].kind);
      log('MIC_TRACK_STATE=' + tracks[0].state);
    }
    var AC = window.AudioContext || window.webkitAudioContext;
    if (!S.audioCtx) S.audioCtx = new AC();
    if (S.audioCtx.state === 'suspended') await S.audioCtx.resume();
    log('AUDIO_CONTEXT_RUNNING state=' + S.audioCtx.state);
    S.micSource = S.audioCtx.createMediaStreamSource(S.stream);
    S.micAnalyser = S.audioCtx.createAnalyser();
    S.micAnalyser.fftSize = 2048;
    S.micAnalyser.smoothingTimeConstant = 0.8;
    // NEVER connect mic to destination (no feedback)
    S.micSource.connect(S.micAnalyser);
    log('MIC_ANALYSER_READY');
    log('WAVEFORM_READY');
    log('MIC_READY');
    // calibrate noise floor
    var t0 = performance.now();
    var samples = [];
    while (performance.now() - t0 < CFG.noiseCalibMs) {
      samples.push(rmsFromAnalyser(S.micAnalyser));
      await new Promise(function(r){ setTimeout(r, 30); });
    }
    samples.sort(function(a,b){return a-b;});
    var mid = samples[Math.floor(samples.length*0.5)] || 0.008;
    S.noiseFloor = Math.max(0.004, mid);
    log('VAD noiseFloor', S.noiseFloor.toFixed(5));
    S.mime = pickMime();
    log('MediaRecorder mime', S.mime || '(default)');
  }

  function stopMicTracks(){
    try {
      if (S.stream) S.stream.getTracks().forEach(function(t){ try{t.stop();}catch(e){} });
    } catch(e){}
    S.stream = null;
    try { if (S.micSource) S.micSource.disconnect(); } catch(e){}
    S.micSource = null;
    S.micAnalyser = null;
  }

  function startRecorder(){
    if (S.recording || !S.stream) return;
    S.recChunks = [];
    try {
      var opts = S.mime ? { mimeType: S.mime, audioBitsPerSecond: 64000 } : { audioBitsPerSecond: 64000 };
      S.mediaRecorder = new MediaRecorder(S.stream, opts);
    } catch(e) {
      try { S.mediaRecorder = new MediaRecorder(S.stream); } catch(e2) {
        log('MediaRecorder failed', e2);
        return;
      }
    }
    S.mediaRecorder.ondataavailable = function(ev){
      if (ev.data && ev.data.size > 0) S.recChunks.push(ev.data);
    };
    S.mediaRecorder.onstop = function(){
      S.recording = false;
      var blob = new Blob(S.recChunks, { type: (S.mime || 'audio/webm') });
      S.recChunks = [];
      log('RECORDING_STOPPED', 'blob', blob.size);
      if (blob.size >= CFG.minPacketBytes) {
        submitBlob(blob);
      } else {
        log('blob too small, back to LISTENING', blob.size);
        if (S.active) { setStatus('LISTENING', 'Listening…'); }
      }
    };
    S.recording = true;
    S.mediaRecorder.start(100);
    setStatus('RECORDING', 'Recording…');
    log('RECORDING_STARTED');
  }

  function stopRecorder(){
    if (!S.recording || !S.mediaRecorder) return;
    try {
      if (S.mediaRecorder.state === 'recording') S.mediaRecorder.stop();
    } catch(e){ log('stopRecorder', e); }
  }

  function _schedVad(){
    if (S.vadTimer) { try { clearTimeout(S.vadTimer); } catch(e){} }
    S.vadTimer = setTimeout(vadTick, 80);
  }
  function vadTick(){
    if (!S.active || S.ending) return;
    if (S.submitting) { _schedVad(); return; }
    if (!S.openingDone) { _schedVad(); return; }
    if (S.mode === 'PROCESSING') { _schedVad(); return; }

    var lvl = rmsFromAnalyser(S.micAnalyser);
    var startTh = S.noiseFloor * CFG.speechStartMul;
    var stopTh  = S.noiseFloor * CFG.speechStopMul;
    var bargeTh = S.noiseFloor * CFG.bargeInMul;
    var now = performance.now();

    // barge-in while speaking
    if (S.mode === 'SPEAKING') {
      if (lvl >= bargeTh) {
        S.bargeMs += 80;
        if (S.bargeMs >= CFG.bargeInMs) {
          log('BARGE-IN');
          interruptPlayback();
          S.bargeMs = 0;
          S.speechOn = true;
          S.speechStartAt = now;
          S.silenceStartAt = 0;
          startRecorder();
        }
      } else {
        S.bargeMs = 0;
      }
      _schedVad();
      return;
    }

    if (S.mode !== 'LISTENING' && S.mode !== 'RECORDING') {
      _schedVad();
      return;
    }

    if (!S.speechOn) {
      if (lvl >= startTh) {
        if (!S.speechStartAt) S.speechStartAt = now;
        if (now - S.speechStartAt >= 90) {
          S.speechOn = true;
          S.silenceStartAt = 0;
          log('SPEECH_START', 'rms', lvl.toFixed(4));
          startRecorder();
        }
      } else {
        S.speechStartAt = 0;
      }
    } else {
      // speaking / recording
      if (lvl < stopTh) {
        if (!S.silenceStartAt) S.silenceStartAt = now;
        var spoken = now - (S.speechStartAt || now);
        var silent = now - S.silenceStartAt;
        if (spoken >= CFG.minSpeechMs && silent >= CFG.silenceHangMs) {
          log('SPEECH_END', 'spokenMs', Math.round(spoken));
          S.speechOn = false;
          S.speechStartAt = 0;
          S.silenceStartAt = 0;
          stopRecorder();
        }
      } else {
        S.silenceStartAt = 0;
      }
      // max utterance
      if (S.recording && S.speechStartAt && (now - S.speechStartAt) > CFG.maxUtterMs) {
        log('MAX_UTTERANCE');
        S.speechOn = false;
        S.speechStartAt = 0;
        S.silenceStartAt = 0;
        stopRecorder();
      }
    }
    _schedVad();
  }

  function b64ToBlob(b64, mime){
    var bin = atob(b64);
    var len = bin.length;
    var arr = new Uint8Array(len);
    for (var i=0;i<len;i++) arr[i] = bin.charCodeAt(i);
    return new Blob([arr], { type: mime || 'audio/wav' });
  }

  async function postTurn(body, headers){
    var url = (window.location.origin || '') + '/ssvoice/turn';
    var res = await fetch(url, {
      method: 'POST',
      headers: headers || {},
      body: body,
      credentials: 'same-origin'
    });
    var data = null;
    try { data = await res.json(); } catch(e){ data = { ok:false, error:'bad_json', status: res.status }; }
    return { http: res.status, data: data };
  }

  async function submitBlob(blob){
    if (!S.active) return;
    if (S.submitting) { log('submit skipped, busy'); return; }
    var now = performance.now();
    if (now - S.lastSubmitAt < CFG.turnCooldownMs) {
      log('cooldown');
      return;
    }
    S.lastSubmitAt = now;
    S.submitting = true;
    S.turn += 1;
    var turnId = String(S.turn);
    S.playbackTurn = S.turn;
    setStatus('PROCESSING', 'Processing…');
    log('PACKET_CREATED bytes=' + blob.size);
    log('PACKET_SENT turn=' + turnId);

    try {
      var headers = {
        'Content-Type': blob.type || 'application/octet-stream',
        'X-SS-Session': S.sessionId,
        'X-SS-Turn': turnId,
        'X-SS-Opening': '0'
      };
      log('REQUEST_START turn=' + turnId + ' bytes=' + blob.size);
      var out = await postTurn(blob, headers);
      log('REQUEST_RESPONSE status=' + out.http, 'status=', (out.data&&out.data.status), 'rx_ok=', !!(out.data&&out.data.ok));
      // True backend ack only after JSON returns
      if (!out.data) {
        setStatus('LISTENING', 'Listening…');
        return;
      }
      if (out.data.status === 'stale_session') {
        log('stale session ignored');
        return;
      }
      // append transcript to voice box if present
      try {
        if (out.data.transcript || out.data.reply) {
          window.__ssAppendVoiceLine && window.__ssAppendVoiceLine(out.data.transcript, out.data.reply, out.data.action);
        }
      } catch(e){}

      if (out.data.audio_b64 && out.data.turn_id && String(out.data.turn_id) !== String(S.playbackTurn) && String(out.data.turn_id) !== turnId) {
        log('stale audio ignored', out.data.turn_id, S.playbackTurn);
      } else if (out.data.audio_b64) {
        await playAudioB64(out.data.audio_b64, out.data.audio_mime || 'audio/wav', turnId);
      } else {
        if (S.active) setStatus('LISTENING', out.data.error === 'no_speech' ? 'Could not hear speech — listening…' : 'Listening…');
      }
    } catch(e) {
      log('submit error', e);
      if (S.active) setStatus('ERROR', 'Processing error — listening…');
      setTimeout(function(){ if (S.active) setStatus('LISTENING','Listening…'); }, 800);
    } finally {
      S.submitting = false;
    }
  }

  function detachPlayGraph(){
    try { if (S.playSource) S.playSource.disconnect(); } catch(e){}
    S.playSource = null;
    S.playAnalyser = null;
  }

  function interruptPlayback(){
    var a = document.getElementById('ss-playback-audio');
    try {
      if (a) { a.pause(); a.removeAttribute('src'); a.load(); }
    } catch(e){}
    detachPlayGraph();
    S.playbackTurn += 1; // invalidate onended
    log('PLAYBACK_INTERRUPTED');
  }

  async function playAudioB64(b64, mime, turnId){
    if (!S.active) return;
    console.log('[SSVoice] AUDIO_RESPONSE_RECEIVED chars=' + (b64||'').length); log('AUDIO_RESPONSE_RECEIVED');
    var a = document.getElementById('ss-playback-audio');
    if (!a) {
      a = document.createElement('audio');
      a.id = 'ss-playback-audio';
      a.preload = 'auto';
      document.body.appendChild(a);
    }
    interruptPlayback(); // stop any previous
    // restore turn id after interrupt bump
    S.playbackTurn = parseInt(turnId, 10) || S.turn;
    var blob = b64ToBlob(b64, mime);
    log('AUDIO_RESPONSE bytes=' + blob.size);
    var url = URL.createObjectURL(blob);
    a.src = url;
    log('AUDIO_SOURCE_DETECTED');

    // wire analyser for playback waveform (once per element carefully)
    try {
      if (S.audioCtx) {
        detachPlayGraph();
        // MediaElementSource can only be created once per element per context.
        // After End Call + re-accept, __ssMediaSource is deleted by stopSession.
        // If it still exists but belongs to a different (closed) context, recreate.
        if (a.__ssMediaSource && a.__ssMediaSource.context !== S.audioCtx) {
          try { delete a.__ssMediaSource; } catch(ie){}
        }
        if (!a.__ssMediaSource) {
          a.__ssMediaSource = S.audioCtx.createMediaElementSource(a);
        }
        S.playSource = a.__ssMediaSource;
        S.playAnalyser = S.audioCtx.createAnalyser();
        S.playAnalyser.fftSize = 2048;
        S.playSource.connect(S.playAnalyser);
        S.playAnalyser.connect(S.audioCtx.destination);
      }
    } catch(e) {
      log('play graph note', e);
      // fallback: just play without custom graph — clear stale reference
      try { delete a.__ssMediaSource; } catch(ie){}
    }

    var myTurn = S.playbackTurn;
    a.onplay = function(){
      if (myTurn !== S.playbackTurn) return;
      console.log('[SSVoice] PLAYBACK_START'); log('PLAYBACK_START');
      setStatus('SPEAKING', 'Sherl0ck speaking…');
    };
    a.onended = function(){
      if (myTurn !== S.playbackTurn) return;
      console.log('[SSVoice] PLAYBACK_END'); log('PLAYBACK_END');
      try { URL.revokeObjectURL(url); } catch(e){}
      if (S.active) setStatus('LISTENING', 'Listening…');
    };
    a.onerror = function(){
      log('PLAYBACK_ERROR');
      if (S.active) setStatus('LISTENING', 'Listening…');
    };

    try {
      console.log('[SSVoice] PLAY_REQUEST'); log('PLAY_REQUEST');
      await a.play();
    } catch(e) {
      log('play() blocked', e);
      // try once more after resume
      try {
        if (S.audioCtx && S.audioCtx.state === 'suspended') await S.audioCtx.resume();
        await a.play();
      } catch(e2) {
        log('play failed', e2);
        setStatus('ERROR', 'Audio playback unavailable');
        setTimeout(function(){ if (S.active) setStatus('LISTENING','Listening…'); }, 900);
      }
    }
  }

  async function requestOpening(){
    if (S.openingDone) return;
    S.submitting = true;
    S.turn += 1;
    var turnId = String(S.turn);
    S.playbackTurn = S.turn;
    log('OPENING_REQUEST_START'); setStatus('PROCESSING', 'Connecting Sherl0ck…');
    try {
      var headers = {
        'X-SS-Session': S.sessionId,
        'X-SS-Turn': turnId,
        'X-SS-Opening': '1'
      };
      var out = await postTurn(new Uint8Array(0), headers);
      log('OPENING server', out.http, out.data && out.data.status);
      if (out.data && (out.data.reply)) {
        window.__ssAppendVoiceLine && window.__ssAppendVoiceLine('', out.data.reply, null);
      }
      if (out.data && out.data.audio_b64) {
        await playAudioB64(out.data.audio_b64, out.data.audio_mime || 'audio/wav', turnId);
      } else if (S.active) {
        setStatus('LISTENING', 'Listening…');
      }
    } catch(e) {
      log('opening fail', e);
      if (S.active) setStatus('LISTENING', 'Listening…');
    } finally {
      S.submitting = false;
    }
  }

  async function startSession(sessionId){
    if (S.ending) return;
    if (S.active && S.sessionId && sessionId && S.sessionId === sessionId) {
      log('startSession noop already active', sessionId);
      return;
    }
    if (S.active) await stopSession(true);
    log('START_SESSION', sessionId);
    // Always stop ringtone first (browser + gradio stream)
    try { window._ssRingStop && window._ssRingStop(); } catch(e){}
    S.sessionId = sessionId || (Date.now().toString(36) + Math.random().toString(36).slice(2,8));
    S.turn = 0;
    S.active = true;
    S.ending = false;
    S.speechOn = false;
    S.submitting = false;
    S.openingDone = false;
    S.bargeMs = 0;
    if (S.vadTimer) { try { clearTimeout(S.vadTimer); } catch(e){} S.vadTimer = 0; }
    ensureCanvas();
    setStatus('ACCEPTING', 'Connecting microphone…');
    await unlockAudio();
    // Mic may already be primed by gesture-time acceptPrepare
    if (!S.stream) {
      await startMic();
    } else {
      log('MIC_REUSED_FROM_GESTURE');
    }
    log('MIC_READY');
    log('VAD_READY');
    setStatus('LISTENING', 'Listening…');
    log('LISTENING');
    if (S.raf) cancelAnimationFrame(S.raf);
    S.raf = requestAnimationFrame(drawWave);
    vadTick();
    // Opening must NOT block the Accept gesture / Gradio js= return.
    // Fire-and-forget; openingDone gates VAD speech capture until complete.
    Promise.resolve()
      .then(function(){ return requestOpening(); })
      .then(function(){ S.openingDone = true; log('OPENING_DONE'); })
      .catch(function(e){
        console.log('[SSVoice] OPENING_FAIL', e);
        S.openingDone = true; // still allow listening after failure
        setStatus('LISTENING', 'Listening…');
      });
  }

  /* Prime mic during the REAL user gesture (before Gradio round-trip). */
  async function acceptPrepare(){
    try { window._ssRingStop && window._ssRingStop(); } catch(e){}
    setStatus('ACCEPTING', 'Connecting microphone…');
    try {
      await unlockAudio();
      if (!S.stream) await startMic();
      ensureCanvas();
      if (S.raf) cancelAnimationFrame(S.raf);
      S.raf = requestAnimationFrame(drawWave);
      log('ACCEPT_PREPARE_OK');
      return true;
    } catch(e) {
      log('ACCEPT_PREPARE_FAIL', e);
      log('MIC_REQUEST_REJECTED name=' + (e.name||'unknown') + ' message=' + (e.message||'unknown'));
      setStatus('ERROR', 'Microphone unavailable');
      return false;
    }
  }

  async function stopSession(silent){
    log('END');
    S.ending = true;
    S.active = false;
    S.speechOn = false;
    S.submitting = false;
    S.openingDone = false;
    try { window._ssRingStop && window._ssRingStop(); } catch(e){}
    try { stopRecorder(); } catch(e){}
    interruptPlayback();
    stopMicTracks();
    if (S.vadTimer) { try { clearTimeout(S.vadTimer); } catch(e){} S.vadTimer = 0; }
    if (S.raf) { try { cancelAnimationFrame(S.raf); } catch(e){} S.raf=0; }
    try {
      fetch((window.location.origin||'') + '/ssvoice/end', {
        method:'POST',
        headers: { 'X-SS-Session': S.sessionId || '' },
        credentials:'same-origin'
      }).catch(function(){});
    } catch(e){}
    // Close AudioContext fully on end to release mic hardware association
    try {
      if (S.audioCtx) {
        var ctx = S.audioCtx;
        S.audioCtx = null;
        S.playSource = null;
        S.playAnalyser = null;
        S.micSource = null;
        S.micAnalyser = null;
        // Clear cached MediaElementSource marker so next call can recreate
        var a = document.getElementById('ss-playback-audio');
        if (a) { try { delete a.__ssMediaSource; } catch(e){} }
        try { await ctx.close(); } catch(e){}
      }
    } catch(e){}
    setStatus('ENDED', 'Call ended');
    S.sessionId = '';
    S.ending = false;
    // Update button text back to 'Start Call'
    try {
      var btns = document.querySelectorAll('button');
      for (var k = 0; k < btns.length; k++) {
        var tx = ((btns[k].textContent||'')+(btns[k].innerText||'')).replace(/[\s\u00a0]+/g,' ').trim().toLowerCase();
        if (tx.indexOf('accept')!==-1 && tx.indexOf('call')!==-1) {
          btns[k].textContent = 'Start Call';
          break;
        }
      }
    } catch(e){}
  }

  // Public API used by Gradio button callbacks via injected JS
  /* Single authoritative Accept path: gesture → mic → /ssvoice/accept → start(session) */
  var _acceptInFlight = false;
  async function acceptFromUserGesture(source){
    source = source || 'unknown';
    console.log('[SSVoice] ACCEPT_CLICK');
    console.log('[SSVoice] SSVoice_AVAILABLE=' + (!!window.SSVoice));
    console.log('[SSVoice] ACCEPT_FUNCTION_AVAILABLE=' + (!!window.SSVoice && typeof window.SSVoice.accept === 'function'));
    console.log('[SSVoice] NATIVE_ACCEPT_CLICK source=' + source);
    if (_acceptInFlight) { console.log('[SSVoice] ACCEPT_IN_FLIGHT ignore'); return S.sessionId || ''; }
    if (S.active && S.sessionId) {
      console.log('[SSVoice] ALREADY_ACTIVE session=' + S.sessionId);
      return S.sessionId;
    }
    _acceptInFlight = true;
    try {
      console.log('[SSVoice] USER_GESTURE_START');
      console.log('[SSVoice] RING_STOP_REQUESTED');
      try { if (window._ssRingStop) window._ssRingStop(); } catch(e){}
      console.log('[SSVoice] MIC_REQUEST_START');
      var prepOk = await acceptPrepare();
      if (!prepOk) {
        console.log('[SSVoice] MIC_REQUEST_FAILED');
        return '';
      }
      console.log('[SSVoice] MIC_REQUEST_RESOLVED');
      console.log('[SSVoice] AUDIO_CONTEXT_STATE=' + (S.audioCtx ? S.audioCtx.state : 'null'));
      console.log('[SSVoice] MIC_STREAM_TRACKS=' + (S.stream ? S.stream.getTracks().length : 0));
      // Health check — prove share tunnel reaches FastAPI
      try {
        console.log('[SSVoice] HEALTH_START');
        var hr = await fetch((window.location.origin||'') + '/ssvoice/health', {credentials:'same-origin'});
        console.log('[SSVoice] HEALTH_STATUS=' + hr.status);
        var hj = await hr.json().catch(function(){return {};});
        console.log('[SSVoice] HEALTH_OK', JSON.stringify(hj));
      } catch(he) {
        console.log('[SSVoice] HEALTH_FAIL', he);
      }
      // Mint/fetch authoritative session from server
      console.log('[SSVoice] ACCEPT_HTTP_START');
      var ar = await fetch((window.location.origin||'') + '/ssvoice/accept', {
        method: 'POST',
        credentials: 'same-origin',
        headers: {'Content-Type': 'application/json'},
        body: '{}'
      });
      console.log('[SSVoice] ACCEPT_HTTP_STATUS=' + ar.status);
      var aj = await ar.json();
      if (!aj || !aj.ok || !aj.session_id) {
        console.log('[SSVoice] ACCEPT_HTTP_FAIL', JSON.stringify(aj));
        setStatus('ERROR', 'Session start failed');
        return '';
      }
      console.log('[SSVoice] SESSION_ID=' + aj.session_id);
      await startSession(aj.session_id);
      console.log('[SSVoice] SESSION_ACTIVE=true');
      return aj.session_id;
    } catch(e) {
      console.log('[SSVoice] ACCEPT_PATH_FAIL', e);
      setStatus('ERROR', 'Accept failed');
      return '';
    } finally {
      _acceptInFlight = false;
    }
  }

  window.SSVoice = {
    start: startSession,
    stop: stopSession,
    prepare: acceptPrepare,
    accept: acceptFromUserGesture,
    isActive: function(){ return !!S.active; },
    state: function(){ return { mode:S.mode, session:S.sessionId, turn:S.turn }; }
  };

  // Helper: append lines into Gradio voice transcript textbox if present
  window.__ssAppendVoiceLine = function(userT, botT, action){
    try {
      var box = document.querySelector('#voice-conv-box textarea, #voice-conv-box input, textarea[data-testid="textbox"]');
      // more robust: find by elem id wrapper
      var root = document.getElementById('voice-conv-box');
      var ta = root ? root.querySelector('textarea, input') : null;
      if (!ta) {
        // fallback any large voice box
        var all = document.querySelectorAll('textarea');
        for (var i=0;i<all.length;i++){
          var ph = (all[i].placeholder||'');
          if (ph.toLowerCase().indexOf('voice') !== -1 || ph.toLowerCase().indexOf('accept the call') !== -1) {
            ta = all[i]; break;
          }
        }
      }
      if (!ta) return;
      var cur = ta.value || '';
      if (userT) cur += '[You]      ' + userT + '\n';
      if (botT)  cur += '[Sherl0ck] ' + botT + '\n';
      if (action === 'report_sent') cur += '[System]   Report dispatched to all channels\n';
      var setter = Object.getOwnPropertyDescriptor(window.HTMLTextAreaElement.prototype, 'value').set;
      setter.call(ta, cur);
      ta.dispatchEvent(new Event('input', { bubbles:true }));
      ta.scrollTop = ta.scrollHeight;
    } catch(e){ log('append line fail', e); }
  };

  log('CONTROLLER_READY');
  log('VERSION=2.1.0');

  /* ── Overlay card creation (replaces _global_call_js <script>) ── */
  function _buildOverlayCard(level, riskPct, mode) {
    mode = mode || 'call';
    var col = (level === 'CRITICAL') ? '#FF4500' : '#FF8C00';
    var card = document.createElement('div');
    card.id = 'ss-call-card';
    if (mode === 'status') {
      // Status-only card: shows CRITICAL level, no call buttons
      card.innerHTML = '<div class="cc-scan"></div>'
        +'<div class="cc-header"><span class="cc-header-label">Alert Status</span><span class="cc-header-dot"></span></div>'
        +'<div class="cc-body">'
        +'<div class="cc-phone-icon stopped"><svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="'+col+'" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"><path d="M10.29 3.86L1.82 18a2 2 0 001.71 3h16.94a2 2 0 001.71-3L13.71 3.86a2 2 0 00-3.42 0z"/><line x1="12" y1="9" x2="12" y2="13"/><line x1="12" y1="17" x2="12.01" y2="17"/></svg></div>'
        +'<div style="flex:1;"><div class="cc-info-title">Sherl0ck Alert</div>'
        +'<div class="cc-info-sub">Risk Score: '+riskPct+'%</div>'
        +'<div class="cc-risk-badge" style="border-color:'+col+';color:'+col+';">'+level+' \u00b7 '+riskPct+'%</div></div>'
        +'</div>'
        +'<div class="cc-actions"><button class="cc-btn-dismiss" onclick="window._ssDismissCall()">Dismiss</button></div>'
        +'<div class="cc-progress"><div class="cc-progress-fill"></div></div>';
    } else {
      // Full call card with Accept/Dismiss
      card.innerHTML = '<div class="cc-scan"></div>'
        +'<div class="cc-header"><span class="cc-header-label">Incoming Alert Call</span><span class="cc-header-dot"></span></div>'
        +'<div class="cc-body" onmouseenter="try{window._ssRingStart&&window._ssRingStart()}catch(e){}" ontouchstart="try{window._ssRingStart&&window._ssRingStart()}catch(e){}">'
        +'<div class="cc-phone-icon"><svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="'+col+'" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"><path d="M22 16.92v3a2 2 0 01-2.18 2 19.79 19.79 0 01-8.63-3.07 19.5 19.5 0 01-6-6 19.79 19.79 0 01-3.07-8.67A2 2 0 014.11 2h3a2 2 0 012 1.72c.127.96.361 1.903.7 2.81a2 2 0 01-.45 2.11L8.09 9.91a16 16 0 006 6l1.27-1.27a2 2 0 012.11-.45c.907.339 1.85.573 2.81.7A2 2 0 0122 16.92z"/></svg></div>'
        +'<div style="flex:1;"><div class="cc-info-title">Sherl0ck Agent Calling</div>'
        +'<div class="cc-info-sub">Risk Score: '+riskPct+'%</div>'
        +'<div class="cc-risk-badge" style="border-color:'+col+';color:'+col+';">'+level+' \u00b7 '+riskPct+'%</div></div>'
        +'</div>'
        +'<div class="cc-actions"><button class="cc-btn-accept" onclick="window._ssHandleAccept()">Accept</button>'
        +'<button class="cc-btn-dismiss" onclick="window._ssDismissCall()">Dismiss</button></div>'
        +'<div class="cc-progress"><div class="cc-progress-fill"></div></div>';
    }
    return card;
  }

  window._showOverlay = function _showOverlay(level, riskPct, mode) {
    if (window._ssCallDismissed) return;
    if (document.getElementById('ss-call-card')) return;
    var root = document.getElementById('ss-overlay-root');
    if (!root) { root = document.createElement('div'); root.id = 'ss-overlay-root'; document.body.appendChild(root); }
    root.appendChild(_buildOverlayCard(level, riskPct, mode));
    log('OVERLAY_SHOWN', level, riskPct);
    try {
      var btns2 = document.querySelectorAll('button');
      for (var k2 = 0; k2 < btns2.length; k2++) {
        var tx2 = ((btns2[k2].textContent||'')+(btns2[k2].innerText||'')).replace(/[\s\u00a0]+/g,' ').trim().toLowerCase();
        if ((tx2.indexOf('accept')!==-1 || tx2.indexOf('start')!==-1) && tx2.indexOf('call')!==-1) {
          btns2[k2].textContent = 'Accept Call';
          break;
        }
      }
    } catch(e){}
  }

  function _hideOverlay() {
    var ph = document.querySelector('#ss-call-card .cc-phone-icon');
    if (ph) ph.classList.add('stopped');
    var c = document.getElementById('ss-call-card');
    if (c) c.remove();
    var r = document.getElementById('ss-overlay-root');
    if (r) r.remove();
  }

  window._ssHandleAccept = function() {
    console.log('[SSVoice] OVERLAY_ACCEPT_CLICK');
    window._ssCallDismissed = true;
    try { if (window._ssRingStop) window._ssRingStop(); } catch(e){}
    // Stop Gradio ringtone audio
    try {
      var ringAudios = document.querySelectorAll('#ss-ringtone-out audio, .ringtone audio');
      ringAudios.forEach(function(a){ try { a.pause(); a.currentTime=0; a.src=''; } catch(e){} });
    } catch(e){}
    try {
      if (window.SSVoice && window.SSVoice.accept) {
        console.log('[SSVoice] FORWARDING_TO_VOICE_ACCEPT');
        window.SSVoice.accept('overlay');
      }
    } catch(e) { console.log('[SSVoice] OVERLAY_ACCEPT_FAIL', e); }
    _hideOverlay();
    var trig = document.getElementById('ss-alert-trigger');
    if (trig) { trig.setAttribute('data-action', 'none'); trig.remove(); }
    // Navigate to agent tab and click Gradio Accept for UI sync
    try {
      var tabs = document.querySelectorAll('.tab-nav button, button[role="tab"]');
      if (tabs.length > 2) tabs[2].click();
      setTimeout(function() {
        var btns = document.querySelectorAll('button');
        for (var j = 0; j < btns.length; j++) {
          var tx = ((btns[j].textContent||'')+(btns[j].innerText||'')).replace(/[\s\u00a0]+/g,' ').trim().toLowerCase();
          if (tx.indexOf('accept')!==-1 && tx.indexOf('call')!==-1 && btns[j].offsetParent!==null) {
            console.log('[SSVoice] UI_ACCEPT_CLICK_AFTER_VOICE');
            btns[j].click(); return;
          }
        }
      }, 350);
    } catch(e) {}
  };

  window._ssDismissCall = function() {
    console.log('[SSVoice] OVERLAY_DISMISS');
    window._ssCallDismissed = true;
    try { if (window._ssRingStop) window._ssRingStop(); } catch(e){}
    _hideOverlay();
    var trig2 = document.getElementById('ss-alert-trigger');
    if (trig2) { trig2.setAttribute('data-action', 'none'); trig2.remove(); }
  };

  /* ── Watch for alert trigger from Python (data-attribute approach) ── */
  function _watchAlertTrigger() {
    var observer = new MutationObserver(function(mutations) {
      var el = document.getElementById('ss-alert-trigger');
      if (!el) return;
      var action = el.getAttribute('data-action');
      if (action === 'ring') {
        var level = el.getAttribute('data-level') || 'HIGH';
        var risk = el.getAttribute('data-risk') || '0';
        var tune = el.getAttribute('data-tune') || '';
        console.log('[SSVoice] RING_TRIGGER level=' + level + ' risk=' + risk);
        if (!window._ssCallDismissed) {
          _showOverlay(level, risk);
          // Try to play ringtone — will succeed if user has interacted with page
          Ring.start(tune);
        }
      } else if (action === 'stop') {
        console.log('[SSVoice] STOP_TRIGGER');
        Ring.stop();
        _hideOverlay();
        window._ssCallDismissed = false;
      }
    });
    // Watch the document for the trigger element appearing or changing
    observer.observe(document.body, { childList: true, subtree: true, attributes: true, attributeFilter: ['data-action'] });
    // Also check immediately in case it already exists
    var existing = document.getElementById('ss-alert-trigger');
    if (existing) {
      var action = existing.getAttribute('data-action');
      if (action === 'ring') {
        Ring.start(existing.getAttribute('data-tune') || '');
        _showOverlay(existing.getAttribute('data-level') || 'HIGH', existing.getAttribute('data-risk') || '0');
      }
    }
    log('ALERT_TRIGGER_WATCHER_ACTIVE');
  }

  // Global ring start function (called from overlay hover/touch)
  window._ssRingStart = function() {
    try {
      var a = Ring.ensure();
      if (a && a.paused && Ring.playing) {
        a.play().catch(function(){});
      }
    } catch(e) {}
  };

  // Start watching after DOM is ready
  if (document.readyState === 'loading') {
    document.addEventListener('DOMContentLoaded', _watchAlertTrigger);
  } else {
    _watchAlertTrigger();
  }

})();
"""





# ══════════════════════════════════════════════════════════════════════════════
# GLOBAL CALL OVERLAY JS + HTML HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def _global_call_js(level: str, risk: float, tune_uri: str) -> str:
    """Return HTML with data attributes. The controller JS (installed via demo.load)
    watches #ss-alert-trigger via MutationObserver and starts ringtone/overlay."""
    show = level in ("CRITICAL", "HIGH")
    risk_pct = round(risk * 100)
    if show:
        return (f'<div id="ss-alert-trigger" data-action="ring" '
                f'data-level="{level}" data-risk="{risk_pct}" '
                f'data-tune="{tune_uri[:200]}" style="display:none;"></div>')
    return '<div id="ss-alert-trigger" data-action="stop" style="display:none;"></div>'
def _stop_call_js() -> str:
    """Return HTML with stop action. Controller watches via MutationObserver."""
    return '<div id="ss-alert-trigger" data-action="stop" style="display:none;"></div>'


def _agent_sec(text: str, icon: str = "") -> str:
    ic = (f"<span style='flex-shrink:0;'>"
          f"{icon.replace('stroke=\"currentColor\"','stroke=\"rgba(255,140,0,.6)\"')}</span>") if icon else ""
    return f"<div class='agent-sec-label'>{ic}{text}</div>"


def _call_strip_html(level: str, risk: float, visible: bool) -> str:
    if not visible or level == "LOW": return ""
    col      = "#FF4500" if level == "CRITICAL" else "#FF8C00"
    label    = "CRITICAL ANOMALY DETECTED" if level == "CRITICAL" else "HIGH RISK ANOMALY"
    risk_pct = round(risk * 100)
    return (
        f"<div class='agent-call-strip' "
        f"onclick='if(window._ssGoToAgentTab){{window._ssGoToAgentTab();}}else{{"
        f"var t=document.querySelectorAll(\"button[role=tab],.tab-nav button\");"
        f"if(t.length>2)t[2].click();}}'>"
        f"<div class='strip-phone'>"
        f"<svg width='26' height='26' viewBox='0 0 24 24' fill='none' stroke='{col}' "
        f"stroke-width='1.8' stroke-linecap='round' stroke-linejoin='round'>"
        f"<path d='M22 16.92v3a2 2 0 01-2.18 2 19.79 19.79 0 01-8.63-3.07 19.5 19.5 0 "
        f"01-6-6 19.79 19.79 0 01-3.07-8.67A2 2 0 014.11 2h3a2 2 0 012 1.72c.127.96.361 "
        f"1.903.7 2.81a2 2 0 01-.45 2.11L8.09 9.91a16 16 0 006 6l1.27-1.27a2 2 0 "
        f"012.11-.45c.907.339 1.85.573 2.81.7A2 2 0 0122 16.92z'/></svg>"
        f"</div>"
        f"<div style='flex:1;'>"
        f"<div style='font-family:\"Roboto Mono\",monospace;font-size:.52rem;letter-spacing:.18em;"
        f"text-transform:uppercase;color:{col};font-weight:700;'>{label}</div>"
        f"<div style='font-family:Charter,Georgia,serif;font-size:.82rem;"
        f"color:rgba(255,255,255,.45);margin-top:4px;'>"
        f"Incoming call &middot; Risk: "
        f"<strong style='color:#fff;font-family:\"Times New Roman\",Times,serif;'>{risk_pct}%</strong>"
        f" &middot; <em style='color:{col};font-style:normal;'>Click to open Agent tab</em>"
        f"</div></div>"
        f"<div style='font-family:\"Roboto Mono\",monospace;font-size:.55rem;font-weight:700;"
        f"letter-spacing:.1em;text-transform:uppercase;color:{col};"
        f"border:1px solid {col};border-radius:3px;padding:6px 12px;flex-shrink:0;'>"
        f"Open Agent &rarr;"
        f"</div></div>"
    )


def _alert_status_html(level: str, risk: float) -> str:
    col      = {"CRITICAL": "#FF4500", "HIGH": "#FF8C00", "LOW": "#4CAF50"}.get(level, "#888")
    label    = {"CRITICAL": "CRITICAL ANOMALY", "HIGH": "HIGH RISK", "LOW": "NO ALERT"}.get(level, level)
    card_cls = {"CRITICAL": "critical", "HIGH": "high", "LOW": ""}.get(level, "")
    risk_pct = round(risk * 100)
    icon_svg = (
        f'<svg width="22" height="22" viewBox="0 0 24 24" fill="none" '
        f'stroke="{col}" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round">'
        + (
            '<path d="M10.29 3.86L1.82 18a2 2 0 001.71 3h16.94a2 2 0 001.71-3L13.71 3.86a2 2 0 00-3.42 0z"/>'
            '<line x1="12" y1="9" x2="12" y2="13"/><line x1="12" y1="17" x2="12.01" y2="17"/>'
            if level != "LOW" else
            '<path d="M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10z"/>'
        ) + '</svg>'
    )
    def _chip(lbl, ok):
        c, col2 = ("#4CAF50", "rgba(76,175,80,.9)") if ok else ("#444", "rgba(255,255,255,.2)")
        return (f'<span class="agent-dep-chip" style="color:{col2};">'
                f'<span style="width:5px;height:5px;border-radius:50%;background:{c};display:inline-block;"></span>'
                f'{lbl}</span>')
    sanim = ' style="animation:status-flash 2s ease-in-out infinite;"' if level == "CRITICAL" else ""
    chips = (
        _chip("LFM2.5-Audio", LFM_AUDIO_AVAILABLE) + _chip("Cohere", COHERE_AVAILABLE) +
        _chip("cmd-a-03-2025", COHERE_AVAILABLE) + _chip("HTTP /ssvoice", True)
    )
    return (
        f"<div class='agent-status-card {card_cls}'>"
        f"<div style='flex:1;'>"
        f"<div style='font-family:\"Roboto Mono\",monospace;font-size:.5rem;letter-spacing:.14em;"
        f"text-transform:uppercase;color:rgba(255,255,255,.28);margin-bottom:6px;'>Alert Status</div>"
        f"<div style='display:flex;align-items:center;gap:10px;'>{icon_svg}"
        f"<div style='font-family:Charter,Georgia,serif;font-size:1.2rem;font-weight:700;"
        f"color:{col};'{sanim}>{label}</div></div>"
        f"<div class='agent-dep-chips'>{chips}</div>"
        f"</div>"
        f"<div style='font-size:2.4rem;font-weight:700;color:{col};"
        f"font-family:\"Times New Roman\",Times,serif;letter-spacing:-.02em;"
        f"{('animation:status-flash 2s ease-in-out infinite;' if level=='CRITICAL' else '')}'>"
        f"{risk_pct}%</div></div>"
    )


def _clean_llm_markdown(text: str) -> str:
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    cleaned = []
    for line in text.split('\n'):
        m = re.match(r'^#{1,6}\s+(.+)$', line.strip())
        if m: cleaned.append(f'**{m.group(1)}**'); continue
        s = re.sub(r'^[\s]*[-\*]\s+(?!\*)', '', line)
        if s != line: cleaned.append(s); continue
        s2 = re.sub(r'^[\s]*[\u2022\u00b7]\s*', '', line)
        if s2 != line: cleaned.append(s2); continue
        cleaned.append(line)
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(cleaned)).strip()


_SHERL0CK_LABEL_HTML = '<span class="brand-name">Sherl<span class="brand-zero">0</span>ck</span>'

_CHAT_SCROLL_OBSERVER_JS = """
<script>
(function() {
  var _lastScrollHeight = -1;
  function _tick() {
    var box = document.getElementById('ss-chat-box');
    if (box && box.scrollHeight !== _lastScrollHeight) {
      box.scrollTop = box.scrollHeight;
      _lastScrollHeight = box.scrollHeight;
    }
    requestAnimationFrame(_tick);
  }
  if (document.readyState === 'loading') document.addEventListener('DOMContentLoaded', function() { requestAnimationFrame(_tick); });
  else requestAnimationFrame(_tick);
})();
</script>
"""


def _render_bubble_content(raw: str) -> str:
    raw = raw.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    raw = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', raw)
    raw = re.sub(r'(\d+\.?\d*%?)', r"<span style='font-family:\"Times New Roman\",Times,serif;'>\1</span>", raw)
    parts = []
    for para in re.split(r'\n\n+', raw):
        para = para.strip()
        if not para: continue
        lines = [l.strip() for l in para.split('\n') if l.strip()]
        if lines and all(re.match(r'^\d+[\.\)]\s+', l) for l in lines):
            lh = ''
            for l in lines:
                m2 = re.match(r'^(\d+)[\.\)]\s+(.+)$', l)
                if m2:
                    lh += (f'<div class="chat-li"><span class="li-num">{m2.group(1)}.</span>'
                           f'<span>{m2.group(2)}</span></div>')
            parts.append(lh)
        else:
            parts.append(f'<div class="chat-para">{para.replace(chr(10),"<br/>")}</div>')
    return ''.join(parts) or f'<div class="chat-para">{raw}</div>'


def _render_chat_html(history: list) -> str:
    if not history:
        return ('<div class="chat-display" id="ss-chat-box">'
                '<div class="msg-empty">Run an analysis then click <strong>Start Chat</strong>.</div></div>')
    parts = ['<div class="chat-display" id="ss-chat-box">']
    for m in history:
        role, content = m.get("role", "user"), m.get("content", "")
        if role == "assistant":
            rendered = _render_bubble_content(_clean_llm_markdown(content))
            parts.append(
                '<div class="msg-agent">'
                f'<div class="msg-lbl"><span class="lbl-dot"></span>{_SHERL0CK_LABEL_HTML}</div>'
                f'<div class="msg-bubble">{rendered}</div></div>'
            )
        elif role == "user":
            safe = content.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;").replace("\n","<br/>")
            parts.append(
                '<div class="msg-you"><div class="msg-lbl">You<span class="lbl-dot"></span></div>'
                f'<div class="msg-bubble">{safe}</div></div>'
            )
        else:
            safe = content.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
            parts.append(f'<div class="msg-system">{safe}</div>')
    parts.append('</div>')
    return ''.join(parts)


def process_video_with_agent(video):
    empty = (no_result_html(), None, None, None,
             _alert_status_html("LOW", 0.0), "", "", gr.update(visible=False))
    if video is None:
        return empty
    try:
        raw = run_pipeline(video)
    except Exception:
        tb = traceback.format_exc()
        print(f"[AgentLayer] Pipeline error:\n{tb}")
        return (f"<div style='color:#FF4500;font-family:\"Roboto Mono\",monospace;"
                f"font-size:.8rem;padding:20px;background:#050505;'>{tb}</div>",
                None, None, None, _alert_status_html("LOW", 0.0), "", "", gr.update(visible=False))

    r = raw["risk_score"]
    report_html = build_report_html(raw["report"], r, raw["risk_label"],
                                    raw["status"], graph_path=raw.get("graph_path"))
    from PIL import Image as _PIL
    fi     = _PIL.fromarray(raw["top_frame"]) if raw.get("top_frame") is not None else None
    jd_str = json.dumps(raw["json_data"], indent=2, default=str)
    pdf_out = raw.get("pdf_path")

    payload = build_alert_payload(raw)
    level   = determine_alert_level(r)
    _AGENT_STATE.update({"payload": payload, "level": level, "voice_live": False, "text_live": False})
    print(f"[AgentLayer] State updated: level={level} risk={r:.3f}")

    _dispatch_log = ""
    if level == "CRITICAL":
        print("[AgentLayer] CRITICAL - auto-dispatching.")
        _results = dispatch_all_channels(payload, level)
        _ts = datetime.now(timezone.utc).strftime("%H:%M UTC")
        _lines = [f"Auto-Dispatch  [{level}]  {_ts}"]
        _lines += [f"  {ch.capitalize():<12s}  {'OK' if ok else 'FAIL'}" for ch, ok in _results.items()]
        _dispatch_log = "\n".join(_lines)

    tune        = _caller_tune_uri()
    status_html = _alert_status_html(level, r)
    strip_html  = _call_strip_html(level, r, visible=(level in ("CRITICAL", "HIGH")))
    banner_js   = _global_call_js(level, r, tune)

    if level in ("CRITICAL", "HIGH"):
        def _prewarm():
            try:
                _load_lfm_audio()
            except Exception:
                pass
        _tts_executor.submit(_prewarm)
    return (report_html, fi, jd_str, pdf_out,
            status_html, strip_html, banner_js,
            gr.update(visible=(level in ("CRITICAL", "HIGH"))),
            _dispatch_log)


def process_video_with_agent_for_agent_tab(video):
    result = list(process_video_with_agent(video))
    result[5] = ""
    return tuple(result)


def accept_call():
    """Server-side Accept: mint/reuse session, stop ring, show voice UI.

    CRITICAL: Browser microphone + SSVoice.start MUST already have been started
    from the trusted user gesture via SSVoice.accept() (Gradio js= / overlay).
    Do NOT inject <script>SSVoice.start()</script> here — Gradio HTML updates
    often do not re-execute scripts, which was the live failure mode.
    """
    global _voice_active_payload
    payload = _AGENT_STATE.get("payload", {})
    level   = _AGENT_STATE.get("level", "LOW")
    # Prefer session already minted by browser POST /ssvoice/accept
    sid = _AGENT_STATE.get("voice_session_id") or ""
    if not sid:
        sid = uuid.uuid4().hex
        _AGENT_STATE["voice_session_id"] = sid
        _AGENT_STATE["voice_turn"] = 0
    _AGENT_STATE["voice_live"] = True
    _AGENT_STATE["ring_stopped"] = True
    _voice_active_payload = payload
    print(f"[AgentLayer] accept_call session={sid} level={level} "
          f"(browser should already own this session)")
    # UI only — no boot script that starts voice
    return (
        "Call connected. Speak naturally — Sherl0ck is listening.\n",
        gr.update(visible=False, interactive=False, value="Accept Call"),
        gr.update(visible=True),
        gr.update(visible=True),
        _stop_call_js(),
        None,  # stop ringtone (gr.Audio needs None, not empty string)
    )


def end_call(conv: str):
    _AGENT_STATE["voice_live"] = False
    sid = _AGENT_STATE.get("voice_session_id", "")
    _AGENT_STATE["voice_session_id"] = ""
    _AGENT_STATE["ring_stopped"] = True
    level = _AGENT_STATE.get("level", "LOW")
    risk = float(_AGENT_STATE.get("payload", {}).get("risk", 0.0))
    risk_pct = int(risk * 100)
    bye = "Sherlock signing off  Investigation session ended  Stay vigilant and safe"
    updated = (conv or "") + f"[Sherl0ck]  {bye}\n"
    end_js = f"""
<script>
(function(){{
  try {{ if (window._ssRingStop) window._ssRingStop(); }} catch(e){{}}
  try {{ if (window.SSVoice) window.SSVoice.stop(); }} catch(e){{}}
  try {{
    window._ssCallDismissed = false;
    var old = document.getElementById("ss-call-card");
    if (old) old.remove();
    var oldRoot = document.getElementById("ss-overlay-root");
    if (oldRoot) oldRoot.remove();
    if (window._showOverlay) {{
      window._showOverlay("{level}", "{risk_pct}", "status");
    }}
  }} catch(e) {{}}
  try {{
    var btns = document.querySelectorAll("button");
    for (var k = 0; k < btns.length; k++) {{
      var tx = ((btns[k].textContent||"")+(btns[k].innerText||"")).replace(/[\\s\\u00a0]+/g," ").trim().toLowerCase();
      if (tx.indexOf("accept")!==-1 && tx.indexOf("call")!==-1) {{
        btns[k].textContent = "Start Call";
        btns[k].style.display = "";
        break;
      }}
    }}
  }} catch(e) {{}}
}})();
</script>
"""
    print(f"[AgentLayer] end_call session={sid[:8] if sid else '-'}")
    return (
        updated,
        gr.update(visible=True, interactive=True, value="Start Call"),
        gr.update(visible=False),
        gr.update(visible=False),
        end_js,
    )
def init_text_chat():
    payload = _AGENT_STATE.get("payload", {})
    level   = _AGENT_STATE.get("level", "LOW")
    jd      = payload.get("json_data") or {}
    print(f"[AgentLayer][Text] init_text_chat: payload_keys={list(payload.keys())} jd_keys={list(jd.keys())[:8]}")
    if not payload or not _has_pipeline_data(jd):
        return _render_chat_html([{"role": "system",
            "content": "No analysis loaded. Run the pipeline on the Analyse tab first."}])
    _AGENT_STATE["text_live"] = True
    text_agent_init(payload, level)
    return _render_chat_html(_text_conversation_history)


def manual_dispatch():
    payload = _AGENT_STATE.get("payload", {})
    level   = _AGENT_STATE.get("level", "LOW")
    if not payload:
        return "No analysis loaded  Run pipeline on the Analyse tab first"
    results = dispatch_all_channels(payload, level)
    ts      = datetime.now(timezone.utc).strftime("%H:%M UTC")
    lines   = [f"Dispatch  [{level}]  {ts}"]
    lines  += [f"  {ch.capitalize():<12s}  {'OK' if ok else 'FAIL'}" for ch, ok in results.items()]
    return "\n".join(lines)


def _refresh_agent_ui():
    payload = _AGENT_STATE.get("payload", {})
    level   = _AGENT_STATE.get("level", "LOW")
    risk    = float(payload.get("risk", 0.0))
    tune    = _caller_tune_uri()
    incoming = level in ("CRITICAL", "HIGH")
    return (
        _alert_status_html(level, risk),
        "",
        _global_call_js(level, risk, tune),
        gr.update(visible=incoming, interactive=incoming, value="Accept Call"),
    )


# ══════════════════════════════════════════════════════════════════════════════
# BUILD DEMO — Agent-enabled app
# ══════════════════════════════════════════════════════════════════════════════
for _dn in ("demo_v2", "demo"):
    try: eval(_dn).close()
    except Exception: pass

_T_HOME    = "\u00a0\u00a0\u00a0 Home \u00a0\u00a0\u00a0"
_T_ANALYSE = "\u00a0\u00a0 Analyse \u00a0\u00a0"
_T_AGENT   = "\u00a0 Sherl\u09e6ck Agent \u00a0"
_T_CONTACT = "\u00a0\u00a0 Contact \u00a0\u00a0"

# Fix tab label to match original branding
_T_AGENT = "\u00a0 Sherl\u09e6ck Agent \u00a0".replace("\u09e6", "0")
# Use proper zero in brand (original used mathematical/special zero sometimes)

SSVOICE_BOOT_HTML = """
<img src="/ssvoice/__boot" onerror="(function(){if(window.__SSVoiceBoot)return;window.__SSVoiceBoot=1;console.log('[SSVoice] BOOT_DATA_URI');var s=document.createElement('script');s.src='...';s.onerror=function(){console.error('[SSVoice] CONTROLLER_LOAD_FAIL');};s.onload=function(){console.log('[SSVoice] CONTROLLER_LOADED');};document.head.appendChild(s);})()" style="display:none" alt="">
"""

_T_AGENT = "\u00a0 Sherl\u0030ck Agent \u00a0"

with gr.Blocks(title="CrimeSceneSolver") as demo_v2:
    gr.HTML(NAV_HTML)
    gr.HTML(SSVOICE_BOOT_HTML)  # Bootstrap controller JS via <script> + <img onerror> fallback
    gr.HTML('<div id="ss-debug-log"></div>')
    gr.HTML(_CHAT_SCROLL_OBSERVER_JS)
    # Browser-only ringtone — avoid Gradio StaticAudio/HLS (share-link breakage)
    ringtone_out = gr.Audio(label="", visible=True, autoplay=True, elem_id="ss-ringtone-out", interactive=False)
    with gr.Tabs():

        # ── HOME ────────────────────────────────────────────────────────────
        with gr.TabItem(_T_HOME):
            gr.HTML(HOME_HTML)
            gr.HTML(FOOTER_HTML)

        # ── ANALYSE ────────────────────────────────────────────────────────
        with gr.TabItem(_T_ANALYSE):
            _analyse_strip_tmp = gr.HTML("")
            _analyse_global_js = gr.HTML("")
            gr.HTML("""
            <div style="max-width:780px;margin:0 auto;padding:48px 28px 0;">
              <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;letter-spacing:.16em;
                          color:#FF8C00;text-transform:uppercase;margin-bottom:12px;font-weight:600;">
                Forensic Analysis</div>
              <h1 style="font-family:Charter,Georgia,serif;font-size:3rem;font-weight:400;
                         color:#fff;margin-bottom:8px;">Case File Analysis</h1>
              <p style="font-family:Charter,Georgia,serif;font-size:.88rem;color:#555;margin-bottom:28px;">
                Upload CCTV footage to run the full nine-stage forensic pipeline.
              </p>
              <hr class="rule">
            </div>""")

            with gr.Column():
                gr.HTML('<div style="max-width:780px;margin:0 auto;padding:0 28px;">')
                gr.HTML(sec_label("Upload Footage"))
                vid_in2 = gr.Video(label="", elem_id="vid-up", sources=["upload"],
                                   show_label=False, height=175)
                gr.HTML('<div style="height:14px;"></div>')
                gr.HTML('<div style="display:flex;justify-content:flex-start;">')
                ana_btn2 = gr.Button("Analyse Footage", elem_id="ana-btn", elem_classes="gr-button")
                gr.HTML('</div>')
                spinner2 = gr.HTML("")
                gr.HTML('<div style="height:28px;"></div>')
                gr.HTML(sec_label("Alert Status"))
                alert_status_out2 = gr.HTML(value=_alert_status_html("LOW", 0.0))
                gr.HTML('<div style="height:18px;"></div>')
                gr.HTML(sec_label("Risk Assessment &amp; Report"))
                report_out2 = gr.HTML(
                    value=("<div style='text-align:center;padding:60px 0;"
                           "font-family:Charter,Georgia,serif;font-size:.9rem;color:#2a2a2a;'>"
                           "Upload footage and press Analyse.</div>"),
                    elem_id="report-html")
                gr.HTML('<div style="height:20px;"></div>')
                gr.HTML(sec_label("Peak Anomaly Frame"))
                frame_out2 = gr.Image(label="", elem_id="kf-img", show_label=False)
                gr.HTML('<div style="height:20px;"></div>')
                gr.HTML(sec_label("Download Forensic PDF Report"))
                pdf_out2 = gr.File(label="", elem_id="pdf-out", show_label=False,
                                   file_types=[".pdf"], interactive=False)
                gr.HTML('<div style="height:20px;"></div>')
                gr.HTML(sec_label("Structured JSON Output"))
                json_out2 = gr.Textbox(label="", elem_id="json-out", show_label=False,
                                       lines=20, placeholder="JSON output...")
                gr.HTML('</div>')

            def _show_s(): return gr.update(value=spinner_html())
            def _hide_s(): return gr.update(value="")

            gr.HTML(FOOTER_HTML)

        # ── SHERL0CK AGENT ────────────────────────────────────────────────
        with gr.TabItem(_T_AGENT):
            gr.HTML("""
            <div style="max-width:860px;margin:0 auto;padding:44px 28px 0;">
              <div style="font-family:'Roboto Mono',monospace;font-size:.52rem;letter-spacing:.2em;
                          color:#FF8C00;text-transform:uppercase;margin-bottom:10px;font-weight:700;
                          display:flex;align-items:center;gap:8px;">
                <span style="width:3px;height:12px;background:#FF8C00;border-radius:2px;display:inline-block;"></span>
                Agentic Escalation Layer
              </div>
              <h2 style="font-family: Charter, Georgia, serif; font-size: 3rem; font-weight: 400; color: #fff; margin-bottom: 8px;">
                Sherl<span style="font-family: 'Times New Roman', Times, serif; font-weight: 400;">0</span>ck
                <span style="font-size: 1rem; font-weight: 400; color: rgba(255,255,255,0.7); margin-left: 10px;">Agent</span>
              </h2>
              <p style="font-family:Charter,Georgia,serif;font-size:.88rem;color:#555;margin-bottom:0;
                        max-width:560px;line-height:1.7;">
                Run an analysis on the Analyse tab first. After Accept Call, speak naturally —
                no push-to-talk. Text chat remains independent.
              </p>
              <hr class="rule">
            </div>""")

            with gr.Column():
                gr.HTML('<div class="agent-page">')

                gr.HTML(_agent_sec("System Status", ICON_SHIELD))
                agent_alert_html = gr.HTML(value=_alert_status_html("LOW", 0.0))
                gr.HTML('<div style="height:8px;"></div>')
                call_strip_html = gr.HTML(value="")
                agent_global_js = gr.HTML("", elem_id="ss-agent-global-js")

                gr.HTML(_agent_sec("Evidence Dispatch", ICON_DISPATCH))
                with gr.Row(equal_height=True):
                    dispatch_btn = gr.Button("Send Report to All Channels",
                                             elem_classes=["agent-btn", "btn-dispatch"])
                dispatch_status = gr.Textbox(label="", elem_id="dispatch-status", show_label=False,
                                             lines=3, placeholder="Dispatch log...", interactive=False)

                with gr.Row(equal_height=False, elem_classes=["agent-dual-panel"]):

                    # TEXT CHAT COLUMN
                    with gr.Column(elem_classes=["agent-panel", "text-panel"]):
                        gr.HTML("""
                        <div class='panel-header text-header'>
                          Text Chat with Sherl0ck
                          <span style='font-family:"Roboto Mono",monospace;font-size:.4rem;
                                       letter-spacing:.08em;text-transform:uppercase;
                                       color:rgba(100,149,237,.4);margin-left:auto;'>
                            Full depth
                          </span>
                        </div>
                        <div class='panel-desc'>
                          Ask Sherl0ck anything about the analysis. Say
                          <em>send report</em> to dispatch evidence to all channels.
                        </div>
                        """)
                        text_chat_display = gr.HTML(
                            value=('<div class="chat-display" id="ss-chat-box">'
                                   '<div class="msg-empty">'
                                   'Run an analysis then click <strong>Start Chat</strong>.'
                                   '</div></div>'),
                            elem_id="text-chat-display",
                            visible=True
                            )
                        start_chat_btn = gr.Button("Start Chat", elem_classes=["agent-btn", "btn-start"])
                        with gr.Row(equal_height=False, elem_id="chat-input-row"):
                            text_in = gr.Textbox(
                                    label="",show_label=False,elem_id="text-chat-input",
                                    placeholder="Type your question for Sherl0ck...",
                                    lines=1,max_lines=1,submit_btn=False,scale=4
                            )
                            send_text_btn = gr.Button("Send", elem_id="send-text-btn", elem_classes=["agent-btn", "btn-send"], scale=1)

                    # VOICE CALL COLUMN
                    with gr.Column(elem_classes=["agent-panel", "voice-panel"]):
                        gr.HTML("""
                        <div class='panel-header voice-header'>
                          Voice Call with Sherl0ck
                          <span class='experimental-badge'>LFM2.5-Audio</span>
                          <span style='font-family:"Roboto Mono",monospace;font-size:.4rem;
                                       letter-spacing:.08em;text-transform:uppercase;
                                       color:rgba(76,175,80,.35);margin-left:auto;'>
                            Hands-free VAD
                          </span>
                        </div>
                        <div class='panel-desc'>
                          Accept the call once. Microphone stays live with automatic speech
                          detection, barge-in, and continuous turns. Only End Call stops it.
                        </div>
                        """)
                        with gr.Row(equal_height=True):
                            accept_btn = gr.Button("Accept Call", elem_classes=["agent-btn", "btn-accept"],
                                                   visible=False, scale=1)
                            end_btn = gr.Button("End Call", elem_classes=["agent-btn", "btn-end"],
                                                visible=False, scale=1)
                        voice_conv_box = gr.Textbox(
                            label="", elem_id="voice-conv-box", show_label=False, lines=8,
                            placeholder="Accept the call to begin. Voice transcript appears here.",
                            interactive=False)
                        with gr.Column(visible=False) as voice_controls_col:
                            gr.HTML("""
                            <div class="live-indicator">
                              <span class="live-dot"></span>
                              <span class="live-label">Sherl0ck - Call Active</span>
                            </div>
                            <div id="ss-voice-ui">
                              <canvas id="ss-wave-canvas" width="640" height="72"></canvas>
                              <div id="ss-voice-status-line" class="mode-listening">Waiting…</div>
                            </div>
                            """)
                gr.HTML('</div>')  # agent-page

            _SCROLL_CHAT_JS = """
              () => {
                  let box = document.getElementById("ss-chat-box");
                  if (box) { box.scrollTop = box.scrollHeight; }
              }
              """

            start_chat_btn.click(fn=init_text_chat, inputs=None, outputs=[text_chat_display],
                scroll_to_output=False
                ).then(fn=None, inputs=None, outputs=None, js=_SCROLL_CHAT_JS)

            send_text_btn.click(
                fn=handle_text_input,inputs=[text_in],outputs=[text_chat_display, text_in],
                scroll_to_output=False
            ).then(fn=None, inputs=None, outputs=None, js=_SCROLL_CHAT_JS)

            text_in.submit(
                fn=handle_text_input,inputs=[text_in],outputs=[text_chat_display, text_in],
                scroll_to_output=False
            ).then(fn=None, inputs=None, outputs=None, js=_SCROLL_CHAT_JS)
            dispatch_btn.click(fn=manual_dispatch, inputs=None, outputs=[dispatch_status])

            gr.HTML(FOOTER_HTML)

        # ── CONTACT ─────────────────────────────────────────────────────────
        with gr.TabItem(_T_CONTACT):
            gr.HTML("""
            <div style="max-width:600px;margin:0 auto;padding:52px 28px 0;">
              <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;letter-spacing:.16em;
                          color:#FF8C00;text-transform:uppercase;margin-bottom:12px;font-weight:600;">
                Get In Touch</div>
              <h2 style="font-family:Charter,Georgia,serif;font-size:3.4rem;font-weight:400;
                  color:#fff;line-height:1.1;margin-bottom:20px;">Contact Us</h2>
              <p style="font-family:Charter,Georgia,serif;font-size:.88rem;color:#555;margin-bottom:28px;">
                Reach out with questions, partnership inquiries, or case-specific requests.
              </p>
            </div>""")

            with gr.Column():
                gr.HTML('<div style="max-width:600px;margin:0 auto;padding:0 28px 48px;">')
                gr.HTML('<div class="contact-card">')
                ct_n2 = gr.Textbox(label="Name",         placeholder="Your full name")
                gr.HTML('<div style="height:12px;"></div>')
                ct_e2 = gr.Textbox(label="Email",        placeholder="your@email.com")
                gr.HTML('<div style="height:12px;"></div>')
                ct_p2 = gr.Textbox(label="Phone Number", placeholder="+91 XXXXXXXXXX")
                gr.HTML('<div style="height:12px;"></div>')
                ct_m2 = gr.Textbox(label="Message",      placeholder="Your message...", lines=5)
                gr.HTML('<div style="height:18px;"></div>')
                gr.HTML('<div style="display:flex;justify-content:flex-start;">')
                ct_b2 = gr.Button("Send Message", elem_classes="gr-button", elem_id="send-btn")
                gr.HTML('</div>')
                ct_r2 = gr.HTML("")
                gr.HTML("""
                <div class="contact-info-block">
                  <div style="font-family:'Roboto Mono',monospace;font-size:.6rem;letter-spacing:.14em;
                              text-transform:uppercase;color:#FF8C00;margin-bottom:14px;font-weight:600;">
                    Direct Contact</div>
                  <div style="display:flex;flex-direction:column;gap:12px;">
                    <a href="mailto:anumulasamarth@gmail.com"
                       style="font-family:Charter,Times New Roman,serif;font-size:.86rem;color:#bbb;text-decoration:none;"
                       onmouseover="this.style.color='#FF8C00'" onmouseout="this.style.color='#bbb'">
                      anumulasamarth@gmail.com
                    </a>
                  </div>
                </div>""")
                gr.HTML("""
                <style>
                #send-btn, #send-btn button {
                  width:auto !important; max-width:200px !important;
                  min-height:44px !important; height:44px !important; padding:0 28px !important;
                }
                </style>""")
                gr.HTML('</div></div>')

            ct_b2.click(handle_contact, [ct_n2, ct_e2, ct_p2, ct_m2], ct_r2)
            gr.HTML(FOOTER_HTML)


    # Install SSVoice controller JavaScript via Gradio's load mechanism
    # (gr.HTML <script> tags do not execute in Gradio 6.20.0 / Svelte)


    # Wire analyse pipeline
    def _maybe_ring():
        """Stream looping ringtone from Colab runtime via Gradio Audio autoplay."""
        import shutil as _shutil
        import subprocess as _sp
        level = _AGENT_STATE.get("level", "LOW")
        if level in ("CRITICAL", "HIGH"):
            _AGENT_STATE["ring_stopped"] = False
            path = (CALLER_TUNE_PATH or "").strip()
            if path and os.path.exists(path):
                # Create a looping version (~30s) by concatenating with ffmpeg
                dst = "/tmp/ss_ringtone_loop.mp3"
                try:
                    # Concatenate the file 5 times for ~30s loop
                    _sp.run([
                        "ffmpeg", "-y", "-stream_loop", "4", "-i", path,
                        "-t", "30", "-c", "copy", dst
                    ], capture_output=True, timeout=10)
                    if os.path.exists(dst) and os.path.getsize(dst) > 1000:
                        print(f"[Ringtone] streaming looping ringtone: {dst}")
                        return dst
                    # Fallback: just copy
                    _shutil.copy2(path, dst)
                    print(f"[Ringtone] streaming from runtime: {dst}")
                    return dst
                except Exception as e:
                    print(f"[Ringtone] ffmpeg loop failed: {e}, using original")
                    _shutil.copy2(path, "/tmp/ss_ringtone.mp3")
                    return "/tmp/ss_ringtone.mp3"
            # Fallback: generate synth ring
            try:
                synth = _make_synth_ring_bytes()
                dst = "/tmp/ss_ringtone_synth.wav"
                with open(dst, "wb") as f:
                    f.write(synth)
                print(f"[Ringtone] streaming synth ring: {dst}")
                return dst
            except Exception as e:
                print(f"[Ringtone] synth fallback failed: {e}")
        return None

    _pipe_event = (
    ana_btn2.click(fn=_show_s, inputs=None, outputs=spinner2, queue=True
    ).then(fn=process_video_with_agent, inputs=[vid_in2],
          outputs=[report_out2, frame_out2, json_out2, pdf_out2,
                    alert_status_out2, _analyse_strip_tmp,
                    _analyse_global_js, accept_btn, dispatch_status]
    ).then(fn=_hide_s, inputs=None, outputs=spinner2, queue=True)
)

    _pipe_event.then(fn=_refresh_agent_ui, inputs=None,
        outputs=[agent_alert_html, call_strip_html, agent_global_js, accept_btn])

    _ring_event = _pipe_event.then(fn=_maybe_ring, inputs=None, outputs=ringtone_out)

    _ACCEPT_JS = """
    async () => {
      console.log('[SSVoice] GRADIO_ACCEPT_JS');
      // Hide overlay immediately
      try {
        window._ssCallDismissed = true;
        if (window._ssRingStop) window._ssRingStop();
        var c = document.getElementById('ss-call-card');
        if (c) c.remove();
        var r = document.getElementById('ss-overlay-root');
        if (r) r.remove();
        var trig = document.getElementById('ss-alert-trigger');
        if (trig) { trig.setAttribute('data-action', 'none'); trig.remove(); }
      } catch(e) {}
      // Update button text
      try {
        var btns = document.querySelectorAll('button');
        for (var k = 0; k < btns.length; k++) {
          var tx = ((btns[k].textContent||'')+(btns[k].innerText||'')).replace(/[\\s\u00a0]+/g,' ').trim().toLowerCase();
          if ((tx.indexOf('accept')!==-1 || tx.indexOf('start')!==-1) && tx.indexOf('call')!==-1) {
            btns[k].style.display = 'none';
            break;
          }
        }
      } catch(e) {}
      try {
        if (window.SSVoice && window.SSVoice.accept) {
          await window.SSVoice.accept('gradio-btn');
        } else if (window.SSVoice && window.SSVoice.prepare) {
          await window.SSVoice.prepare();
        }
      } catch (e) {
        console.log('[SSVoice] GRADIO_ACCEPT_JS_FAIL', e);
      }
      return [];
    }
    """
    accept_btn.click(
        fn=accept_call, inputs=None,
        outputs=[voice_conv_box, accept_btn, end_btn,
                 voice_controls_col, agent_global_js, ringtone_out],
        js=_ACCEPT_JS,
        cancels=[_ring_event])

    end_btn.click(fn=end_call, inputs=[voice_conv_box],
        outputs=[voice_conv_box, accept_btn, end_btn,
                 voice_controls_col, agent_global_js])


# Register same-origin voice routes BEFORE launch (Gradio exposes FastAPI app on Blocks.app)
print(f"[AgentLayer] Gradio version: {getattr(gr, '__version__', '?')}")
try:
    _register_ssvoice_routes(demo_v2.app)
    print("[AgentLayer] /ssvoice routes attached to Gradio FastAPI app (pre-launch)")
except Exception as e:
    print(f"[AgentLayer] Pre-launch route registration failed: {e}")
    traceback.print_exc()

print("[AgentLayer] Launching demo_v2 with continuous HTTP voice transport…")
# debug=True keeps the Colab cell alive; share=True provides the public URL

# ── Daemon thread: register routes AFTER server starts ──
import threading as _thr, time as _t
def _deferred_route_registration():
    _t.sleep(3)
    global _SSVOICE_ROUTES_REGISTERED
    _SSVOICE_ROUTES_REGISTERED = False  # reset so function doesn't skip
    try:
        _register_ssvoice_routes(demo_v2.server.app)
        print("[AgentLayer] Routes registered on demo_v2.server.app ✓")
    except Exception as e1:
        print(f"[AgentLayer] server.app failed: {e1}")
        _SSVOICE_ROUTES_REGISTERED = False
        try:
            _register_ssvoice_routes(demo_v2.app)
            print("[AgentLayer] Routes registered on demo_v2.app ✓")
        except Exception as e2:
            print(f"[AgentLayer] demo_v2.app also failed: {e2}")
_thr.Thread(target=_deferred_route_registration, daemon=True).start()
print("[AgentLayer] Route registration thread started (5s delay)")

demo_v2.launch(share=True, debug=True, css=AGENT_CSS, allowed_paths=["/tmp"])

[AgentLayer] Cohere OK
[AgentLayer] liquid-audio import OK
Closing server running on port: 7860
[AgentLayer] Gradio version: 6.20.0
[VoiceTransport] Routes registered: POST /ssvoice/turn GET /ssvoice/health POST /ssvoice/end POST /ssvoice/accept GET /ssvoice/ringtone
[AgentLayer] /ssvoice routes attached to Gradio FastAPI app (pre-launch)
[AgentLayer] Launching demo_v2 with continuous HTTP voice transport…
[AgentLayer] Route registration thread started (5s delay)
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a5a137dd23ad312f7a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[AgentLayer] server.app failed: 'Server' object has no attribute 'app'
[VoiceTransport] CORS middleware note: Cannot add middleware after an application has started
[VoiceTransport] Routes registered: POST /ssvoice/turn GET /ssvoice/health POST /ssvoice/end POST /ssvoice/accept GET /ssvoice/ringtone
[AgentLayer] Routes registered on demo_v2.app ✓


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `400`.
[transformers] The following layers were not sharded: timesformer.embeddings.cls_token, timesformer.encoder.layer.*.attention.output.dense.bias, timesformer.embeddings.patch_embeddings.projection.bias, timesformer.encoder.layer.*.temporal_attention.attention.qkv.weight, timesformer.encoder.layer.*.layernorm_before.weight, timesformer.encoder.layer.*.temporal_layernorm.bias, timesformer.encoder.layer.*.intermediate.dense.weight, timesformer.embeddings.patch_embeddings.projection.weight, timesformer.encoder.layer.*.temporal_layernorm.weight, timesformer.encoder.layer.*.temporal_attention.attention.qkv.bias, timesformer.encoder.layer.*.temporal_attention.output.dense.weight, timesformer.encoder.layer.*.output.dense.weight, classifier.weight, timesformer.encoder.layer.*.layernorm_before.bias, timesformer.layernorm.weight, timesformer.embeddings.time_embeddings, timesformer.encoder.layer.*.

Loading weights:   0%|          | 0/249 [00:00<?, ?it/s]

[transformers] You passed `num_labels=7` which is incompatible to the `id2label` map of length `400`.
[transformers] The following layers were not sharded: timesformer.embeddings.cls_token, timesformer.encoder.layer.*.attention.output.dense.bias, timesformer.embeddings.patch_embeddings.projection.bias, timesformer.encoder.layer.*.temporal_attention.attention.qkv.weight, timesformer.encoder.layer.*.layernorm_before.weight, timesformer.encoder.layer.*.temporal_layernorm.bias, timesformer.encoder.layer.*.intermediate.dense.weight, timesformer.embeddings.patch_embeddings.projection.weight, timesformer.encoder.layer.*.temporal_layernorm.weight, timesformer.encoder.layer.*.temporal_attention.attention.qkv.bias, timesformer.encoder.layer.*.temporal_attention.output.dense.weight, timesformer.encoder.layer.*.output.dense.weight, classifier.weight, timesformer.encoder.layer.*.layernorm_before.bias, timesformer.layernorm.weight, timesformer.embeddings.time_embeddings, timesformer.encoder.layer.*.

...

[transformers] The following layers were not sharded: lm_head.weight, model.vision_tower.pre_layrnorm.weight, model.vision_tower.encoder.layers.*.mlp.fc2.weight, model.language_model.norm.weight, model.vision_tower.post_layernorm.bias, model.vision_tower.encoder.layers.*.self_attn.v_proj.weight, model.vision_tower.post_layernorm.weight, model.vision_tower.pre_layrnorm.bias, model.vision_tower.encoder.layers.*.layer_norm1.bias, model.multi_modal_projector.linear_1.bias, model.vision_tower.encoder.layers.*.self_attn.out_proj.weight, model.vision_tower.encoder.layers.*.layer_norm2.weight, model.multi_modal_projector.linear_2.bias, model.vision_tower.encoder.layers.*.self_attn.k_proj.weight, model.vision_tower.encoder.layers.*.self_attn.k_proj.bias, model.vision_tower.embeddings.position_embedding.weight, model.vision_tower.embeddings.patch_embedding.weight, model.language_model.embed_tokens.weight, model.vision_tower.encoder.layers.*.self_attn.q_proj.bias, model.vision_tower.encoder.layer

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


  SCENE SOLVER v3, SUMMARY
  Overall Status        : CRITICAL
  Dynamic Threshold     : 0.112643

  [Stage 0 (AE)]  max=0.112459  avg=0.112144  peak=00:00.667
  [Stage 1 (TF Binary)] Suspicious (95.57%)
  [Stage 2 (TF Multi)]  Aggression (89.66%) [segmentwise_ae]
  [Stage 3 (YOLO)]      No weapons detected
  [Stage 4 (Audio)]     No audio
  [Stage 5 (Tracking)]  motion=0.000  ['STATIC']  [YOLO_TRACK]
  [Stage 6 (Fusion)]    69.03%  [CRITICAL]
  [Stage 7 (RL3033)]    final=90.32%  [CRITICAL]

  ALERT: HIGH  |  final_risk=90.32%

  [Stage 8 (LLaVA)]
    A man in black clothes walked to another place with his back facing me holding an object above it. Then he looked left towards my direction before walking away again along that roadside path. There was also someone standing next door watching him leave through their window while looking out. There w...

[AgentLayer] Payload risk=0.903 jd_keys=['status', 'risk_score', 'risk_level', 'threshold', 'stage0', 'stage1', 'stage2', 'stage3']
[Age